# Project overview

## Summary

## workflow


## project folders

In [ ]:
%%script bash

#make dataset folders

cd /mnt/datawk1/data/Lara

#make analysis folders

cd /mnt/datawk1/analysis/Lara/

mkdir Lara_ChIPseq_nospikein_A2
mkdir Lara_ChIPseq_nospikein_A2_unique
mkdir Lara_ChIPseq_nospikein_A1
mkdir Lara_ChIPseq_nospikein_A1_unique



## Documentation

### samplesheets

Main Samplesheet chip-seq : https://docs.google.com/spreadsheets/d/1qka7K36erZNLOrqktTqEGD-cPEBJPkqktTCROyASHto/edit?gid=0#gid=0


Main Samplesheet RNA-seq : https://docs.google.com/spreadsheets/d/1hHq9lsfRQ2Nt2cpwgc0BAid6LioyB2FJB-PoUGJBwpI/edit?gid=0#gid=0

Main Samplesheet microC : https://docs.google.com/spreadsheets/d/1uY5NbxQ3ZNaRQaMbKecmI2kU_lheT4NL6VcBaOujXjE/edit?gid=0#gid=0





In [ ]:
%%script bash

# save bam file on nas

#with spikein

#unique
prjNames="Lara_ChIP_with_spikein_A1 "

#random
prjName




## Load global environment

In [ ]:
%%script bash

#make symlink in "in" folder

prj_name="MLL2_L1_regulation"


cd git
git clone git@github.com:lucidif/base.git
cd ..

chmod +x git/base/bin/*.sh

#prj_name="Lara_MLL2"
source "git/${prj_name}/bin/initialize.sh"

bash git/base/bin/get_config.sh sheets/

bash git/base/bin/link_to_analysis.sh \
  --table sheets/analysis_folders.tsv \
  --out outs/ \
  --project MLL2_L1_regulation \
  --apply --verbose

bash git/base/bin/link_references.sh \
  --table sheets/local_standard_folders.tsv \
  --out in/ \
  --verbose \
  --apply


# ChIPseq

## 0. Save raw Data on long-time storage ✅

### environment def

#### R functions TODO save fun in git and source from git

In [ ]:
%%R

reformat_macrogen<-function(dwnlinkFolder, outfile){

  #dwlinkFolder="/mnt/datawk1/data/flid/HN00215148_2024_04_Lara_chip/HN00215148_6samples_md5sum_DownloadLink.txt"
  #outfile="/mnt/datawk1/data/flid/HN00215148_2024_04_Lara_chip/SS_HN00215148_2024_04_Lara_chip.tsv"


  dwnlink<-read.table(dwnlinkFolder,
                      header=TRUE)

  name<-dwnlink[,1]
  name<-gsub("_1.fastq.gz", "", dwnlink[,1])
  name<-gsub("_2.fastq.gz", "", name)

  uname<-unique(name)

  link1<-c()
  link2<-c()
  md5_1<-c()
  md5_2<-c()

  for (j in 1:length(uname)){

    # link1[j]<-dwnlink[grep(paste0(uname[j], "_1.fastq.gz"), dwnlink[,4]),4]
    # link2[j]<-dwnlink[grep(paste0(uname[j], "_2.fastq.gz"), dwnlink[,4]),4]
    # md5_1[j]<-dwnlink[grep(paste0(uname[j], "_1.fastq.gz"), dwnlink[,4]),3]
    # md5_2[j]<-dwnlink[grep(paste0(uname[j], "_2.fastq.gz"), dwnlink[,4]),3]

    link1[j]<-dwnlink[which(dwnlink[,1]==paste0(uname[j], "_1.fastq.gz")),4]
    link2[j]<-dwnlink[which(dwnlink[,1]==paste0(uname[j], "_2.fastq.gz")),4]
    md5_1[j]<-dwnlink[which(dwnlink[,1]==paste0(uname[j], "_1.fastq.gz")),3]
    md5_2[j]<-dwnlink[which(dwnlink[,1]==paste0(uname[j], "_2.fastq.gz")),3]


  }

  reformatTable<-cbind(sample=uname,link1,link2,md5sum1=md5_1,md5sum2=md5_2)

  write.table(reformatTable, file=outfile, sep="\t",
              row.names = FALSE,
              col.names=TRUE,
              quote=FALSE)

}


### HN00214038_2024_02

In [ ]:
%%script bash

cd /mnt/datawk1/data/flid/HN00214038_240219_Lara_chip_day4/fastq/240219_Lara_chip_day4/
md5sum *.fastq.gz > hash.txt

cd /mnt/datawk1/data/flid/HN00214038_240219_Lara_chip_day4/fastq/Others
md5sum *.fastq.gz > hash.txt

scp -r /mnt/datawk1/data/flid/HN00214038_240219_Lara_chip_day4 lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Data/flid/

ssh lucio@193.144.215.241

cd /volume1/Data_Just_In_NAS/Lucio/Data/flid/HN00214038_240219_Lara_chip_day4/fastq/240219_Lara_chip_day4

nrow=$(wc -l < hash.txt)
for ((i = 1; i <= nrow; i++)); do
    ##echo $i
    tarhash=`cut -d ' ' -f 1 hash.txt | head -n$i | tail -n1`
    trname=`cut -d ' ' -f 3 hash.txt | head -n$i | tail -n1`
    echo $flname
    echo $trname
    examin_hash=`md5sum $trname | cut -d ' ' -f 1`
    if [ "$tarhash" = "$examin_hash" ]; then
      checkout="ok"
    else
      checkout="ERROR"
    fi
    echo "${trname}: ${tarhash} = ${examin_hash}; ${checkout}"
done

cd /volume1/Data_Just_In_NAS/Lucio/Data/flid/HN00214038_240219_Lara_chip_day4/fastq/Others

nrow=$(wc -l < hash.txt)
for ((i = 1; i <= nrow; i++)); do
    ##echo $i
    tarhash=`cut -d ' ' -f 1 hash.txt | head -n$i | tail -n1`
    trname=`cut -d ' ' -f 3 hash.txt | head -n$i | tail -n1`
    examin_hash=`md5sum $trname | cut -d ' ' -f 1`
    if [ "$tarhash" = "$examin_hash" ]; then
      checkout="ok"
    else
      checkout="ERROR"
    fi
    echo "${trname}: ${tarhash} = ${examin_hash}; ${checkout}"
done


### HN00215148_2024_04

In [ ]:
%%R


#bring the function from environment
reformat_macrogen ("/mnt/datawk1/data/flid/HN00215148_2024_04_Lara_chip/HN00215148_6samples_md5sum_DownloadLink.txt",
                   "/mnt/datawk1/data/flid/HN00215148_2024_04_Lara_chip/SS_HN00215148_2024_04_Lara_chip.tsv")



In [ ]:
%%script bash

cd /mnt/datawk1/data/flid/HN00215148_2024_04_Lara_chip/fastq

ss="/mnt/datawk1/data/flid/HN00215148_2024_04_Lara_chip/SS_HN00215148_2024_04_Lara_chip.tsv"

nrow=`wc -l $ss | cut -d' ' -f1`

for (( i=2; i<=nrow; i++ )); do
  samplename=`cut -f1 ${ss} | head -n $i | tail -n 1`
  url1=`cut -f2 ${ss} | head -n $i | tail -n 1`
  url2=`cut -f3 ${ss} | head -n $i | tail -n 1`
  echo $samplename
  echo $url1
  echo $url2
  wget $url1 ./
  wget $url2 ./
done

#md5 check
for (( i=2; i<=nrow; i++ )); do
  samplename=`cut -f1 ${ss} | head -n $i | tail -n 1`
  prevmd5_1=$(cut -f4 ${ss} | head -n $i | tail -n 1)
  curmd5_1=`md5sum ./${samplename}_1.fastq.gz | cut -d" " -f1`
  if [ "$curmd5_1" == "$prevmd5_1" ]; then
    echo "$samplename R1:${prevmd5_1} => ${curmd5_1} MD5 OK"
  else
    echo "$samplename R1:${prevmd5_1} => ${curmd5_1} MD5 WRONG"
  fi

  prevmd5_2=$(cut -f5 ${ss} | head -n $i | tail -n 1)
  curmd5_2=`md5sum ./${samplename}_2.fastq.gz | cut -d" " -f1`
  if [ "$curmd5_2" == "$prevmd5_2" ]; then
    echo "$samplename R2:${prevmd5_2} => ${curmd5_2} MD5 OK"
  else
    echo "$samplename R2:${prevmd5_2} => ${curmd5_2} MD5 WRONG"
  fi
done





In [ ]:
cd /mnt/datawk1/data/flid/HN00215148_2024_04_Lara_chip
md5sum *.fastq.gz > hash.txt

scp -r /mnt/datawk1/data/flid/HN00215148_2024_04_Lara_chip lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Data/flid/

ssh lucio@193.144.215.241

cd /volume1/Data_Just_In_NAS/Lucio/Data/flid/HN00215148_2024_04_Lara_chip/fastq

nrow=$(wc -l < hash.txt)
for ((i = 1; i <= nrow; i++)); do
    ##echo $i
    tarhash=`cut -d ' ' -f 1 hash.txt | head -n$i | tail -n1`
    trname=`cut -d ' ' -f 3 hash.txt | head -n$i | tail -n1`
    echo $flname
    echo $trname
    examin_hash=`md5sum $trname | cut -d ' ' -f 1`
    if [ "$tarhash" = "$examin_hash" ]; then
      checkout="ok"
    else
      checkout="ERROR"
    fi
    echo "${trname}: ${tarhash} = ${examin_hash}; ${checkout}"
done

exit

### HN00220667_2024_07


In [ ]:
%%R

reformat_macrogen ("/mnt/datawk1/data/flid/20240704_HN00220667/20240704_HN00220667_CHS_Report/assets/spgs/HN00220667_16samples_md5sum_DownloadLink.txt",
                   "/mnt/datawk1/data/flid/20240704_HN00220667/SS_HN00220667.tsv")


save on long-time storage

In [ ]:
%%script bash

cd /mnt/datawk1/data/flid/20240704_HN00220667/fastq

ss="/mnt/datawk1/data/flid/20240704_HN00220667/SS_HN00220667.tsv"

nrow=`wc -l $ss | cut -d' ' -f1`

for (( i=2; i<=nrow; i++ )); do
  samplename=`cut -f1 ${ss} | head -n $i | tail -n 1`
  url1=`cut -f2 ${ss} | head -n $i | tail -n 1`
  url2=`cut -f3 ${ss} | head -n $i | tail -n 1`
  echo $samplename
  echo $url1
  echo $url2
  wget $url1 ./
  wget $url2 ./
done

#md5 check
for (( i=2; i<=nrow; i++ )); do
  samplename=`cut -f1 ${ss} | head -n $i | tail -n 1`
  prevmd5_1=$(cut -f4 ${ss} | head -n $i | tail -n 1)
  curmd5_1=`md5sum ./${samplename}_1.fastq.gz | cut -d" " -f1`
  if [ "$curmd5_1" == "$prevmd5_1" ]; then
    echo "$samplename R1:${prevmd5_1} => ${curmd5_1} MD5 OK"
  else
    echo "$samplename R1:${prevmd5_1} => ${curmd5_1} MD5 WRONG"
  fi

  prevmd5_2=$(cut -f5 ${ss} | head -n $i | tail -n 1)
  curmd5_2=`md5sum ./${samplename}_2.fastq.gz | cut -d" " -f1`
  if [ "$curmd5_2" == "$prevmd5_2" ]; then
    echo "$samplename R2:${prevmd5_2} => ${curmd5_2} MD5 OK"
  else
    echo "$samplename R2:${prevmd5_2} => ${curmd5_2} MD5 WRONG"
  fi
done

In [ ]:
%%script bash


cd /mnt/datawk1/data/flid/20240704_HN00220667_only_Lara_samples/fastq

md5sum *.fastq.gz > hash.txt

scp -r /mnt/datawk1/data/flid/20240704_HN00220667_only_Lara_samples lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Data/flid/

ssh lucio@193.144.215.241

cd /volume1/Data_Just_In_NAS/Lucio/Data/flid/20240704_HN00220667_only_Lara_samples/fastq

nrow=$(wc -l < hash.txt)
for ((i = 1; i <= nrow; i++)); do
    ##echo $i
    tarhash=`cut -d ' ' -f 1 hash.txt | head -n$i | tail -n1`
    trname=`cut -d ' ' -f 3 hash.txt | head -n$i | tail -n1`
    echo $flname
    echo $trname
    examin_hash=`md5sum $trname | cut -d ' ' -f 1`
    if [ "$tarhash" = "$examin_hash" ]; then
      checkout="ok"
    else
      checkout="ERROR"
    fi
    echo "${trname}: ${tarhash} = ${examin_hash}; ${checkout}"
done

exit





### HN00226631_2024_09

In [ ]:
%%R

reformat_macrogen ("/mnt/datawk1/data/Lara/240926_HN00226631_Lara_chip/20240926_HN00226631_CHS_Report/assets/spgs/HN00226631_27samples_md5sum_DownloadLink.txt",
                   "/mnt/datawk1/data/Lara/240926_HN00226631_Lara_chip/SS_HN00220667.tsv")


save on long-time storage

In [ ]:
%%script bash

cd /mnt/datawk1/data/Lara/240926_HN00226631_Lara_chip/fastq

ss="/mnt/datawk1/data/Lara/240926_HN00226631_Lara_chip/SS_HN00220667_LaraSamples.tsv"

nrow=`wc -l $ss | cut -d' ' -f1`

for (( i=2; i<=nrow; i++ )); do
  samplename=`cut -f1 ${ss} | head -n $i | tail -n 1`
  url1=`cut -f2 ${ss} | head -n $i | tail -n 1`
  url2=`cut -f3 ${ss} | head -n $i | tail -n 1`
  echo $samplename
  echo $url1
  echo $url2
  wget --content-disposition $url1 ./
  wget --content-disposition $url2 ./
done

#md5 check
for (( i=2; i<=nrow; i++ )); do
  samplename=`cut -f1 ${ss} | head -n $i | tail -n 1`
  prevmd5_1=$(cut -f4 ${ss} | head -n $i | tail -n 1)
  curmd5_1=`md5sum ./${samplename}_1.fastq.gz | cut -d" " -f1`
  if [ "$curmd5_1" == "$prevmd5_1" ]; then
    echo "$samplename R1:${prevmd5_1} => ${curmd5_1} MD5 OK"
  else
    echo "$samplename R1:${prevmd5_1} => ${curmd5_1} MD5 WRONG"
  fi

  prevmd5_2=$(cut -f5 ${ss} | head -n $i | tail -n 1)
  curmd5_2=`md5sum ./${samplename}_2.fastq.gz | cut -d" " -f1`
  if [ "$curmd5_2" == "$prevmd5_2" ]; then
    echo "$samplename R2:${prevmd5_2} => ${curmd5_2} MD5 OK"
  else
    echo "$samplename R2:${prevmd5_2} => ${curmd5_2} MD5 WRONG"
  fi
done

In [ ]:
%%script bash


cd /mnt/datawk1/data/Lara/240926_HN00226631_Lara_chip/fastq

md5sum *.fastq.gz > hash.txt

scp -r /mnt/datawk1/data/Lara/240926_HN00226631_Lara_chip lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Data/Lara/

ssh lucio@193.144.215.241

cd /volume1/Data_Just_In_NAS/Lucio/Data/flid/20240704_HN00220667_only_Lara_samples/fastq

nrow=$(wc -l < hash.txt)
for ((i = 1; i <= nrow; i++)); do
    ##echo $i
    tarhash=`cut -d ' ' -f 1 hash.txt | head -n$i | tail -n1`
    trname=`cut -d ' ' -f 3 hash.txt | head -n$i | tail -n1`
    echo $flname
    echo $trname
    examin_hash=`md5sum $trname | cut -d ' ' -f 1`
    if [ "$tarhash" = "$examin_hash" ]; then
      checkout="ok"
    else
      checkout="ERROR"
    fi
    echo "${trname}: ${tarhash} = ${examin_hash}; ${checkout}"
done

exit


import in ziggy

In [ ]:
%%bash

scp lucio@maindevices.58-11-22-cc-7a-c7@cloud.shellhub.io:/media/lucio/bioData1/3.testFolder/Lara/240926_HN00226631_Lara_chip/nfout/random/star/mergedLibrary/bigwig/deeptools/* \
/mnt/datawk1/analysis/Lara/240926_HN00226631_Lara_chip/nfout/random/star/mergedLibrary/bigwig/deeptools/


### 20260226_HN00264851

In [ ]:
%%R

source("git/base/bin/reformat_macrogen.R")

reformat_macrogen ("/mnt/datawk1/data/Lara/20260226_HN00264851/20260226_HN00264851_CHS_Report/assets/spgs/HN00264851_12samples_md5sum_DownloadLink.txt", outfile="/mnt/datawk1/data/Lara/20260226_HN00264851/20260226_HN00264851_ss.tsv")



In [ ]:
%%script bash

mkdir /mnt/datawk1/data/Lara/20260226_HN00264851/fastq

cd /mnt/datawk1/data/Lara/20260226_HN00264851/fastq

ss="/mnt/datawk1/data/Lara/20260226_HN00264851/20260226_HN00264851_ss.tsv"

nrow=`wc -l $ss | cut -d' ' -f1`

for (( i=2; i<=nrow; i++ )); do
  samplename=`cut -f1 ${ss} | head -n $i | tail -n 1`
  url1=`cut -f2 ${ss} | head -n $i | tail -n 1`
  url2=`cut -f3 ${ss} | head -n $i | tail -n 1`
  echo $samplename
  echo $url1
  echo $url2
  wget --content-disposition $url1 ./
  wget --content-disposition $url2 ./
done

#md5 check
for (( i=2; i<=nrow; i++ )); do
  samplename=`cut -f1 ${ss} | head -n $i | tail -n 1`
  prevmd5_1=$(cut -f4 ${ss} | head -n $i | tail -n 1)
  curmd5_1=`md5sum ./${samplename}_1.fastq.gz | cut -d" " -f1`
  if [ "$curmd5_1" == "$prevmd5_1" ]; then
    echo "$samplename R1:${prevmd5_1} => ${curmd5_1} MD5 OK"
  else
    echo "$samplename R1:${prevmd5_1} => ${curmd5_1} MD5 WRONG"
  fi

  prevmd5_2=$(cut -f5 ${ss} | head -n $i | tail -n 1)
  curmd5_2=`md5sum ./${samplename}_2.fastq.gz | cut -d" " -f1`
  if [ "$curmd5_2" == "$prevmd5_2" ]; then
    echo "$samplename R2:${prevmd5_2} => ${curmd5_2} MD5 OK"
  else
    echo "$samplename R2:${prevmd5_2} => ${curmd5_2} MD5 WRONG"
  fi
done

#import on NAS

#cd /mnt/datawk1/data/Lara/20260226_HN00264851/fastq

md5sum *.fastq.gz > hash.txt

#TODO
scp -r /mnt/datawk1/data/Lara/20260226_HN00264851 lucio@193.144.215.241:/volume2/luciov2/data/Lara/

ssh lucio@193.144.215.241

cd /volume2/luciov2/data/Lara/20260226_HN00264851/fastq


nrow=$(wc -l < hash.txt)
for ((i = 1; i <= nrow; i++)); do
    ##echo $i
    tarhash=`cut -d ' ' -f 1 hash.txt | head -n$i | tail -n1`
    trname=`cut -d ' ' -f 3 hash.txt | head -n$i | tail -n1`
    echo $flname
    echo $trname
    examin_hash=`md5sum $trname | cut -d ' ' -f 1`
    if [ "$tarhash" = "$examin_hash" ]; then
      checkout="ok"
    else
      checkout="ERROR"
    fi
    echo "${trname}: ${tarhash} = ${examin_hash}; ${checkout}"
done






## 1. test pipeline ✅

### a) test : remove human reads from fastq

TODO add this:  

*   add trimminig step
*   add this to script: samtools view -f 0xc file.bam | samtools fasta | grep '>' | akw -F'[>/]' ' {print $2 }' | uniq > totake.txt
*   add this to pipeline : seqtk subseq r1.fastq.gz totake.txt | gzip > r1_cleaned.fq.gz #(same with r2)


and modify pileline

test the spikein comprensive pipeline

In [ ]:
sudo nextflow run /home/lucio/MEGA/linux/ziggy/git/remove_contaminator \
--input /mnt/datawk1/analysis/Lara/Lara_cleaning_test/ss_in_one_sample.csv \
--outdir /mnt/datawk1/analysis/Lara/Lara_cleaning_test \
--index /mnt/datawk1/references/UCSC_GRCh38_star_2_7_10a \
--gtf /mnt/datawk1/references/annotations/UCSC_hg38.ensGene.gtf \
--ref_index /mnt/datawk1/references/UCSC_GRCm38_star_2_7_10a \
--ref_gtf /mnt/datawk1/references/annotations/UCSC_mm10.ensGene.gtf \
--hybrid_index /mnt/datawk1/references/mm10_hg19_bowtie2 \
--hybrid_fasta /mnt/datawk1/references/fasta/mm10_hg19.fa \
--reference_genome "mm10" --spikein_genome "hg19" \
-c "/mnt/datawk1/analysis/Lara/Lara_cleaning_test/mappingParameters.config" \
-profile docker

sudo nextflow run /home/lucio/MEGA/linux/ziggy/git/chipseq_RE \
--input /mnt/datawk1/analysis/Lara/Lara_cleaning_test/ss_mapping_in_one_sample.csv \
--outdir /mnt/datawk1/analysis/Lara/Lara_cleaning_test/chipseq_RE \
--read_length 150 --fasta "/mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa" \
--gtf "/mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf" \
--aligner "star" --filters_disable --effectiveGenomeSize 2494787188 \
--email difilippolucio@gmail.com \
-profile docker \
-c /mnt/datawk1/analysis/Lara/Lara_chipseq_spikein_STARtests/random.config









#### 2)

### b) test : hybrid genome allignment and mouse bam extraction

In [ ]:
~/MEGA/linux/ziggy/git/chipseq_RE

git checkout master

nextflow run /home/lucio/MEGA/linux/ziggy/git/chipseq_RE \
--input /mnt/datawk1/analysis/Lara/Lara_test_hybrid_mapping/ss_in.csv \
--outdir /mnt/datawk1/analysis/Lara/Lara_test_hybrid_mapping --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--spikein_fasta /mnt/datawk1/references/fasta/UCSC_hg19/hg19.fa \
--spikein_gtf /mnt/datawk1/references/annotations/UCSC_hg19.refGene/hg19.refGene.gtf \
--aligner star --reference_genome mm10 --spikein_genome hg19 --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c /mnt/datawk1/analysis/Lara/Lara_test_hybrid_mapping/random.config -resume

### c) test no spikein analysis

##### i) download toy data

In [ ]:
curl https://raw.githubusercontent.com/nf-core/test-datasets/chipseq/samplesheet/v2.0/samplesheet_test.csv \
-o /media/lucio/caddyHDD/bioinformatics/wkdir/test_nospikein/samplesheet_test.csv

rsync --partial --progress -av -e ssh lucio@193.144.215.206:/mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
/media/lucio/caddyHDD/bioinformatics/references/

rsync --partial --progress -av -e ssh lucio@193.144.215.206:/mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
/media/lucio/caddyHDD/bioinformatics/references/

rsync --partial --progress -av -e ssh lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/Lara_test_hybrid_mapping/random.config \
/media/lucio/caddyHDD/bioinformatics/wkdir/test_nospikein



##### ii) run scipt

In [ ]:
nextflow run /mnt/783E92803E9236DA/linMEGA/linux/barbone/github/chipseq_RE \
--input /media/lucio/caddyHDD/bioinformatics/wkdir/test_nospikein/samplesheet_test.csv \
--outdir /media/lucio/caddyHDD/bioinformatics/wkdir/test_nospikein --read_length 150 \
--fasta /media/lucio/caddyHDD/bioinformatics/references/mm10.fa \
--gtf /media/lucio/caddyHDD/bioinformatics/references/mm10.refGene.gtf \
--aligner star --filters_disable --max_cpus 6 --max_memory "16.GB" \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c /media/lucio/caddyHDD/bioinformatics/wkdir/test_nospikein/random.config -resume

### d) test downtream indipendent pipeline

import test data

In [ ]:
cd /mnt/c/wkdir/test_downstreamChip

rsync --progress -av \
lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2_unique/star/mergedLibrary/1_input.mLb.mkD.sorted.bam* \
/mnt/c/wkdir/test_downstreamChip/indata/

rsync -av \
lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2_unique/star/mergedLibrary/Anti-GFP.mLb.mkD.sorted.bam* \
/mnt/c/wkdir/test_downstreamChip/indata/

mkdir git
mkdir work
mkdir nfout

cd git

git clone git@github.com:lucidif/chipseq_downstream.git

git clone git@github.com:lucidif/chipseq_RE.git

cd chipseq_RE
git checkout -b downstreamOnly


cd ..

sudo nextflow run ./git/chipseq_downstream --input sstest.csv \
--fasta /mnt/c/wkdir/mm39.fa --outdir nfout -profile docker -resume

sudo nextflow run ./git/chipseq_RE \
--input --input ss_in.csv \
--outdir nfout --read_length 150 \
--fasta /mnt/c/wkdir/mm10.fa \
--gtf /mnt/c/wkdir/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-resume




## 2. Analyze samples with spikein ✅

### a) environment ✅

In [ ]:
ssh lucio@193.144.215.206

###  b) remove spikein reads from files and analyze them :

#### Lara_ChIP_with_spikein_A1 ✅

In [ ]:
%%script bash

sudo nextflow run /home/lucio/MEGA/linux/ziggy/git/chipseq_RE \
--input /mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/SS_ChIPseq_Lara_with_spikein_A1.csv \
--outdir /mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1_unique --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--spikein_fasta /mnt/datawk1/references/fasta/UCSC_hg19/hg19.fa \
--spikein_gtf /mnt/datawk1/references/annotations/UCSC_hg19.refGene/hg19.refGene.gtf \
--aligner star --reference_genome mm10 --spikein_genome hg19 --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com \
-profile docker -c /mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/unique.config -resume


importa su working pc

In [ ]:
rsync -av lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/Lara_chipseq_spikein_STARtests/random/star/mergedLibrary/F_F_K4me3_A.mLb.mkD.sorted.* /media/lucio/DATI/wkdir/bioinfo/deeptools_old/

#### 2024_04_Lara_chip ❌ TODEL questo esperimento era senza spikein ed e' stato rilanciato nella sezione analyse without spikein

random ✅

In [ ]:
cd ~/wkdir/analysis/2024_04_Lara_chip

sudo su

export NXF_VER=23.10.0
export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MjI2fS44YmY5NmZkODg5Nzg4NWU1Y2U1MWUzMTA0ZDUwNDVlMzI2NzJiNDFj

sudo nextflow run ./git/chipseq_RE \
--input ssin_2024_04_Lara_chip.csv \
--outdir ./nfout/random --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--spikein_fasta /mnt/datawk1/references/fasta/UCSC_hg19/hg19.fa \
--spikein_gtf /mnt/datawk1/references/annotations/UCSC_hg19.refGene/hg19.refGene.gtf \
--aligner star --reference_genome mm10 --spikein_genome hg19 --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com \
-profile docker -c /mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/random.config \
-resume -with-tower



unique ✅

In [ ]:
cd ~/wkdir/analysis/

mkdir 2024_04_Lara_chip

cd ./2024_04_Lara_chip

mkdir git
mkdir work
mkdir nfout

cd git
git clone git@github.com:lucidif/chipseq_RE.git

cd ..

sudo su

export NXF_VER=23.10.0
export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MjI2fS44YmY5NmZkODg5Nzg4NWU1Y2U1MWUzMTA0ZDUwNDVlMzI2NzJiNDFj

sudo nextflow run ./git/chipseq_RE \
--input ssin_2024_04_Lara_chip.csv \
--outdir ./nfout/unique --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--spikein_fasta /mnt/datawk1/references/fasta/UCSC_hg19/hg19.fa \
--spikein_gtf /mnt/datawk1/references/annotations/UCSC_hg19.refGene/hg19.refGene.gtf \
--aligner star --reference_genome mm10 --spikein_genome hg19 --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com \
-profile docker -c /mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/unique.config \
-resume -with-tower



### TODEL c) control results

#### Lara_ChIP_with_spikein_A1

In [ ]:
%%script bash

rsync --partial --progress -av -e ssh lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/multiqc/broadPeak/multiqc_report.html \
/media/lucio/caddyHDD/bioinformatics/wkdir/Lara_ChIP_with_spikein_A1/multiqc

rsync --partial --progress -av -e ssh lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/star/mergedLibrary/bigwig/deeptools \
/media/lucio/caddyHDD/bioinformatics/wkdir/Lara_ChIP_with_spikein_A1/deeptools

rsync --partial --progress -av -e ssh lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1_unique/multiqc/broadPeak/multiqc_report.html \
/media/lucio/caddyHDD/bioinformatics/wkdir/Lara_ChIP_with_spikein_A1_unique/multiqc

rsync --partial --progress -av -e ssh lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1_unique/star/mergedLibrary/bigwig/deeptools \
/media/lucio/caddyHDD/bioinformatics/wkdir/Lara_ChIP_with_spikein_A1_unique/deeptools




## 3. Analyze samples without spikein ✅

### run 1

In [ ]:
%%script bash
ssh house.mane.st -p 6022

#### a) download files on fisso

In [ ]:
%%script bash

prjpath="/media/lucio/bioData1/2.Datasets/Lara/ChIPseq/CHiPseq_MLL_Day4"
ss="${prjpath}/ChIPseq_day4_down_links.tsv"
out="${prjpath}/fastq"

mkdir ${prjpath}/fastq
mkdir ${prjpath}/fastq/ChIP_MLL_day4
mkdir ${prjpath}/fastq/Other

rsync --progress --append --inplace -av -e 'ssh -p 6022' \
/mnt/datawk1/data/Lara/CHiPseq_MLL_Day4/ChIPseq_day4_down_links.tsv \
lucio@house.mane.st:/media/lucio/bioData1/2.Datasets/Lara/ChIPseq/CHiPseq_MLL_Day4/

for i in {2..12} ; do filename=`cut -f 1 ${ss} | head -n ${i}  | tail -n 1` ; echo ${filename}; projectFolder=`cut -f 3 ${ss} | head -n ${i}  | tail -n 1`; echo "${projectFolder}" ; r1link=`cut -f 11 ${ss} | head -n ${i}  | tail -n 1` ; r2link=`cut -f 12 ${ss} | head -n ${i}  | tail -n 1` ; cd ${out}/${projectFolder} ;echo "curl -o ${filename}_1_fastq.gz $r1link" ;echo "curl -o ${filename}.2_fastq.gz $r2link" ; curl -o ${filename}.1_fastq.gz $r1link ; curl -o ${filename}_2_fastq.gz $r2link ; done

rsync --progress --append --inplace -av -e 'ssh -p 6022' \
/mnt/datawk1/data/Lara/CHiPseq_MLL_Day4/ChIPseq_day4_SS_nextflow_fisso.csv \
lucio@house.mane.st:/media/lucio/bioData1/2.Datasets/Lara/ChIPseq/CHiPseq_MLL_Day4/

rsync --progress --append --inplace -av -e 'ssh -p 6022' \
/mnt/datawk1/analysis/Lara/Lara_chipseq_spikein_STARtests/random.config \
lucio@house.mane.st:/media/lucio/bioData1/2.Datasets/Lara/ChIPseq/CHiPseq_MLL_Day4/



#### b) download reference on fisso

In [ ]:
  %%script bash

  rsync --progress --append --inplace -av -e 'ssh -p 6022' \
  /mnt/datawk1/references/UCSC_GRCm38_star_2_7_10a \
  lucio@house.mane.st:/media/lucio/bioData1/2.References/

  rsync --progress --append --inplace -av -e 'ssh -p 6022' \
  /mnt/datawk1/references/fasta/UCSC_GRCm38 \
  lucio@house.mane.st:/media/lucio/bioData1/2.References/fasta/

  rsync --progress --append --inplace -av -e 'ssh -p 6022'  \
  /mnt/datawk1/references/annotations/UCSC_mm10.refGene \
  lucio@house.mane.st:/media/lucio/bioData1/2.References/gtf


#### c) run random map analysis on fisso

In [ ]:
%%script bash

cd /media/lucio/wkssd_crucial

In [ ]:
%%script bash

ssh lucio@193.144.215.206

paths="/mnt/datawk1/data/Lara/CHiPseq_MLL_spikein/renamed"

samples="1_input Anti-GFP Anti-MLL1 Anti-MENIN Anti-RBPP1 Anti-WDR5 Anti-Mll1_A \
Anti-GFP_B Anti-RbBp5_B Anti-Menin_B Anti-Mll1_B Mll2_KO_Mll1_A Mll2_KO_Mll1_B \
Double_KO_RbBP5_A Double_KO_RbBP5_B"

for i in $samples ; do rsync -av --progress --append --inplace -av -e 'ssh -p 6022' \
${paths}/*_${i}*.fastq.gz lucio@house.mane.st:/media/lucio/bioData1/2.Datasets/Lara/ChIPseq/CHiPseq_MLL_Day0_1/ ; done

rsync -av --progress --append --inplace -av -e 'ssh -p 6022' \
/media/lucio/caddyHDD/bioinformatics/wkdir/ssin_nf-core_without_spikein_fisso.csv \
lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/Lara_CHiPseq_nospikein_A2/

ssh house.mane.st -p 6022
cd /media/lucio/wkssd_crucial/analysis/

sudo nextflow run ~/nfpipe/chipseq_RE \
--input /media/lucio/wkssd_crucial/analysis/Lara_CHiPseq_nospikein_A2/ssin_nf-core_without_spikein_fisso.csv \
--outdir /media/lucio/wkssd_crucial/analysis/Lara_CHiPseq_nospikein_A2 --read_length 150 \
--fasta /media/lucio/bioData1/2.References/fasta/UCSC_GRCm38/mm10.fa \
--gtf /media/lucio/bioData1/2.References/gtf/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c /media/lucio/bioData1/2.Datasets/Lara/ChIPseq/CHiPseq_MLL_Day4/random.config -resume




#### d) run unique on ziggy

In [ ]:
ssh lucio@193.144.215.206

rsync -av --progress --append --inplace -av \
/media/lucio/caddyHDD/bioinformatics/wkdir/ziggy_ssin_nf-core_without_spikein.csv \
lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2_unique/

cd /home/lucio/wkdir/

sudo nextflow run /home/lucio/MEGA/linux/ziggy/git/chipseq_RE \
--input /mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2_unique/ziggy_ssin_nf-core_without_spikein.csv \
--outdir /mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2_unique --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c /mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/unique.config -resume

#### e) upload file

In [ ]:
%%script bash

rsync --progress --append --inplace -av -e 'ssh -p 6022' \
lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/Lara_CHiPseq_MLL_Day4_random/multiqc/broadPeak/multiqc_report.html \
/mnt/datawk1/analysis/Lara/Lara_ChIPseq_MLL_Day4/

rsync --progress --append --inplace -av -e 'ssh -p 6022' \
lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/Lara_CHiPseq_MLL_Day4_random/pipeline_info \
/mnt/datawk1/analysis/Lara/Lara_ChIPseq_MLL_Day4/

rsync --progress --append --inplace -av -e 'ssh -p 6022' \
lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/Lara_CHiPseq_MLL_Day4_random/star/mergedLibrary/bigwig/deeptools \
/mnt/datawk1/analysis/Lara/Lara_ChIPseq_MLL_Day4/

rsync --partial --progress -av -e ssh lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2_unique/star/mergedLibrary/bigwig/deeptools/ \
/media/lucio/caddyHDD/bioinformatics/wkdir/Lara_CHiPseq_nospikein_A2_unique

rsync --progress --append --inplace -av -e 'ssh -p 6022' \
lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/Lara_CHiPseq_nospikein_A2/star/mergedLibrary/bigwig/deeptools \
/media/lucio/DATI/wkdir/bioinfo/Lara_day0_ChIPseq/nospikein_unique/

rsync --progress --append --inplace -av -e 'ssh -p 6022' \
lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/Lara_CHiPseq_nospikein_A2/star \
/mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2/









### run2 : h3k27ac were repeated (ad replacement of previous ones (spike-in run1))


#### random on ziggy

In [ ]:
cd ~/wkdir/analysis/

mkdir 2024_04_Lara_chip

cd ./2024_04_Lara_chip

mkdir git
mkdir work
mkdir nfout

cd git
git clone git@github.com:lucidif/chipseq_RE.git

cd ..

sudo su

    export NXF_VER=23.10.0
    export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MjI2fS44YmY5NmZkODg5Nzg4NWU1Y2U1MWUzMTA0ZDUwNDVlMzI2NzJiNDFj

nextflow run ./git/chipseq_RE \
--input --input ssin_2024_04_Lara_chip.csv \
--outdir ./nfout/random --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c random.config -resume -with-tower



#### unique on fisso

###### download on fisso

In [ ]:
%%script bash

rsync -av --progress --append --inplace -av -e 'ssh -p 6022' \
/mnt/datawk1/data/flid/HN00215148_2024_04_Lara_chip \
lucio@house.mane.st:/media/lucio/bioData1/2.Datasets/Lara/ChIPseq/

rsync -av --progress --append --inplace -av -e 'ssh -p 6022' \
/home/lucio/wkdir/analysis/2024_04_Lara_chip/fisso_ssin_2024_04_Lara_chip.csv \
lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/2024_04_Lara_chip

ssh house.mane.st -p 6022

prjpath="/media/lucio/wkssd_crucial/analysis/2024_04_Lara_chip"

mkdir ${prjpath}/git
mkdir ${prjpath}/nfout
mkdir ${prjpath}/work

cd ${prjpath}/git

git clone https://github.com/lucidif/chipseq_RE.git

/media/lucio/bioData1/2.Datasets/Lara/ChIPseq/HN00215148_2024_04_Lara_chip



###### compute main analysis

In [ ]:
%%script bash

ssh house.mane.st -p 6022

cd /media/lucio/wkssd_crucial/analysis/2024_04_Lara_chip

export NXF_VER=23.10.0
export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MjI2fS44YmY5NmZkODg5Nzg4NWU1Y2U1MWUzMTA0ZDUwNDVlMzI2NzJiNDFj

nextflow run ./git/chipseq_RE \
--input --input fisso_ssin_2024_04_Lara_chip.csv \
--outdir ./nfout/unique --read_length 150 \
--fasta /media/lucio/bioData1/2.References/fasta/UCSC_GRCm38/mm10.fa \
--gtf /media/lucio/bioData1/2.References/gtf/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c unique.config -resume -with-tower



###### import on ziggy results




In [ ]:
%%script bash

rsync -av --progress --append --inplace -av -e 'ssh -p 6022' \
lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/2024_04_Lara_chip/nfout/unique \
/home/lucio/wkdir/analysis/2024_04_Lara_chip/nfout/


### run3 D4


#### make environment

In [ ]:
%%script bash

sample="36 37 38 39 40"

for i in $sample ;  do echo "scp -r ssh lucio@193.144.215.241:volume1/Data_Just_In_NAS/Lucio/Data/flid/20240705_HN00219288/fastq/${i}_*.fastq.gz /mnt/datawk1/data/Lara/2024_07_Lara_chip/fastq/"
scp -r ssh lucio@193.144.215.241:volume1/Data_Just_In_NAS/Lucio/Data/flid/20240705_HN00219288/fastq/${i}_*.fastq.gz /mnt/datawk1/data/Lara/2024_07_Lara_chip/fastq/
done

mkdir /mnt/datawk1/analysis/Lara/2024_07_Lara_chip
mkdir /mnt/datawk1/analysis/Lara/2024_07_Lara_chip/git
mkdir /mnt/datawk1/analysis/Lara/2024_07_Lara_chip/nfout

cd /mnt/datawk1/analysis/Lara/2024_07_Lara_chip/git

git clone git@github.com:lucidif/chipseq_RE.git

cd /mnt/datawk1/analysis/Lara/2024_07_Lara_chip







#### run random

In [ ]:
%%script bash

sudo su

export NXF_VER=23.10.0
export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MjI2fS44YmY5NmZkODg5Nzg4NWU1Y2U1MWUzMTA0ZDUwNDVlMzI2NzJiNDFj

nextflow run ./git/chipseq_RE \
--input ssin_2024_07_Lara_chip.csv \
--outdir ./nfout/random --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c ./git/chipseq_RE/bin/random.config -resume -with-tower




### run 4

#### environment

In [ ]:
%%script bash

mkdir /media/lucio/bioData1/3.testFolder/Lara/240926_HN00226631_Lara_chip

cd /media/lucio/bioData1/3.testFolder/Lara/240926_HN00226631_Lara_chip

mkdir ./nfout
mkdir ./work
mkdir ./git

cd git

git clone git@github.com:lucidif/chipseq_RE.git

cd ..

cp


#### run random

In [ ]:
%%script bash

sudo su

export NXF_VER=23.10.0
export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MjI2fS44YmY5NmZkODg5Nzg4NWU1Y2U1MWUzMTA0ZDUwNDVlMzI2NzJiNDFj

nextflow run ./git/chipseq_RE \
--input /media/lucio/bioData1/2.Datasets/Lara/ChIPseq/240926_HN00226631_Lara_chip/ssin_2024_09_Lara_chip.csv \
--outdir ./nfout/random --read_length 150 \
--fasta /media/lucio/bioData1/2.References/fasta/UCSC_GRCm38/mm10.fa \
--gtf /media/lucio/bioData1/2.References/gtf/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c ./git/chipseq_RE/bin/random.config -resume -with-tower



### run 5 : 20240704_HN00220667_only_Lara_samples


#### environment


In [ ]:
%%script bash

mkdir /media/lucio/bioData1/3.testFolder/Lara/20240704_HN00220667_only_Lara_samples

cd /media/lucio/bioData1/3.testFolder/Lara/20240704_HN00220667_only_Lara_samples

mkdir ./nfout
mkdir ./work
mkdir ./git

cd git

git clone git@github.com:lucidif/chipseq_RE.git

cd ..

#### run random

In [ ]:
%%script bash

sudo su

export NXF_VER=23.10.0
export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MjI2fS44YmY5NmZkODg5Nzg4NWU1Y2U1MWUzMTA0ZDUwNDVlMzI2NzJiNDFj

nextflow run ./git/chipseq_RE \
--input /media/lucio/bioData1/3.testFolder/Lara/20240704_HN00220667_only_Lara_samples/ssin_20240704_HN00220667.csv \
--outdir ./nfout/random --read_length 150 \
--fasta /media/lucio/bioData1/2.References/fasta/UCSC_GRCm38/mm10.fa \
--gtf /media/lucio/bioData1/2.References/gtf/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c ./git/chipseq_RE/bin/random.config -resume -with-tower

### run 6 - all d4 of Lara_chipseq_nospikein_a2 and 240926_HN00226631 together on ziggy 

e' inesatto, in questa sezione ci sono solo i campioni runnati su fisso e che ho rianalizzato dsu ziggy. quelli che erano gia' presenti su ziggy non ci sono (vedi run 2024_07_Lara_chip)

#### make environment

In [ ]:
cd /mnt/datawk1/analysis/Lara/240926_chip_D4

#### run random

In [ ]:

sudo su

export NXF_VER=23.10.0
export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MjI2fS44YmY5NmZkODg5Nzg4NWU1Y2U1MWUzMTA0ZDUwNDVlMzI2NzJiNDFj

nextflow run ./git/chipseq_RE \
--input ssin_D4_ziggy.csv \
--outdir ./nfout/random --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable --skip_consensus_peaks \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c ./git/chipseq_RE/bin/random.config -resume -with-tower




### rerun missied HN00226631_HN00220667 chipseqs


In [ ]:
cd outs/HN00226631_HN00220667_chip


sudo su

export NXF_VER=23.10.0
export TOWER_ACCESS_TOKEN=eyJ0aWQiOiAxMzI4NX0uYzdjMjBiZjQ5MzYwMjY5MmNjYTIyMmQxNTQwNTUzOTM3ZTZjMGQxOQ==

nextflow run ../../git/chipseq_RE \
--input ../../sheets/nfss_HN00226631_HN00220667.csv \
--outdir ./nfout/random --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable --skip_consensus_peaks \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-c ../../git/chipseq_RE/bin/random.config -resume -with-tower

### run 7: 20260226_HN00264851

In [ ]:
%%bash

sh git/Lara_MLL2/bin/20260226_HN00264851_chipseq_all.sh

## 3.2 spikeinfree normalization strategies

Spike-in-free normalization For histone mark ChIP-seq experiments, normalization was performed using the ChIPseqSpikeInFree R package (v1.2.4), which provides a spike-in-free method to estimate scaling factors based on the internal properties of the enrichment profiles. This approach was adopted to overcome the eventual high variability observed in our initial normalization strategy, which involved mixing mouse chromatin with human HEK293 chromatin (90%:10% proportion).
Scaling factors (SF) were calculated for each sample using the ChIPseqSpikeInFree function, providing the corresponding BAM files and a metadata file containing group and antibody information. Only samples passing quality control (QC "pass") and with valid SF estimates were used for downstream analysis.
Signal track generation and averaging To generate normalized signal tracks (bigWig files), we applied a scaling factor defined as: 
Scale= Target Reads/(Library Size×SF)

 where Target Reads was set to 10^8 and Library Size was calculated as the number of properly paired fragments. BedGraph files were generated using bedtools genomecov with the -pc (paired-end) and -scale options, and subsequently converted to bigWig format using bedGraphToBigWig.
For visualization and comparative analyses, average bigWig signal tracks from biological replicates were generated using the bigwigAverage tool from deepTools (v3.5.5). Peak calling was performed on individual and merged replicates using MACS2 v2.2.7.1 with the --broad option.

In [ ]:
%%script bash

#run normalization script
main_folder="${PWD}"

# test script
#sh git/Lara_MLL2/bin/quant_spike_free_normalizations.sh

#execution script
#sh git/Lara_MLL2/bin/Lara_spikein_free_normalization.sh

#all old samples normalization
sh git/Lara_MLL2/bin/all_samples_spikein_normalization.sh

#new run normalization
sh git/Lara_MLL2/bin/spikeinfree_norm_20260226_HN00264851.sh

#separate normalization for all samples
#generate divided samples

bash git/Lara_MLL2/bin/split_spikeinfree_samples.sh

sudo -s

bash git/Lara_MLL2/bin/spikeinfree_Dsplit_allSamples.sh

bash git/Lara_MLL2/bin/spikeinfree_raname_samples.sh

sudo bash git/Lara_MLL2/bin/spikeinfree_averaging.sh





## 4. macs peakcalling ✅

### a) prepare Environment

*   enter in workspace
*   make folder
*   copy git




In [ ]:
%%script bash

sudo su

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream

mkdir nfout
mkdir git
mkdir work

cd git
git clone git@github.com:lucidif/chipseq_downstream_macs.git

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream

mkdir in/240926_chip_D4_bam

cp /mnt/datawk1/analysis/Lara/240926_chip_D4/nfout/random/star/mergedLibrary/D4WTH3K27ac*.mLb.mkD.sorted.bam* in/240926_chip_D4_bam/
cp /mnt/datawk1/analysis/Lara/240926_chip_D4/nfout/random/star/mergedLibrary/D4WTMLL2B.mLb.mkD.sorted.bam* in/240926_chip_D4_bam/
cp /mnt/datawk1/analysis/Lara/240926_chip_D4/nfout/random/star/mergedLibrary/240219_KO_D4_input.mLb.mkD.sorted.bam* in/240926_chip_D4_bam/


### b) run analysis ✅

#### i. MACS peak calling on anti-GFP ✅

compare anti-gfp replicates

In [ ]:

sudo docker run -v /mnt/datawk1/analysis/Lara:/mnt/datawk1/analysis/Lara quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 multiBamSummary bins --bamfiles /mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2_unique/star/mergedLibrary/Anti-GFP.mLb.mkD.sorted.bam /mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2_unique/star/mergedLibrary/Anti-GFP_B.mLb.mkD.sorted.bam -o /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/correlations/correlation.npz

sudo docker run -v /mnt/datawk1/analysis/Lara:/mnt/datawk1/analysis/Lara quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotCorrelation -in /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/correlations/correlation.npz -c spearman -p scatterplot -o /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/correlations/correlation_heatmap.png

sudo docker run -v /mnt/datawk1/analysis/Lara:/mnt/datawk1/analysis/Lara quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 multiBamSummary bins --bamfiles /mnt/datawk1/analysis/Lara/240926_chip_D4/nfout/random/star/mergedLibrary/D4WTH3K27acA.mLb.mkD.sorted.bam /mnt/datawk1/analysis/Lara/240926_chip_D4/nfout/random/star/mergedLibrary/D4WTH3K27acB.mLb.mkD.sorted.bam -o /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/correlations/D4WTH3K27ac_correlation.npz

sudo docker run -v /mnt/datawk1/analysis/Lara:/mnt/datawk1/analysis/Lara quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotCorrelation -in /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/correlations/D4WTH3K27ac_correlation.npz -c spearman -p scatterplot -o /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/correlations/D4WTH3K27ac_correlation_plot.png



standard (broad 0.1) ✅

In [ ]:
%%script bash

sudo nextflow run ./git/chipseq_downstream_macs \
--input --input ss_in.csv \
--outdir ./nfout/broad_1 --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-resume

broad fdr 0.05 ✅

In [ ]:
%%script bash

sudo nextflow run ./git/chipseq_downstream_macs \
--input --input ss_in.csv \
--outdir ./nfout/broad_0_5 --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 \
--broad_cutoff 0.05 \
--email difilippolucio@gmail.com -profile docker \
-resume


narrow 0.05 ✅

In [ ]:
%%script bash

sudo nextflow run ./git/chipseq_downstream_macs \
--input --input ss_in.csv \
--outdir ./nfout/narrow_0_5 --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 \
--narrow_peak true \
--email difilippolucio@gmail.com -profile docker \
-resume


narrow 0.06 ✅

In [ ]:
%%script bash

sudo nextflow run ./git/chipseq_downstream_macs \
--input --input ss_in.csv \
--outdir ./nfout/narrow_0_6 --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 \
--narrow_peak true \
--macs_fdr 0.06 \
--email difilippolucio@gmail.com -profile docker \
-resume

narrow 0.01 ✅

In [ ]:
%%script bash

sudo nextflow run ./git/chipseq_downstream_macs \
--input --input ss_in.csv \
--outdir ./nfout/narrow_0_1 --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 \
--narrow_peak true \
--macs_fdr 0.01 \
--email difilippolucio@gmail.com -profile docker \
-resume

#### ii. filter results by fold coverage and pval ✅


narrow peaks 0.01 ✅

In [ ]:
%%script bash

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/narrow_0_1/star/mergedLibrary/macs2/narrowPeak/Anti-GFP.mLb.mkD.sorted_peaks.narrowPeak \
./nfout/narrow_0_1/star/mergedLibrary/macs2/narrowPeak/ "narrowPeak" 2

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/narrow_0_1/star/mergedLibrary/macs2/narrowPeak/Anti-GFP.mLb.mkD.sorted_peaks.narrowPeak \
./nfout/narrow_0_1/star/mergedLibrary/macs2/narrowPeak/ "narrowPeak" 3

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/narrow_0_1/star/mergedLibrary/macs2/narrowPeak/Anti-GFP.mLb.mkD.sorted_peaks.narrowPeak \
./nfout/narrow_0_1/star/mergedLibrary/macs2/narrowPeak/ "narrowPeak" 4

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/narrow_0_1/star/mergedLibrary/macs2/narrowPeak/Anti-GFP.mLb.mkD.sorted_peaks.narrowPeak \
./nfout/narrow_0_1/star/mergedLibrary/macs2/narrowPeak/ "narrowPeak" 6

narrow peaks 0.05 ✅

In [ ]:
./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/Anti-GFP.mLb.mkD.sorted_peaks.narrowPeak \
./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/ "narrowPeak" 3

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/Anti-GFP.mLb.mkD.sorted_peaks.narrowPeak \
./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/ "narrowPeak" 4

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/Anti-GFP.mLb.mkD.sorted_peaks.narrowPeak \
./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/ "narrowPeak" 6

./git/chipseq_downstream_macs/bin/filter_qval_fold.r ./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/Anti-GFP.mLb.mkD.sorted_peaks.narrowPeak \
./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/ "narrowPeak" 0.01 3

./git/chipseq_downstream_macs/bin/filter_qval_fold.r ./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/Anti-GFP.mLb.mkD.sorted_peaks.narrowPeak \
./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/ "narrowPeak" 0.01 4

./git/chipseq_downstream_macs/bin/filter_qval_fold.r ./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/Anti-GFP.mLb.mkD.sorted_peaks.narrowPeak \
./nfout/narrow_0_5/star/mergedLibrary/macs2/narrowPeak/ "narrowPeak" 0.01 6


broad peaks standard (0.1) ✅

In [ ]:
./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.broadPeak \
./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 2

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.broadPeak \
./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 4

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.broadPeak \
./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 3

./git/chipseq_downstream_macs/bin/filter_qval_fold.r ./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.broadPeak \
./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 0.05 2

./git/chipseq_downstream_macs/bin/filter_qval_fold.r ./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.broadPeak \
./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 0.05 3

./git/chipseq_downstream_macs/bin/filter_qval_fold.r ./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.broadPeak \
./nfout/broad_1/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 0.05 4

broad peaks stringent (0.05) ✅

In [ ]:
./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/broad_0_5/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.broadPeak \
./nfout/broad_0_5/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 2

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/broad_0_5/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.broadPeak \
./nfout/broad_0_5/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 4

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/broad_0_5/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.broadPeak \
./nfout/broad_0_5/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 3


selected : threshold 3 broad peak

#### iii. run peak calling and filtering on other samples with same parameters NOTE : questi peaks non sono stati utilizzanti. Per tutti gli altri picchi (eccetto anti-GFP) sono stati usati i bigwig prodotti dalla pipeline di chipseqRE (broad 0.05 ? controlla)

In [ ]:
%%script bash

sudo su

export NXF_VER=23.10.0
export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MjI2fS44YmY5NmZkODg5Nzg4NWU1Y2U1MWUzMTA0ZDUwNDVlMzI2NzJiNDFj

#broad peaks

nextflow run ./git/chipseq_downstream_macs \
--input MACS_peakcalling_a1_ss.csv \
--outdir ./nfout/MACS_peakcalling_a1 --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 --email difilippolucio@gmail.com -profile docker \
-resume -with-tower

#narrow peaks

nextflow run ./git/chipseq_downstream_macs \
--input MACS_peakcalling_a1_ss.csv \
--outdir ./nfout/MACS_peakcalling_a2_narrow --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 \
--narrow_peak true \
--email difilippolucio@gmail.com -profile docker \
-resume -with-tower




#### macs2 peak calling on D4 h3k27ac 

In [ ]:
%%script bash

sudo su

export NXF_VER=23.10.0
export TOWER_ACCESS_TOKEN=eyJ0aWQiOiAxMTE1NH0uM2JhMzRjNmQ2NmE4M2FlNGNkYzE4YzEwOTRjMWEwYmVhNTljNTk2Mg==

sudo nextflow run ./git/chipseq_downstream_macs \
--input --input D4_ss_in_h3k27ac.csv \
--outdir ./nfout/D4_H3K27ac_broad_0_5 --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 \
--broad_cutoff 0.05 \
--email difilippolucio@gmail.com -profile docker \
-resume


./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/D4_H3K27ac_broad_0_5/star/mergedLibrary/macs2/broadPeak/D4WTH3K27acA.mLb.mkD_peaks.broadPeak \
./nfout/D4_H3K27ac_broad_0_5/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 3

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/D4_H3K27ac_broad_0_5/star/mergedLibrary/macs2/broadPeak/D4WTH3K27acA.mLb.mkD_peaks.broadPeak \
./nfout/D4_H3K27ac_broad_0_5/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 5

sed 's/ /\t/g' nfout/D4_H3K27ac_broad_0_5/star/mergedLibrary/macs2/broadPeak/D4WTH3K27acA.mLb.mkD_peaks.threshold_3.broadPeak | cut -f 1-6 > ./D4WTH3K27acA_macs_coordinate.bed

sed 's/ /\t/g' nfout/D4_H3K27ac_broad_0_5/star/mergedLibrary/macs2/broadPeak/D4WTH3K27acA.mLb.mkD_peaks.threshold_5.broadPeak | cut -f 1-6 > ./thr5_D4WTH3K27acA_macs_coordinate.bed


#### macs peak calling d4 MLL2

In [ ]:
%%script bash


sudo docker run -v /mnt/datawk1/analysis/Lara:/mnt/datawk1/analysis/Lara quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 multiBamSummary bins --bamfiles /mnt/datawk1/analysis/Lara/240926_chip_D4/nfout/random/star/mergedLibrary/D4WTMLL2A.mLb.mkD.sorted.bam /mnt/datawk1/analysis/Lara/240926_chip_D4/nfout/random/star/mergedLibrary/D4WTMLL2B.mLb.mkD.sorted.bam -o /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/correlations/D4WTMLL2_correlation.npz

sudo docker run -v /mnt/datawk1/analysis/Lara:/mnt/datawk1/analysis/Lara quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotCorrelation -in /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/correlations/D4WTMLL2_correlation.npz -c spearman -p scatterplot -o /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/correlations/D4WTMLL2_correlation_heatmap.png

#rep A seems lot noisy instead rep B seems a good one

sudo nextflow run ./git/chipseq_downstream_macs \
--input --input ss_in_d4mll2.csv \
--outdir ./nfout/D4_MLL2_broad_0_5 --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 \
--broad_cutoff 0.05 \
-profile docker \
-resume

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/D4_MLL2_broad_0_5/star/mergedLibrary/macs2/broadPeak/D4WTMLL2B_peaks.broadPeak \
./nfout/D4_MLL2_broad_0_5/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 3

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/D4_MLL2_broad_0_5/star/mergedLibrary/macs2/broadPeak/D4WTMLL2B_peaks.broadPeak \
./nfout/D4_MLL2_broad_0_5/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 2

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/D4_MLL2_broad_0_5/star/mergedLibrary/macs2/broadPeak/D4WTMLL2B_peaks.broadPeak \
./nfout/D4_MLL2_broad_0_5/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 1

sudo nextflow run ./git/chipseq_downstream_macs \
--input --input ss_in_d4mll2.csv \
--outdir ./nfout/D4_MLL2_broad_std --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 \
--broad_cutoff 0.1 \
-profile docker \
-resume

#D4WTMLL2B_peaks.threshold_3.broadPeak

./git/chipseq_downstream_macs/bin/filter_fold_enrichment.r ./nfout/D4_MLL2_broad_std/star/mergedLibrary/macs2/broadPeak/D4WTMLL2B_peaks.broadPeak \
./nfout/D4_MLL2_broad_std/star/mergedLibrary/macs2/broadPeak/ "broadPeak" 3

d4mll2_macs_peaks="./nfout/D4_MLL2_broad_std/star/mergedLibrary/macs2/broadPeak/D4WTMLL2B_peaks.threshold_3.broadPeak"
sed 's/ /\t/g' ${d4mll2_macs_peaks} | cut -f 1-6 > ./nfout/D4_MLL2_broad_std/star/mergedLibrary/macs2/broadPeak/D4_MLL2B_broad_std_thr3.bed



#narrow 0.05

sudo nextflow run ./git/chipseq_downstream_macs \
--input ss_in_d4mll2.csv \
--outdir ./nfout/D4_MLL2_narrow_0_5 --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 \
--narrow_peak true \
--email difilippolucio@gmail.com -profile docker \
-resume

d4mll2_macs_peaks="./nfout/D4_MLL2_narrow_0_5/star/mergedLibrary/macs2/narrowPeak/D4WTMLL2B_peaks.narrowPeak"
sed 's/ /\t/g' ${d4mll2_macs_peaks} | cut -f 1-6 > ./nfout/D4_MLL2_narrow_0_5/star/mergedLibrary/macs2/narrowPeak/D4_MLL2_narrow_0_5.bed


#narrow 0.1

sudo nextflow run ./git/chipseq_downstream_macs \
--input ss_in_d4mll2.csv \
--outdir ./nfout/D4_MLL2_narrow_1 --read_length 150 \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--gtf /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
--aligner star --filters_disable \
--effectiveGenomeSize 2494787188 \
--narrow_peak true \
--macs_fdr 0.1 \
--email difilippolucio@gmail.com -profile docker \
-resume




## 5. deeptools heatmap plot ✅

### prepare environment

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream

mkdir ./otherouts
cd ./otherouts

mkdir deeptools_heatmaps
cd deeptools_heatmaps
mkdir tmp

macs_peaks="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/broad_0_5/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.threshold_3.broadPeak"
sed 's/ /\t/g' ${macs_peaks} | cut -f 1-6 > ./coordinate.bed


### average bigwig ✅

#### average first files ✅

In [ ]:
%%script bash

sudo su

bash ./git/chipseq_downstream_macs/bin/parallel_averageBigwig.sh \
"/mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/star/mergedLibrary/bigwig/deeptools/" \
"Double_KO_K27me3_ Double_KO_K4me1_ Double_KO_K4me2_ Double_KO_K4me3_ FC_FC_K27me3_ FC_FC_K4me1_ FC_FC_K4me2_ FC_FC_K4me3_ F_F_K27me3_ F_F_K4me1_ F_F_K4me2_ F_F_K4me3_ KO_D4_K27ac KO_D4_K27me3 KO_D4_K4me2 KO_D4_K4me3 Mll1-KO_K27me3 Mll1-KO_K4me1 Mll1-KO_K4me2 Mll1-KO_K4me3" \
/mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/star/mergedLibrary/bigwig/deeptools/average

cd /mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2/star/mergedLibrary/bigwig/deeptools/

cp /mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2/star/mergedLibrary/bigwig/deeptools/Anti-MENIN
.bigWig /mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2/star/mergedLibrary/bigwig/deeptools/Anti-Menin.bigWig

cp /mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2/star/mergedLibrary/bigwig/deeptools/Anti-RBPP1.bigWig \
/mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2/star/mergedLibrary/bigwig/deeptools/Anti-RbBp5_A.bigWig

bash ./git/chipseq_downstream_macs/bin/parallel_averageBigwig.sh \
"/mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2/star/mergedLibrary/bigwig/deeptools/" \
"Anti-GFP Anti-Menin  Anti-Mll1_ Double_KO_RbBP5_ Mll2_KO_Mll1_ Anti-RbBp5_" \
/mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2/star/mergedLibrary/bigwig/deeptools/average

cp /mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/star/mergedLibrary/bigwig/deeptools/average/* ./tmp/
cp /mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1
cp /mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2/star/mergedLibrary/bigwig/deeptools/average/* ./tmp/
cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/broad_0_5/star/mergedLibrary/bigwig/deeptools/Anti-GFP.mLb.mkD.sorted.bigWig ./tmp/



for the k27ac the experiment are repeated and for now we have only one replicate for this reason their are used without avarage. The file are here:

In [ ]:
ls ~/wkdir/analysis/2024_04_Lara_chip/nfout/random/star/mergedLibrary/bigwig/deeptools/

#### averange of latest samples ✅

##### envirnoment preparation :

In [ ]:
%%script bash

#from enlitebook

#cd /mnt/d/bioinfo/wkdir/Lara/test_chipseq_dowstream/
cd /media/lucio/external.wk/bioinfo/wkdir/Lara/test_chipseq_dowstream/in/samples_bigwig/


mkdir git_elite

git clone git@github.com:lucidif/chipseq_downstream_macs.git

cd ..

cd ./in/samples_bigwig/

ls

# 1_input.bigWig D0_KO_RING1B_A.bigWig D0_WT_H3K27ac_B.bigWig  D0_input_A.bigWig Mll1KO_H3K27ac_C.bigWig 220907_1_input_1.bigWig D0_KO_RING1B_B.bigWig   D0_WT_H3K9me3_B.bigWig D0_input_B.bigWig Mll2KO_H3K27ac_C.bigWig D0_Double-KO_H3K27ac_B.bigWig D0_Mll1-KO_H3K27ac_B.bigWig D0_WT_RING1B_A.bigWig DoubleKO_H3K27ac_C.bigWig WT_H3K27ac_C.bigWig D0_KO_H3K9me3_B.bigWig D0_Mll2-KO_H3K27ac_B.bigWig D0_WT_RING1B_B.bigWig DoubleKO_H3K9me3_A.bigWig WT_H3K9me3_A.bigWig

#D0_KO_RING1B_A.bigWig
#D0_KO_RING1B_B.bigWig
# ok

#D0_WT_RING1B_A.bigWig
#D0_WT_RING1B_B.bigWig
# ok

#WT_H3K9me3_A.bigWig
#D0_WT_H3K9me3_B.bigWig
cp WT_H3K9me3_A.bigWig D0_WT_H3K9me3_A.bigWig


#D0_KO_H3K9me3_B.bigWig
#DoubleKO_H3K9me3_A.bigWig
cp DoubleKO_H3K9me3_A.bigWig D0_KO_H3K9me3_A.bigWig

#D0_WT_H3K27ac_B.bigWig
#WT_H3K27ac_C.bigWig
cp WT_H3K27ac_C.bigWig D0_WT_H3K27ac_C.bigWig

#Mll1KO_H3K27ac_C.bigWig
#D0_Mll1-KO_H3K27ac_B.bigWig
cp D0_Mll1-KO_H3K27ac_B.bigWig D0_Mll1KO_H3K27ac_B.bigWig
cp Mll1KO_H3K27ac_C.bigWig D0_Mll1KO_H3K27ac_C.bigWig

#D0_Mll2-KO_H3K27ac_B.bigWig
#Mll2KO_H3K27ac_C.bigWig
cp D0_Mll2-KO_H3K27ac_B.bigWig D0_Mll2KO_H3K27ac_B.bigWig
cp Mll2KO_H3K27ac_C.bigWig D0_Mll2KO_H3K27ac_C.bigWig

#D0_Double-KO_H3K27ac_B.bigWig
#DoubleKO_H3K27ac_C.bigWig
cp D0_Double-KO_H3K27ac_B.bigWig D0_DoubleKO_H3K27ac_B.bigWig
cp DoubleKO_H3K27ac_C.bigWig D0_DoubleKO_H3K27ac_C.bigWig


#D0_input_B.bigWig
#D0_input_A.bigWig

#220907_1_input_1.bigWig not run this


sudo apt install parallel

docker pull quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0

mkdir /mnt/d/bioinfo/wkdir/Lara/chipseq_dowstream_2/otherouts/parallel_averageBigwig

#D4

mkdir ./in/samples_bigwig/d4

cp ../240926_chip_D4/nfout/random/star/mergedLibrary/bigwig/deeptools/*.bigWig ./in/samples_bigwig/d4/
cp ../2024_07_Lara_chip/nfout/random/star/mergedLibrary/bigwig/deeptools/*.bigWig ./in/samples_bigwig/d4/

#rename samples 

name="D4_DKO_H3K27ac"
fq1="240219_KO_D4_K27ac.bigWig" ; fq2="D4DoubKOK27acB.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig"
mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig
mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig


name="D4_DKO_H3K27me3"
fq1="240219_KO_D4_K27me3.bigWig" ; fq2="D4DoubKOK27me3B.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig"
mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig
mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig

#rerun this
name="D4_DKO_H3K4me2"
fq1="240219_KO_D4_K4me2.bigWig" ; fq2="D4DoubKOK4me2B.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig"
mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig
mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig

#rerun this
name="D4_DKO_H3K4me3"
fq1="240219_KO_D4_K4me3.bigWig" ; fq2="D4DoubKOK4me3B.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig"
mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig
mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig


name="D4_WT_H3K27ac"
fq1="D4WTH3K27acA.bigWig" ; fq2="D4WTH3K27acB.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig"
mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig
mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig


name="D4_WT_H3K27me3"
fq1="D4_WT_H3K27me3_A.bigWig" ; fq2="D4WTH3K27me3B.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig"
mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig
mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig

#rerun this

cp ./in/samples_bigwig/d4/D4_WT_H3K4me3_A.bigWig ./in/samples_bigwig/d4/old_D4_WT_H3K4me3_A.bigWig
cp ./in/samples_bigwig/d4/D4_WT_H3K4me2_A.bigWig ./in/samples_bigwig/d4/old_D4_WT_H3K4me2_A.bigWig

name="D4_WT_H3K4me2"
fq1="D4_WT_H3K4me3_A.bigWig" ; fq2="D4WTH3K4me2B.bigWig"
echo "mv ./in/samples_bigwig/d4/old_${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig"
mv ./in/samples_bigwig/d4/old_${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig
mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig

#rerun this
name="D4_WT_H3K4me3"
fq1="D4_WT_H3K4me2_A.bigWig" ; fq2="D4_WT_H3K4me3_B.bigWig"
echo "mv ./in/samples_bigwig/d4/old_${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig"
mv ./in/samples_bigwig/d4/old_${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig
mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig


name="D4_WT_MLL2"
fq1="D4WTMLL2A.bigWig" ; fq2="D4WTMLL2B.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig"
mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig
mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig


name="D4_WT_RbBP5"
fq1="D4WTRbBP5A.bigWig" ; fq2="D4WTRbBP5B.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig"
echo "mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig"
mv ./in/samples_bigwig/d4/${fq1} ./in/samples_bigwig/d4/${name}_A.bigWig
mv ./in/samples_bigwig/d4/${fq2} ./in/samples_bigwig/d4/${name}_B.bigWig


list_names="D4_DKO_H3K27ac D4_DKO_H3K27me3 D4_DKO_H3K4me2 D4_DKO_H3K4me3 D4_WT_H3K27ac D4_WT_H3K27me3 D4_WT_H3K4me2 D4_WT_H3K4me3 D4_WT_MLL2 D4_WT_RbBP5"

for l in $list_names ; do ls ./in/samples_bigwig/d4/${l}_*.bigWig ; done



##### do average

In [ ]:
%%script bash

sudo su

pattern="D0_KO_RING1B_  D0_WT_RING1B_ D0_WT_H3K9me3_ D0_KO_H3K9me3_ D0_WT_H3K27ac_ D0_Mll1KO_H3K27ac_ D0_Mll2KO_H3K27ac_ D0_DoubleKO_H3K27ac_"

bash ./git_elite/chipseq_downstream_macs/bin/parallel_averageBigwig.sh "/mnt/d/bioinfo/wkdir/Lara/chipseq_dowstream_2/in/samples_bigwig/" "D0_KO_RING1B_  D0_WT_RING1B_ D0_WT_H3K9me3_ D0_KO_H3K9me3_ D0_WT_H3K27ac_ D0_Mll1KO_H3K27ac_ D0_Mll2KO_H3K27ac_ D0_DoubleKO_H3K27ac_" /mnt/d/bioinfo/wkdir/Lara/test_chipseq_dowstream/out/parallel_averageBigwig/

rsync -av /mnt/d/bioinfo/wkdir/Lara/test_chipseq_dowstream/out/parallel_averageBigwig lucio@193.144.215.206:/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig/

##test , try unparallel way

sudo docker run -v /media/lucio/external.wk:/media/lucio/external.wk quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 bigwigAverage -b /media/lucio/external.wk/bioinfo/wkdir/Lara/test_chipseq_dowstream/in/samples_bigwig/D0_WT_RING1B_A.bigWig /media/lucio/external.wk/bioinfo/wkdir/Lara/test_chipseq_dowstream/in/samples_bigwig/D0_WT_RING1B_B.bigWig -o /media/lucio/external.wk/bioinfo/wkdir/Lara/test_chipseq_dowstream/out/unparallel_averageBigwig/D0_WT_RING1B_av.bigWig



#### average D4

In [ ]:
%%script bash

mkdir otherouts/parallel_averageBigwig_D4
cd otherouts/parallel_averageBigwia
sudo su

pattern="D4_DKO_H3K27ac_ D4_DKO_H3K27me3_ D4_WT_H3K27ac_ D4_WT_H3K27me3_ D4_WT_MLL2_ D4_WT_RbBP5_"

bash /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/git/chipseq_downstream_macs/bin/averageBigwig.sh "/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/in/samples_bigwig/d4" "D4_DKO_H3K27ac_ D4_DKO_H3K27me3_ D4_WT_H3K27ac_ D4_WT_H3K27me3_ D4_WT_MLL2_ D4_WT_RbBP5_" "/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4"

pattern2="D4_WT_H3K4me2_ D4_WT_H3K4me3_ D4_DKO_H3K4me2_ D4_DKO_H3K4me3_"

bash /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/git/chipseq_downstream_macs/bin/averageBigwig.sh "/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/in/samples_bigwig/d4" "D4_WT_H3K4me2_ D4_WT_H3K4me3_ D4_DKO_H3K4me2_ D4_DKO_H3K4me3_" "/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4"

### TODEL_create distal/proximal NMIs (OLD OLD)

In [ ]:
%%R

source("./git/chipseq_downstream_macs/bin/Regions_Removing_Proximal_ToTSS.R")

nmis_peaks<-regions_removingProximal(regions_bedfile = "./otherouts/deeptools_heatmaps/coordinate.bed" ,
                                     TSS_bedfile = "./otherouts/deeptools_heatmaps/NMIs_mm10_mESC_afterLiftover_from_mm9_to_mm10.bed" ,
                                     kbDistance=5)

write.table(nmis_peaks$distal,"./otherouts/deeptools_heatmaps/distal_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")

write.table(nmis_peaks$proximal,"./otherouts/deeptools_heatmaps/proximal_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")


noinpeak<- regions_removingProximal(regions_bedfile = "./otherouts/deeptools_heatmaps/coordinate.bed" ,
                                     TSS_bedfile = "./otherouts/deeptools_heatmaps/NMIs_mm10_mESC_afterLiftover_from_mm9_to_mm10.bed" ,
                                     kbDistance=0)

write.table(noinpeak$distal,"./otherouts/deeptools_heatmaps/noinpeak_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")
write.table(noinpeak$proximal,"./otherouts/deeptools_heatmaps/inpeak_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")


proximal_noinpeak <-regions_removingProximal(regions_bedfile = "./otherouts/deeptools_heatmaps/noinpeak_NMIs_preaks.bed" ,
                                     TSS_bedfile = "./otherouts/deeptools_heatmaps/NMIs_mm10_mESC_afterLiftover_from_mm9_to_mm10.bed" ,
                                     kbDistance=5)

write.table(proximal_noinpeak$proximal,"./otherouts/deeptools_heatmaps/noinpeak_proximal_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")


proximal_farnear_noinpeak <- regions_removingProximal(regions_bedfile = "./otherouts/deeptools_heatmaps/noinpeak_proximal_NMIs_preaks.bed" ,
                                     TSS_bedfile = "./otherouts/deeptools_heatmaps/NMIs_mm10_mESC_afterLiftover_from_mm9_to_mm10.bed" ,
                                     kbDistance=2.5)

write.table(proximal_farnear_noinpeak$distal,"./otherouts/deeptools_heatmaps/far_proximal_noinpeaks_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")

write.table(proximal_farnear_noinpeak$proximal,"./otherouts/deeptools_heatmaps/near_proximal_noinpeaks_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")

distal_farnear<-regions_removingProximal(regions_bedfile = "./otherouts/deeptools_heatmaps/distal_NMIs_preaks.bed" ,
                                     TSS_bedfile = "./otherouts/deeptools_heatmaps/NMIs_mm10_mESC_afterLiftover_from_mm9_to_mm10.bed" ,
                                     kbDistance=30)

proximal_farnear<-regions_removingProximal(regions_bedfile = "./otherouts/deeptools_heatmaps/proximal_NMIs_preaks.bed" ,
                                     TSS_bedfile = "./otherouts/deeptools_heatmaps/NMIs_mm10_mESC_afterLiftover_from_mm9_to_mm10.bed" ,
                                     kbDistance=2.5)

write.table(distal_farnear$distal,"./otherouts/deeptools_heatmaps/far_distal_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")

write.table(distal_farnear$proximal,"./otherouts/deeptools_heatmaps/near_distal_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")

write.table(proximal_farnear$distal,"./otherouts/deeptools_heatmaps/far_proximal_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")

write.table(proximal_farnear$proximal,"./otherouts/deeptools_heatmaps/near_proximal_NMIs_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")







### create all tracks main heatmap (e' necessario ???) ◼

In [ ]:
%%script bash

cd /mnt/d/bioinfo/wkdir/Lara/chipseq_dowstream_2/
#cd /mnt/d/bioinfo/wkdir/Lara/

cd ./otherouts

mkdir deeptools_heatmaps



##### TODEL old one

In [ ]:
%%script bash

mkdir ./otherouts
cd ./otherouts

mkdir deeptools_heatmaps
cd deeptools_heatmaps

cp /mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1/star/mergedLibrary/bigwig/deeptools/average/* ./
cp /mnt/datawk1/analysis/Lara/Lara_ChIP_with_spikein_A1
cp /mnt/datawk1/analysis/Lara/Lara_CHiPseq_nospikein_A2/star/mergedLibrary/bigwig/deeptools/average/* ./
cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/broad_0_5/star/mergedLibrary/bigwig/deeptools/Anti-GFP.mLb.mkD.sorted.bigWig ./



outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"
inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"

bedir=$(dirname "$bedfile")

samples="Anti-GFP.mLb.mkD.sorted.bigWig Anti-GFP_average.bw \
FC_FC_K27me3_average.bw  KO_D4_K27me3_average.bw \
Anti-Menin_average.bw FC_FC_K4me1_average.bw KO_D4_K4me2_average.bw \
Anti-Mll1_average.bw FC_FC_K4me2_average.bw KO_D4_K4me3_average.bw \
Anti-RbBp5_average.bw FC_FC_K4me3_average.bw Mll1-KO_K27me3_average.bw \
Double_KO_K27me3_average.bw  F_F_K27me3_average.bw \
Mll1-KO_K4me1_average.bw Double_KO_K4me1_average.bw F_F_K4me1_average.bw \
Mll1-KO_K4me2_average.bw Double_KO_K4me2_average.bw F_F_K4me2_average.bw \
Mll1-KO_K4me3_average.bw Double_KO_K4me3_average.bw F_F_K4me3_average.bw \
Mll2_KO_Mll1_average.bw Double_KO_RbBP5_average.bw KO_D4_K27ac_average.bw"

../../git/chipseq_downstream_macs/bin/deeptools_plotHeatmap.sh \
"Anti-GFP_average.bw Anti-Mll1_average.bw Anti-RbBp5_average.bw Anti-Menin_average.bw Mll2_KO_Mll1_average.bw Double_KO_RbBP5_average.bw" \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/broad_0_5/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.threshold_3.broadPeak \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/ \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/ "first_"


panel1="Anti-GFP_average.bw Anti-Menin_average.bw Anti-Mll1_average.bw Mll2_KO_Mll1_average.bw Anti-RbBp5_average.bw Double_KO_RbBP5_average.bw"
thrsholds="10 10 10 10 15 15"
outname="panel1"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel1} -R ${outpath}/coordinate.bed \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--zMax ${thrsholds} --missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--outFileName $outpath/${outname}_plotHeatmap.pdf --outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab


panel2="WT_H3K27ac_C.bigWig Mll2KO_H3K27ac_C.bigWig Mll1KO_H3K27ac_C.bigWig DoubleKO_H3K27ac_C.bigWig \
F_F_K27me3_average.bw FC_FC_K27me3_average.bw Mll1-KO_K27me3_average.bw Double_KO_K27me3_average.bw \
F_F_K4me3_average.bw FC_FC_K4me3_average.bw Mll1-KO_K4me3_average.bw Double_KO_K4me3_average.bw \
F_F_K4me2_average.bw FC_FC_K4me2_average.bw Mll1-KO_K4me2_average.bw Double_KO_K4me2_average.bw \
F_F_K4me1_average.bw FC_FC_K4me1_average.bw Mll1-KO_K4me1_average.bw Double_KO_K4me1_average.bw \
"
outname="panel2"
thrsholds="10 10 10 10 5 5 5 5 30 30 30 30 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel2} -R ${outpath}/coordinate.bed \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--zMax ${thrsholds} --missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 1 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

#all tracks together

panel3="Anti-GFP_average.bw Anti-Menin_average.bw Anti-Mll1_average.bw Mll2_KO_Mll1_average.bw Anti-RbBp5_average.bw Double_KO_RbBP5_average.bw \
WT_H3K27ac_C.bigWig Mll2KO_H3K27ac_C.bigWig Mll1KO_H3K27ac_C.bigWig DoubleKO_H3K27ac_C.bigWig \
F_F_K27me3_average.bw FC_FC_K27me3_average.bw Mll1-KO_K27me3_average.bw Double_KO_K27me3_average.bw \
F_F_K4me3_average.bw FC_FC_K4me3_average.bw Mll1-KO_K4me3_average.bw Double_KO_K4me3_average.bw \
F_F_K4me2_average.bw FC_FC_K4me2_average.bw Mll1-KO_K4me2_average.bw Double_KO_K4me2_average.bw \
F_F_K4me1_average.bw FC_FC_K4me1_average.bw Mll1-KO_K4me1_average.bw Double_KO_K4me1_average.bw \
"
outname="all_tracks"
thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R ${outpath}/coordinate.bed \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--zMax ${thrsholds} --missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab



### create distal/proximal ✅

In [ ]:
%%R

source("./git/chipseq_downstream_macs/bin/Regions_Removing_Proximal_ToTSS.R")

great.ref<-read.table("./otherouts/deeptools_heatmaps/GREATv4.genes.mm10.tsv", sep="\t")

start<-great.ref[,3]-1
end<-great.ref[,3]+1
bed<-cbind(great.ref[,2],start,end,great.ref[,4])

write.table(bed, file="./otherouts/deeptools_heatmaps/GREATv4_genes.bed", sep="\t",col.names=FALSE, row.names=FALSE, quote=FALSE)

tss_peaks<-regions_removingProximal(regions_bedfile = "./otherouts/deeptools_heatmaps/coordinate.bed" ,
                                     TSS_bedfile = "./otherouts/deeptools_heatmaps/GREATv4_genes.bed" ,
                                     kbDistance=5)

write.table(tss_peaks$distal, "./otherouts/deeptools_heatmaps/distal_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")

write.table(tss_peaks$proximal, "./otherouts/deeptools_heatmaps/proximal_preaks.bed", col.names=FALSE, row.names=FALSE, quote= FALSE , sep="\t")



### TODEL create proximal/ distral heatmaps (OLD)  ⭕

In [ ]:
%%script bash

cd ./otherouts/deeptools_heatmaps

outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"
inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"

#all tracks together

panel3="Anti-GFP_average.bw Anti-Menin_average.bw Anti-Mll1_average.bw Mll2_KO_Mll1_average.bw Anti-RbBp5_average.bw Double_KO_RbBP5_average.bw \
WT_H3K27ac_C.bigWig Mll2KO_H3K27ac_C.bigWig Mll1KO_H3K27ac_C.bigWig DoubleKO_H3K27ac_C.bigWig \
F_F_K27me3_average.bw FC_FC_K27me3_average.bw Mll1-KO_K27me3_average.bw Double_KO_K27me3_average.bw \
F_F_K4me3_average.bw FC_FC_K4me3_average.bw Mll1-KO_K4me3_average.bw Double_KO_K4me3_average.bw \
F_F_K4me2_average.bw FC_FC_K4me2_average.bw Mll1-KO_K4me2_average.bw Double_KO_K4me2_average.bw \
F_F_K4me1_average.bw FC_FC_K4me1_average.bw Mll1-KO_K4me1_average.bw Double_KO_K4me1_average.bw \
"

outname="distal_peaks"
macs_peaks="distal_preaks.bed"

thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

outname="proximal_peaks"
macs_peaks="proximal_preaks.bed"

thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab


### CpG+ and CpG- subdivide ✅

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/

mkdir ./bedtools_window

cp ./deeptools_heatmaps/proximal_preaks.bed ./bedtools_window/
cp ./deeptools_heatmaps/distal_preaks.bed ./bedtools_window/

cd ./bedtools_window

outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/bedtools_window/"
inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/bedtools_window/"

echo "proximal_preaks"
wc -l proximal_preaks.bed

#============================
#proximal CpG +
sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools window -w 1000  -a proximal_preaks.bed -b NMIs_mm10_mESC_afterLiftover_from_mm9_to_mm10.bed \
> proximal_CpG_plus.bed

Rscript -e 'proximal_CpG_plus<-read.table("proximal_CpG_plus.bed")[,c(1,2,3)];proximal_CpG_plus<-read.table("proximal_CpG_plus.bed")[,c(1,2,3)];proximal_CpG_plus_unique<-proximal_CpG_plus[which(duplicated(proximal_CpG_plus)==FALSE),];write.table(proximal_CpG_plus_unique, file="proximal_CpG_plus_unique.bed",col.names = FALSE, row.names=FALSE, quote=FALSE, sep="\t")'

echo "proximal_CpG_plus_unique"
wc -l proximal_CpG_plus_unique.bed

#proximal CpG -
sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools subtract -A -a proximal_preaks.bed -b proximal_CpG_plus_unique.bed \
> proximal_CpG_minus.bed

echo "proximal_CpG_minus.bed"
wc -l proximal_CpG_minus.bed

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -a proximal_CpG_plus_unique.bed -b proximal_CpG_minus.bed \
> proximal_CpG_intersect_plusminus.bed

echo "proximal_CpG_intersect_plusminus"
wc -l proximal_CpG_intersect_plusminus.bed

#=============================


echo "distal_peaks.bed"
wc -l distal_peaks.bed


#distal CpG +
sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools window -w 1000 -a distal_peaks.bed -b NMIs_mm10_mESC_afterLiftover_from_mm9_to_mm10.bed \
> distal_CpG_plus.bed

Rscript -e 'distal_CpG_plus<-read.table("distal_CpG_plus.bed")[,c(1,2,3)];distal_CpG_plus_unique<-distal_CpG_plus[which(duplicated(distal_CpG_plus)==FALSE),];write.table(distal_CpG_plus_unique, file="distal_CpG_plus_unique.bed",col.names = FALSE, row.names=FALSE, quote=FALSE, sep="\t")'

echo "distal_CpG_plus_unique.bed"
wc -l distal_CpG_plus_unique.bed


#distal CpG -
sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools subtract -A -a distal_peaks.bed -b distal_CpG_plus_unique.bed \
> distal_CpG_minus.bed

echo "distal_CpG_minus.bed"
wc -l distal_CpG_minus.bed



### create CpG+/- maps ✅

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps

cp ../bedtools_window/proximal_CpG_plus_unique.bed ./
cp ../bedtools_window/proximal_CpG_minus.bed ./
cp ../bedtools_window/distal_CpG_plus_unique.bed ./
cp ../bedtools_window/distal_CpG_minus.bed ./

outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"
inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"
wkdir="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream"

plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"
macs_peaks="./tmp/proximal_CpG_plus_unique.bed ./tmp/proximal_CpG_minus.bed ./tmp/distal_CpG_plus_unique.bed ./tmp/distal_CpG_minus.bed"


panel3_A="./tmp/Anti-GFP_average.bw \
./tmp/Anti-Menin_average.bw \
./tmp/Anti-Mll1_average.bw \
./tmp/Mll2_KO_Mll1_average.bw \
./tmp/Anti-RbBp5_average.bw \
./tmp/Double_KO_RbBP5_average.bw \
../parallel_averageBigwig/D0_WT_H3K27ac__average.bw \
../parallel_averageBigwig/D0_Mll1KO_H3K27ac__average.bw \
../parallel_averageBigwig/D0_Mll2KO_H3K27ac__average.bw \
../parallel_averageBigwig/D0_DoubleKO_H3K27ac__average.bw \
./tmp/F_F_K27me3_average.bw \
./tmp/Mll1-KO_K27me3_average.bw \
./tmp/FC_FC_K27me3_average.bw \
./tmp/Double_KO_K27me3_average.bw \
./tmp/F_F_K4me3_average.bw \
./tmp/Mll1-KO_K4me3_average.bw \
./tmp/FC_FC_K4me3_average.bw \
./tmp/Double_KO_K4me3_average.bw \
./tmp/F_F_K4me2_average.bw \
./tmp/Mll1-KO_K4me2_average.bw \
./tmp/FC_FC_K4me2_average.bw \
./tmp/Double_KO_K4me2_average.bw \
./tmp/F_F_K4me1_average.bw \
./tmp/Mll1-KO_K4me1_average.bw \
./tmp/FC_FC_K4me1_average.bw \
./tmp/Double_KO_K4me1_average.bw \
"

panel3_B="./tmp/Anti-GFP_average.bw \
./tmp/Anti-Menin_average.bw \
./tmp/Anti-Mll1_average.bw \
./tmp/Mll2_KO_Mll1_average.bw \
./tmp/Anti-RbBp5_average.bw \
./tmp/Double_KO_RbBP5_average.bw \
../parallel_averageBigwig/D0_WT_H3K27ac__average.bw \
../parallel_averageBigwig/D0_WT_RING1B__average.bw \
../parallel_averageBigwig/D0_KO_RING1B__average.bw \
../parallel_averageBigwig/D0_WT_H3K9me3__average.bw \
../parallel_averageBigwig/D0_KO_H3K9me3__average.bw \
"

# panel3_B="Anti-GFP_average.bw \
# Anti-Menin_average.bw \
# Anti-Mll1_average.bw \
# Mll2_KO_Mll1_average.bw \
# Anti-RbBp5_average.bw \
# Double_KO_RbBP5_average.bw \
# ../parallel_averageBigwig/D0_WT_H3K27ac__average.bw \
# ../unparallel_averageBigwig/D0_WT_RING1B_av.bigWig \
# ../parallel_averageBigwig/D0_KO_RING1B__average.bw \
# ../parallel_averageBigwig/D0_WT_H3K9me3__average.bw \
# ../parallel_averageBigwig/D0_KO_H3K9me3__average.bw \
# "

#all track

outname="alltracks"

slabels="Anti-GFP_av Anti-Menin_av Anti-Mll1_av Mll2_KO_Mll1_av Anti-RbBp5_av Double_KO_RbBP5_av \
WT_H3K27ac_av Mll2KO_H3K27ac_av Mll1KO_H3K27ac_av DoubleKO_H3K27ac_av \
F_F_K27me3_av Mll1-KO_K27me3_av FC_FC_K27me3_av Double_KO_K27me3_av \
F_F_K4me3_av Mll1-KO_K4me3_av FC_FC_K4me3_av Double_KO_K4me3_av \
F_F_K4me2_av Mll1-KO_K4me2_av FC_FC_K4me2_av Double_KO_K4me2_av \
F_F_K4me1_av Mll1-KO_K4me1_av FC_FC_K4me1_av Double_KO_K4me1_av \
"

thrsholds="10 10 10 10 15 15 10 10 10 10 10 10 10 10 40 40 40 40 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) -e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3_A} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) -e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--samplesLabel ${slabels} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

# panel B

thrsholds_B="10 10 10 10 15 15 10 20 20 3 3"
outname_B="additional_tracks"
slabels_B="Anti-GFP_av Anti-Menin_av Anti-Mll1_av Mll2_KO_Mll1_av Anti-RbBp5_av Double_KO_RbBP5_av \
WT_H3K27ac_av \
WT_RING1B_av \
KO_RING1B_av \
WT_H3K9me3_av \
KO_H3K9me3_av \
"

sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) -e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3_B} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname_B}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) -e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname_B}_deeptools_matrix.gzip \
--zMax ${thrsholds_B} --outFileName $outpath/${outname_B}_plotHeatmap.pdf --sortUsingSamples 7 \
--samplesLabel ${slabels_B} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname_B}_plotHeatmap.mat.tab




#### heatmap spkein free tracks


In [ ]:
%%script bash

git/Lara_MLL2/bin/heatmap_d0_spikeinfree.sh

### create CpG+/- heatmap D4

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/D4_deeptools_heatmaps

outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/D4_deeptools_heatmaps/"
inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/D4_deeptools_heatmaps/"
wkdir="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream"

plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"
macs_peaks="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/tmp/proximal_CpG_plus_unique.bed /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/tmp/proximal_CpG_minus.bed /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/tmp/distal_CpG_plus_unique.bed /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/tmp/distal_CpG_minus.bed"

panel3_A="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_MLL2__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_RbBP5__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_H3K27ac__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_DKO_H3K27ac__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_H3K27me3__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_DKO_H3K27me3__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_H3K4me3__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_DKO_H3K4me3__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_H3K4me2__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_DKO_H3K4me2__average.bw \
    "

#all track

outname="alltracks"

slabels="D4_WT_MLL2_av \
        D4_WT_RbBP5_av \
        D4_WT_H3K27ac_av \
        D4_DKO_H3K27ac_av \
        D4_WT_H3K27me3_av \
        D4_DKO_H3K27me3_av \
        D4_WT_H3K4me3_av \
        D4_DKO_H3K4me3_av \
        D4_WT_H3K4me2_av \
        D4_DKO_H3K4me2_av \
"

thrsholds="10 10 10 10 10 10 40 40 30 30"

sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) -e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${panel3_A} -R $macs_peaks \
-b 1000 --sortUsingSamples 3 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) -e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 3 \
--samplesLabel ${slabels} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

### TODEL_create heatmans NMIs (OLD OLD)

In [ ]:
%%script bash

cd ./otherouts/deeptools_heatmaps

macs_peaks="distal_NMIs_preaks.bed"
outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"
inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"

#all tracks together

panel3="Anti-GFP_average.bw Anti-Menin_average.bw Anti-Mll1_average.bw Mll2_KO_Mll1_average.bw Anti-RbBp5_average.bw Double_KO_RbBP5_average.bw \
WT_H3K27ac_C.bigWig Mll2KO_H3K27ac_C.bigWig Mll1KO_H3K27ac_C.bigWig DoubleKO_H3K27ac_C.bigWig \
F_F_K27me3_average.bw FC_FC_K27me3_average.bw Mll1-KO_K27me3_average.bw Double_KO_K27me3_average.bw \
F_F_K4me3_average.bw FC_FC_K4me3_average.bw Mll1-KO_K4me3_average.bw Double_KO_K4me3_average.bw \
F_F_K4me2_average.bw FC_FC_K4me2_average.bw Mll1-KO_K4me2_average.bw Double_KO_K4me2_average.bw \
F_F_K4me1_average.bw FC_FC_K4me1_average.bw Mll1-KO_K4me1_average.bw Double_KO_K4me1_average.bw \
"
outname="all_tracks_distal_NMIs"
thrsholds="5 5 5 5 7 7 5 5 5 5 5 5 5 5 10 10 10 10 15 15 15 15 5 5 5 5"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

outname="all_tracks_proximal_NMIs"
macs_peaks="proximal_NMIs_preaks.bed"
thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

outname="all_tracks_near_proximal_NMIs"
macs_peaks="near_distal_NMIs_preaks.bed"
thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab


outname="all_tracks_far_proximal_NMIs"
macs_peaks="far_distal_NMIs_preaks.bed"
thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

outname="all_tracks_near_distal_NMIs"
macs_peaks="near_distal_NMIs_preaks.bed"
thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

outname="all_tracks_far_distal_NMIs"
macs_peaks="near_distal_NMIs_preaks.bed"
thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab


outname="far_proximal_noinpeaks_NMIs"
macs_peaks="far_proximal_noinpeaks_NMIs_preaks.bed"
thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab


outname="near_proximal_noinpeaks_NMIs"
macs_peaks="near_proximal_noinpeaks_NMIs_preaks.bed"
thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

outname="inpeak_NMIs_preaks"
macs_peaks="inpeak_NMIs_preaks.bed"
thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab



## TODEL 5.5 extract near line peaks

In [ ]:
%%R

#1)
setwd("/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/")

library(ggplot2)
library(RColorBrewer)

#=======functions

source("./git/downstream_multiomics/bin/distance_fucntion.R")


#========input files

refpath.fasta<-"/mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa"

antimml2.annotated.peaksFile="in/test_chipseq_downstream/macs_broadpeaks/Anti-GFP.mLb.mkD.sorted_peaks.annotatePeaks.txt"

antimml2.peaksfile="in/test_chipseq_downstream/bedtools_window/coordinate.bed"

#==================parameters
infolder.diffpeaks="in/test_chipseq_downstream/diffbind/"
infolder.mll2peaks="in/test_chipseq_downstream/deeptools_heatmaps_filtered_by_diffDoubleKO/"
outfolder="outs/"
infolder.ucsc="in/ucsc/"

#==================number of peaks by subclassification
antimml2.annotated.peaks<-read.table(antimml2.annotated.peaksFile, sep="\t", header=TRUE)

peaks.strands<-cbind(id=paste0(antimml2.annotated.peaks$Chr,"_",
                               antimml2.annotated.peaks$Start,"_",
                               antimml2.annotated.peaks$End
),antimml2.annotated.peaks$Strand)

antimml2.peaks<-read.table(antimml2.peaksfile,sep="\t", header=FALSE)
total.peaks<-nrow(antimml2.peaks)

#==========================================
# extract line annotations from repeat mask
#==========================================
repeat_mask<-read.table("in/ucsc/rmsk.txt",sep="\t",header=FALSE)
nrow(repeat_mask)

print(unique(repeat_mask$V12))

#  [1] "LINE"           "Simple_repeat"  "LTR"            "SINE"           "Low_complexity"
#  [6] "DNA"            "snRNA"          "Other"          "Satellite"      "Unknown"
# [11] "SINE?"          "srpRNA"         "tRNA"           "LTR?"           "RNA"
# [16] "scRNA"          "RC"             "rRNA"           "DNA?"           "RC?"
# [21] "LINE?"          "Retroposon"

repeat_mask_line<-repeat_mask[grep("\\b(LINE|LINE?)\\b",repeat_mask$V12),]

bedrm<-repeat_mask[,c("V6","V7","V8","V11")]

write.table(bedrm ,file="./in/ucsc/l1_rmsk.bed", sep="\t",
            col.names = FALSE, row.names = FALSE,
            quote=FALSE
            )

#==========================================
#calc distribution of distances between
#==========================================


#atb<-c("K4me3","K4me2")

atb<-c("K4me3")
distances_list<-NULL
for (i in 1:length(atb)){

  file_pattern="_distal_CpG_minus_Double_KO_vs_F_F.bed"
  d<-read.table(paste0(infolder.mll2peaks,atb[i],file_pattern),sep="\t")
  distances<-extractdistances( peaks.coords = d,
                               repeat_mask_line=repeat_mask_line,
                               addfamily = TRUE,
                               addpeaks = TRUE,
                               addlines = TRUE,
                               add5prime = TRUE
                               )

  distances_list[length(distances_list)+1]<-paste0(atb[i],file_pattern)
  assign(paste0(atb[i],file_pattern),distances)

  densval<-as.numeric(as.character(distances[,1]))
  pdf(paste0(outfolder,atb[i],file_pattern,".pdf"))
  plot(density(densval),
       main=paste0(atb[i],file_pattern," nearest LINE"),
       sub=paste0("median=",median(densval)))
  abline(v=median(densval))
  dev.off()

  file_pattern="_distal_CpG_plus_Double_KO_vs_F_F.bed"
  d<-read.table(paste0(infolder.mll2peaks,atb[i],file_pattern),sep="\t")
  distances<-extractdistances( peaks.coords = d,
                               repeat_mask_line=repeat_mask_line,
                               addfamily = TRUE,
                               addpeaks = TRUE,
                               addlines = TRUE,
                               add5prime = TRUE
  )

  distances_list[length(distances_list)+1]<-paste0(atb[i],file_pattern)
  assign(paste0(atb[i],file_pattern),distances)

  densval<-as.numeric(as.character(distances[,1]))
  pdf(paste0(outfolder,atb[i],file_pattern,".pdf"))
  plot(density(densval),
       main=paste0(atb[i],file_pattern," nearest LINE"),
       sub=paste0("median=",median(densval)))
  abline(v=median(densval))
  dev.off()

  file_pattern="_proximal_CpG_minus_Double_KO_vs_F_F.bed"
  d<-read.table(paste0(infolder.mll2peaks,atb[i],file_pattern),sep="\t")
  distances<-extractdistances( peaks.coords = d,
                               repeat_mask_line=repeat_mask_line,
                               addfamily = TRUE,
                               addpeaks = TRUE,
                               addlines = TRUE,
                               add5prime = TRUE
  )

  distances_list[length(distances_list)+1]<-paste0(atb[i],file_pattern)
  assign(paste0(atb[i],file_pattern),distances)

  densval<-as.numeric(as.character(distances[,1]))
  pdf(paste0(outfolder,atb[i],file_pattern,".pdf"))
  plot(density(densval),
       main=paste0(atb[i],file_pattern," nearest LINE"),
       sub=paste0("median=",median(densval)))
  abline(v=median(densval))
  dev.off()

  file_pattern="_proximal_CpG_plus_Double_KO_vs_F_F.bed"
  d<-read.table(paste0(infolder.mll2peaks,atb[i],file_pattern),sep="\t")
  distances<-extractdistances( peaks.coords = d,
                               repeat_mask_line=repeat_mask_line,
                               addfamily = TRUE,
                               addpeaks = TRUE,
                               addlines = TRUE,
                               add5prime = TRUE
  )

  distances_list[length(distances_list)+1]<-paste0(atb[i],file_pattern)
  assign(paste0(atb[i],file_pattern),distances)

  densval<-as.numeric(as.character(distances[,1]))
  pdf(paste0(outfolder,atb[i],file_pattern,".pdf"))
  plot(density(densval),
       main=paste0(atb[i],file_pattern," nearest LINE"),
       sub=paste0("median=",median(densval)))
  abline(v=median(densval))
  dev.off()


}

#cumulative distribution plot

#====================================
#  line1 distribution around subgroups
#====================================

#========================================
#====extract interesting line coordinates
#========================================

#================================
# extract other line1 coordinate
#================================

dcp_anno<-K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed[,c("line_chr","line_start","line_end","family","distances","line_strand")]
dcp_anno[,"family"]<-paste0(dcp_anno[,"family"],"_",rep(1:nrow(dcp_anno)))

dcp_tarpeak<-K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed[,c("peaks_chr","peaks_start","peaks_end")]
dcp_tarpeak<-cbind(dcp_tarpeak,
               name=paste0("peak_",rep(1:nrow(dcp_tarpeak))),
               score=K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed[,"distances"],
               strand=rep(".",nrow(dcp_tarpeak))
)



write.table(dcp_anno, file="./outs/dcp.l1.bed",
            col.names = FALSE,
            row.names = FALSE,
            quote= FALSE,
            sep="\t"
)

write.table(dcp_tarpeak, file="./outs/dcp.target.peaks.bed",
            col.names = FALSE,
            row.names = FALSE,
            quote= FALSE,
            sep="\t"
)

filterbydistance(extractdistances.out=K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed,
                 fildist=2500,
                 out="outs/distfilter_DKO_K4me3_dcp")

filterbydistance(extractdistances.out=K4me3_distal_CpG_minus_Double_KO_vs_F_F.bed,
                 fildist=2500,
                 out="outs/distfilter_DKO_K4me3_dcm")

filterbydistance(extractdistances.out=K4me3_proximal_CpG_minus_Double_KO_vs_F_F.bed,
                 fildist=2500,
                 out="outs/distfilter_DKO_K4me3_pcm")

filterbydistance(extractdistances.out=K4me3_proximal_CpG_plus_Double_KO_vs_F_F.bed,
                 fildist=2500,
                 out="outs/distfilter_DKO_K4me3_pcp")

#make strand specific peaks

window<-5000
recentered_start<-c()
recentered_end<-c()

recentered_start<-as.numeric(subdist[,"fprime"])-window
recentered_end<-as.numeric(subdist[,"fprime"])+window

subdist.tarfam_anno_2=subdist.tarfam_anno
subdist.tarfam_anno_2[,"line_start"]<-recentered_start
subdist.tarfam_anno_2[,"line_end"]<-recentered_end
subdist.tarfam_anno_2[,"line_strand"]<-rep(".", nrow(subdist.tarfam_anno_2))

write.table(subdist.tarfam_anno_2, file="./outs/recentered_distfil_dcm.l1.bed",
            col.names = FALSE,
            row.names = FALSE,
            quote= FALSE,
            sep="\t"
)





## 6. Differential Peak Calling ✅

### environment setting

In [ ]:
%%script bash

peaks_file="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/F_F_K4me2_A_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/F_F_K4me3_A_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/FC_FC_K4me2_A_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/FC_FC_K4me3_A_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/F_F_K4me2_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/F_F_K4me3_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/FC_FC_K4me2_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/FC_FC_K4me3_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Mll1-KO_K4me2_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Mll1-KO_K4me3_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Mll1-KO_K4me2_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Mll1-KO_K4me3_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Double_KO_K4me2_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Double_KO_K4me3_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Anti-GFP_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Anti-GFP_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Double_KO_K4me2_A_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Double_KO_K4me3_A_peaks.xls"

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/git

sudo git clone git@github.com:lucidif/bioinfoGenerals.git


#check if file exitsts
for i in $peaks_file ; do ls $i ; done

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/diffbind

mkdir ./input

cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/coordinate.bed ./in/test_chipseq_downstream/deeptools_heatmaps/

Rscript -e 'tb<-read.table("./in/test_chipseq_downstream/deeptools_heatmaps/coordinate.bed", sep="\t")
window<-1000 ; tb[which(tb$V2>=window),"V2"]<-tb[which(tb$V2>=window),"V2"]-window
tb$V3<-tb$V3+window
write.table(tb,file="./in/test_chipseq_downstream/deeptools_heatmaps/coordinates_extended.bed",sep="\t",quote=FALSE, col.names = FALSE, row.names = FALSE )'

inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream"
outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/diffbind"
root="/mnt/datawk1/"








### run statistics

In [ ]:
%%script bash

#/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/coordinate.bed

#gzip -c /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/coordinate.bed \
#> /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/coordinate.bed.gz

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -v ${root}:${root} -w $outpath -u $(id -u):$(id -g) -it \
lucidif/diffbind:3.14.0

source("../../git/chipseq_downstream_macs/bin/diffbind_DP.R")

diffbind_DP (
  inssfile="./in/test_chipseq_downstream/deeptools_heatmaps/Diffbind_SS.csv",
  antibodies=c("K4me2","K4me3"),
  comparisons=data.frame(
     c1=c(numerator="Double_KO",denominator="F_F")
   ),
  th=0.05,
  fold=2
)

quit()


## 7. heatmap mll2 peaks filtered by DoubleKO vs F_F differential preaks ▶



### make environment D0

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/

mkdir deeptools_heatmaps_filtered_by_diffDoubleKO
mkdir ./deeptools_heatmaps_filtered_by_diffDoubleKO/input

cd deeptools_heatmaps_filtered_by_diffDoubleKO

#cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/tmp/*.bigWig ./input
cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/*.bw ./input
cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig/*.bw ./input

cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/

ls /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/

refbed="distal_CpG_minus.bed distal_CpG_plus_unique.bed proximal_CpG_minus.bed proximal_CpG_plus_unique.bed proximal_preaks.bed distal_preaks.bed"

for i in $refbed ; do echo "cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/${i} ./in/test_chipseq_downstream/deeptools_heatmaps/" ; cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/${i} ./in/test_chipseq_downstream/deeptools_heatmaps/ ; done

cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/diffbind/fold2_th0_05_K4me2_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed \
./in/test_chipseq_downstream/deeptools_heatmaps/

cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/diffbind/fold2_th0_05_K4me3_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed \
./in/test_chipseq_downstream/deeptools_heatmaps/

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO

outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO"
inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO"


### heatmaps D0 ▶

#### panel A D0

In [ ]:
%%script bash

samples="./in/Anti-GFP_average.bw \
./in/Anti-Menin_average.bw \
./in/Anti-Mll1_average.bw \
./in/Mll2_KO_Mll1_average.bw \
./in/Anti-RbBp5_average.bw \
./in/Double_KO_RbBP5_average.bw \
./in/D0_WT_H3K27ac__average.bw \
./in/D0_Mll1KO_H3K27ac__average.bw \
./in/D0_Mll2KO_H3K27ac__average.bw \
./in/D0_DoubleKO_H3K27ac__average.bw \
./in/F_F_K27me3_average.bw \
./in/Mll1-KO_K27me3_average.bw \
./in/FC_FC_K27me3_average.bw \
./in/Double_KO_K27me3_average.bw \
./in/F_F_K4me3_average.bw \
./in/Mll1-KO_K4me3_average.bw \
./in/FC_FC_K4me3_average.bw \
./in/Double_KO_K4me3_average.bw \
./in/F_F_K4me2_average.bw \
./in/Mll1-KO_K4me2_average.bw \
./in/FC_FC_K4me2_average.bw \
./in/Double_KO_K4me2_average.bw \
./in/F_F_K4me1_average.bw \
./in/Mll1-KO_K4me1_average.bw \
./in/FC_FC_K4me1_average.bw \
./in/Double_KO_K4me1_average.bw\
"


#intersect peaks

#K2ME2 DOWN
#distal/proximal
sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/proximal_preaks.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me2_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me2_proximal_Double_KO_vs_F_F.bed

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/distal_preaks.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me2_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me2_distal_Double_KO_vs_F_F.bed

#proximal NMIs +/-
sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/proximal_CpG_plus_unique.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me2_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me2_proximal_CpG_plus_Double_KO_vs_F_F.bed

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/proximal_CpG_minus.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me2_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me2_proximal_CpG_minus_Double_KO_vs_F_F.bed


#proximal NMIs +/-
sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/distal_CpG_plus_unique.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me2_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me2_distal_CpG_plus_Double_KO_vs_F_F.bed

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/distal_CpG_minus.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me2_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me2_distal_CpG_minus_Double_KO_vs_F_F.bed

wc -l K4me2_proximal_Double_KO_vs_F_F.bed > K4me2_intersectReport.txt
wc -l K4me2_proximal_CpG_plus_Double_KO_vs_F_F.bed >> K4me2_intersectReport.txt
wc -l K4me2_proximal_CpG_minus_Double_KO_vs_F_F.bed >> K4me2_intersectReport.txt
wc -l K4me2_distal_Double_KO_vs_F_F.bed >> K4me2_intersectReport.txt
wc -l K4me2_distal_CpG_plus_Double_KO_vs_F_F.bed >> K4me2_intersectReport.txt
wc -l K4me2_distal_CpG_minus_Double_KO_vs_F_F.bed >> K4me2_intersectReport.txt

#K4ME3 DOWN
#distal/proximal
sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/proximal_preaks.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me3_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me3_proximal_Double_KO_vs_F_F.bed

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/distal_preaks.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me3_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me3_distal_Double_KO_vs_F_F.bed

#proximal NMIs +/-
sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/proximal_CpG_plus_unique.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me3_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me3_proximal_CpG_plus_Double_KO_vs_F_F.bed

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/proximal_CpG_minus.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me3_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me3_proximal_CpG_minus_Double_KO_vs_F_F.bed


#proximal NMIs +/-
sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/distal_CpG_plus_unique.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me3_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./in/test_chipseq_downstream/deeptools_heatmaps/distal_CpG_minus.bed -b ./in/test_chipseq_downstream/deeptools_heatmaps/fold2_th0_05_K4me3_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed  \
> K4me3_distal_CpG_minus_Double_KO_vs_F_F.bed

wc -l K4me3_proximal_Double_KO_vs_F_F.bed > K4me3_intersectReport.txt
wc -l K4me3_proximal_CpG_plus_Double_KO_vs_F_F.bed >> K4me3_intersectReport.txt
wc -l K4me3_proximal_CpG_minus_Double_KO_vs_F_F.bed >> K4me3_intersectReport.txt
wc -l K4me3_distal_Double_KO_vs_F_F.bed >> K4me3_intersectReport.txt
wc -l K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed >> K4me3_intersectReport.txt
wc -l K4me3_distal_CpG_minus_Double_KO_vs_F_F.bed >> K4me3_intersectReport.txt


#make plots
outname="K4me2_DOWN_Double_KO_vs_F_F"
macs_peaks="K4me2_proximal_CpG_plus_Double_KO_vs_F_F.bed K4me2_proximal_CpG_minus_Double_KO_vs_F_F.bed K4me2_distal_CpG_plus_Double_KO_vs_F_F.bed K4me2_distal_CpG_minus_Double_KO_vs_F_F.bed"
plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"
slabels="Anti-GFP_av \
Anti-Menin_av \
Anti-Mll1_av \
Mll2_KO_Mll1_av \
Anti-RbBp5_av \
Double_KO_RbBP5_av \
WT_H3K27ac_av \
Mll1KO_H3K27ac_av \
Mll2KO_H3K27ac_av \
DoubleKO_H3K27ac_av \
F_F_K27me3_av \
Mll1-KO_K27me3_av \
FC_FC_K27me3_av \
Double_KO_K27me3_av \
F_F_K4me3_av \
Mll1-KO_K4me3_av \
FC_FC_K4me3_av \
Double_KO_K4me3_av \
F_F_K4me2_av \
Mll1-KO_K4me2_av \
FC_FC_K4me2_av \
Double_KO_K4me2_av \
F_F_K4me1_av \
Mll1-KO_K4me1_av \
FC_FC_K4me1_av \
Double_KO_K4me1_av \
"

thrsholds="10 10 10 10 15 15 10 10 10 10 10 10 10 10 40 40 40 40 30 30 30 30 10 10 10 10"


sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${samples} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--samplesLabel ${slabels} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

outname="K4me3_DOWN_Double_KO_vs_F_F"
macs_peaks="K4me3_proximal_CpG_plus_Double_KO_vs_F_F.bed K4me3_proximal_CpG_minus_Double_KO_vs_F_F.bed K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed K4me3_distal_CpG_minus_Double_KO_vs_F_F.bed"
plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"
# slabels="Anti-GFP_av Anti-Menin_av Anti-Mll1_av Mll2_KO_Mll1_av Anti-RbBp5_av Double_KO_RbBP5_av \
# WT_H3K27ac_C Mll2KO_H3K27ac_C Mll1KO_H3K27ac_C DoubleKO_H3K27ac_C \
# F_F_K27me3_av FC_FC_K27me3_av Mll1-KO_K27me3_av Double_KO_K27me3_av \
# F_F_K4me3_av FC_FC_K4me3_av Mll1-KO_K4me3_av Double_KO_K4me3_av \
# F_F_K4me2_av FC_FC_K4me2_av Mll1-KO_K4me2_av Double_KO_K4me2_av \
# F_F_K4me1_av FC_FC_K4me1_av Mll1-KO_K4me1_av Double_KO_K4me1_av \
# "
slabels="Anti-GFP_av Anti-Menin_av Anti-Mll1_av Mll2_KO_Mll1_av Anti-RbBp5_av Double_KO_RbBP5_av \
WT_H3K27ac_av Mll1KO_H3K27ac_av Mll2KO_H3K27ac_av DoubleKO_H3K27ac_av \
F_F_K27me3_av Mll1-KO_K27me3_av FC_FC_K27me3_av Double_KO_K27me3_av \
F_F_K4me3_av Mll1-KO_K4me3_av FC_FC_K4me3_av Double_KO_K4me3_av \
F_F_K4me2_av Mll1-KO_K4me2_av FC_FC_K4me2_av Double_KO_K4me2_av \
F_F_K4me1_av Mll1-KO_K4me1_av FC_FC_K4me1_av Double_KO_K4me1_av \
"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${samples} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyh
# samples="./input/Anti-GFP_average.bw ./input/Anti-Menin_average.bw ./input/Anti-Mll1_average.bw ./input/Mll2_KO_Mll1_average.bw ./input/Anti-RbBp5_average.bw ./input/Double_KO_RbBP5_average.bw ./input/D0_WT_H3K27ac__average.bw ./input/D0_Mll1KO_H3K27ac__average.bw ./input/D0_Mll2KO_H3K27ac__average.bw ./input/D0_DoubleKO_H3K27ac__average.bw ./input/F_F_K27me3_average.bw ./input/Mll1-KO_K27me3_average.bw ./input/FC_FC_K27me3_average.bw ./input/Double_KO_K27me3_average.bw ./input/F_F_K4me3_average.bw ./input/Mll1-KO_K4me3_average.bw ./input/FC_FC_K4me3_average.bw ./input/Double_KO_K4me3_average.bw ./input/F_F_K4me2_average.bw ./input/Mll1-KO_K4me2_average.bw ./input/FC_FC_K4me2_average.bw ./input/Double_KO_K4me2_average.bw ./input/F_F_K4me1_average.bw ./input/Mll1-KO_K4me1_average.bw ./input/FC_FC_K4me1_average.bw ./input/Double_KO_K4me1_average.bw"

# slabels="Anti-GFP_av Anti-Menin_av Anti-Mll1_av Mll2_KO_Mll1_av Anti-RbBp5_av Double_KO_RbBP5_av \
# WT_H3K27ac_av Mll1KO_H3K27ac_av Mll2KO_H3K27ac_av DoubleKO_H3K27ac_C \
# F_F_K27me3_av Mll1-KO_K27me3_av FC_FC_K27me3_av Double_KO_K27me3_av \
# F_F_K4me3_av Mll1-KO_K4me3_av FC_FC_K4me3_av Double_KO_K4me3_av \
# F_F_K4me2_av Mll1-KO_K4me2_av FC_FC_K4me2_av Double_KO_K4me2_av \
# F_F_K4me1_av Mll1-KO_K4me1_av FC_FC_K4me1_av Double_KO_K4me1_av \
# "

dfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--samplesLabel ${slabels} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 \
--samplesLabel ${slabels} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

#sorting test

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_sorted_by_k4me3_plotHeatmap.pdf --sortUsingSamples 15 \
--samplesLabel ${slabels} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname}_sorted_by_k4me3_plotHeatmap.mat.tab

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_sorted_by_k27me3_plotHeatmap.pdf --sortUsingSamples 11 \
--samplesLabel ${slabels} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname}_sorted_by_k27me3_plotHeatmap.mat.tab --sortRegions "ascend"


sh git/Lara_MLL2/bin/profiles_d0_spikeinfree.sh



#### pannel B plots D0

In [ ]:
%%script bash

#TODO non prendere i file direttamente dal tmp

mainpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/"

macs_peaks="K4me3_proximal_CpG_plus_Double_KO_vs_F_F.bed K4me3_proximal_CpG_minus_Double_KO_vs_F_F.bed K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed K4me3_distal_CpG_minus_Double_KO_vs_F_F.bed"
plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"

# OLD_panel3_B="../deeptools_heatmaps/tmp/Anti-GFP_average.bw \
# ../deeptools_heatmaps/tmp/Anti-Menin_average.bw \
# ../deeptools_heatmaps/tmp/Anti-Mll1_average.bw \
# ../deeptools_heatmaps/tmp/Mll2_KO_Mll1_average.bw \
# ../deeptools_heatmaps/tmp/Anti-RbBp5_average.bw \
# ../deeptools_heatmaps/tmp/Double_KO_RbBP5_average.bw \
# ../parallel_averageBigwig/D0_WT_H3K27ac__average.bw \
# ./in/test_chipseq_downstream/deeptools_heatmaps/F_F_K27me3_average.bw \
# ./in/test_chipseq_downstream/deeptools_heatmaps/F_F_K4me3_average.bw \
# ../parallel_averageBigwig/D0_WT_RING1B__average.bw \
# ../parallel_averageBigwig/D0_KO_RING1B__average.bw \
# ../parallel_averageBigwig/D0_WT_H3K9me3__average.bw \
# ../parallel_averageBigwig/D0_KO_H3K9me3__average.bw \
# "

panel3_B="../deeptools_heatmaps/tmp/Anti-GFP_average.bw \
../deeptools_heatmaps/tmp/F_F_K27me3_average.bw \
../deeptools_heatmaps/tmp/Double_KO_K27me3_average.bw \
../parallel_averageBigwig/D0_WT_RING1B__average.bw \
../parallel_averageBigwig/D0_KO_RING1B__average.bw \
../deeptools_heatmaps/tmp/F_F_K4me3_average.bw \
../deeptools_heatmaps/tmp/Double_KO_K4me3_average.bw \
"

thrsholds_B="10 10 10 50 50 40 40"

slabels_B="Anti-GFP_av \
WT_K27me3_av \
DKO_K27me3_av \
WT_RING1B_av \
DKO_RING1B_av \
WT_H3K4me3_av \
DKO_H3K4me3_av \
"

outname="K4me3_loss_RIG1B"
sudo docker run -v $mainpath:$mainpath -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
-e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${panel3_B} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

# sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
# -e MPLCONFIGDIR=$outpath/.matplotlib \
# quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
# --missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
# --zMax ${thrsholds_B} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples  \
# --samplesLabel ${slabels_B} --regionsLabel ${plabels} \
# --outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab

#sorting test

sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
-e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds_B} --outFileName $outpath/${outname}_sorted_by_k4me3_plotHeatmap.pdf --sortUsingSamples 6 \
--samplesLabel ${slabels_B} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname}_sorted_by_k4me3_plotHeatmap.mat.tab

# sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
# -e MPLCONFIGDIR=$outpath/.matplotlib \
# quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
# --missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
# --zMax ${thrsholds_B} --outFileName $outpath/${outname}_sorted_by_k27me3_plotHeatmap.pdf --sortUsingSamples 8 \
# --samplesLabel ${slabels_B} --regionsLabel ${plabels} \
# --outFileNameMatrix $outpath/${outname}_sorted_by_k27me3_plotHeatmap.mat.tab \
# --sortRegions "ascend"

#=============RING1B=================

outname="K4me3_DOWN_Double_KO_vs_F_F_RING1B_only"
macs_peaks="K4me3_proximal_CpG_plus_Double_KO_vs_F_F.bed K4me3_proximal_CpG_minus_Double_KO_vs_F_F.bed K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed K4me3_distal_CpG_minus_Double_KO_vs_F_F.bed"
plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"

samples="
../parallel_averageBigwig/D0_WT_RING1B__average.bw \
../parallel_averageBigwig/D0_KO_RING1B__average.bw \
"

slabels="WT DKO"

sudo docker run -v /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/:/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/ -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
-e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${samples} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/:/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/ -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) -e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotProfile -m $outpath/${outname}_deeptools_matrix.gzip -out ${outpath}/${outname}_Profile.png --perGroup --colors "#715eee" "#ffb00e" --plotTitle "RIG1B loss K4me3 in dko vs wt" --samplesLabel ${slabels} --regionsLabel ${plabels}






#### heatmap spkein free tracks

### heatmpas D4

#### make environment

In [ ]:
%%script bash

mkdir /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/D4_deeptools_heatmaps_filtered_by_diffDoubleKO

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/D4_deeptools_heatmaps_filtered_by_diffDoubleKO

#### make plots

In [ ]:
%%script bash

outname="D4_K4me3_DOWN_Double_KO_vs_F_F"
outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/D4_deeptools_heatmaps_filtered_by_diffDoubleKO"
prjpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream"

macs_peaks="../deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_proximal_CpG_plus_Double_KO_vs_F_F.bed \
../deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_proximal_CpG_minus_Double_KO_vs_F_F.bed \
../deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed \
../deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_minus_Double_KO_vs_F_F.bed"


plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"

panel3_A="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_MLL2__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_RbBP5__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_H3K27ac__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_DKO_H3K27ac__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_H3K27me3__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_DKO_H3K27me3__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_H3K4me3__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_DKO_H3K4me3__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_WT_H3K4me2__average.bw \
        /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/D4_DKO_H3K4me2__average.bw \
    "

slabels="D4_WT_MLL2_av \
        D4_WT_RbBP5_av \
        D4_WT_H3K27ac_av \
        D4_DKO_H3K27ac_av \
        D4_WT_H3K27me3_av \
        D4_DKO_H3K27me3_av \
        D4_WT_H3K4me3_av \
        D4_DKO_H3K4me3_av \
        D4_WT_H3K4me2_av \
        D4_DKO_H3K4me2_av \
"

thrsholds="10 10 10 10 10 10 40 40 30 30"


sudo docker run -v $prjpath:$prjpath -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) -e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${panel3_A} -R $macs_peaks \
-b 1000 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"


sudo docker run -v $prjpath:$prjpath -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname}_deeptools_matrix.gzip \
--zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 3 \
--samplesLabel ${slabels} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname}_plotHeatmap.mat.tab


## 8. filter by distance from LINE (nearest) and exec preliminary post processing analysis and make GREAT input files

### make environment

In [ ]:
%%script bash

#in realta' questa analisi e' ancora solo una analisi di chipseq quindi non e' corretto creare ora il progetto di multiomica, ma sarebbe stato piu' corretto salvare tutto nel post processing di chipseq

mkdir -p /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/

mkdir -p ./git
mkdir -p ./in
mkdir -p ./in/test_chipseq_downstream
mkdir -p ./in/test_chipseq_downstream/bedtools_window

mkdir -p ./outs/CHiP_postprocessing_line1_dist

cp ???/Anti-GFP.mLb.mkD.sorted_peaks.annotatePeaks.txt ./in/test_chipseq_downstream/bedtools_window/

cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/bedtools_window/.. ./in/test_chipseq_downstream/bedtools_window/

cp ???/coordinate.bed ./in/test_chipseq_downstream/bedtools_window/

#refpath.fasta<-"/mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa"

cd git

git clone git@github.com:lucidif/downstream_multiomic.git

cd ..




### execute core analysis

In [ ]:
%%R

setwd("/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/")

source("./git/downstream_multiomics/bin/distance_functions.R")

source("./git/downstream_multiomics/bin/Lara_CHiP_postprocessing_line1_dist.R")






##  TODEL: peak calling h3k27ac

#### macs peaks

#### diffbind

In [ ]:
%%script bash

peaks_file="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/F_F_K4me2_A_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/F_F_K4me3_A_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/FC_FC_K4me2_A_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/FC_FC_K4me3_A_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/F_F_K4me2_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/F_F_K4me3_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/FC_FC_K4me2_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/FC_FC_K4me3_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Mll1-KO_K4me2_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Mll1-KO_K4me3_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Mll1-KO_K4me2_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Mll1-KO_K4me3_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Double_KO_K4me2_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Double_KO_K4me3_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Anti-GFP_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Anti-GFP_B_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Double_KO_K4me2_A_peaks.xls \
/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/MACS_peakcalling_a2_narrow/star/mergedLibrary/macs2/narrowPeak/Double_KO_K4me3_A_peaks.xls"

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/git

sudo git clone git@github.com:lucidif/bioinfoGenerals.git


#check if file exitsts
for i in $peaks_file ; do ls $i ; done

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/diffbind

mkdir ./input

cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/coordinate.bed ./in/test_chipseq_downstream/deeptools_heatmaps/

Rscript -e 'tb<-read.table("./in/test_chipseq_downstream/deeptools_heatmaps/coordinate.bed", sep="\t")
window<-1000 ; tb[which(tb$V2>=window),"V2"]<-tb[which(tb$V2>=window),"V2"]-window
tb$V3<-tb$V3+window
write.table(tb,file="./in/test_chipseq_downstream/deeptools_heatmaps/coordinates_extended.bed",sep="\t",quote=FALSE, col.names = FALSE, row.names = FALSE )'

inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream"
outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/diffbind"
root="/mnt/datawk1/"

%%script bash

#/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/coordinate.bed

#gzip -c /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/coordinate.bed \
#> /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/coordinate.bed.gz

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -v ${root}:${root} -w $outpath -u $(id -u):$(id -g) -it \
lucidif/diffbind:3.14.0

source("../../git/chipseq_downstream_macs/bin/diffbind_DP.R")

diffbind_DP (
  inssfile="./in/test_chipseq_downstream/deeptools_heatmaps/Diffbind_SS.csv",
  antibodies=c("K4me2","K4me3"),
  comparisons=data.frame(
     c1=c(numerator="Double_KO",denominator="F_F")
   ),
  th=0.05,
  fold=2
)

quit()


#### distal cpg + lose k4me3 subset by h3k27ac

## 9. save analyzed data on long-time storage ▶

# RNA-seq

## 1. Download dataset

In [ ]:
%%script bash

mkdir /mnt/c/wkdir/Lara/RNAseq/rnaseq_lara_fastq_0423
scp lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Sarah/june2023/move_nas0623/rnaseq_lara_fastq_0423/* \
/mnt/c/wkdir/Lara/RNAseq/rnaseq_lara_fastq_0423/

rsync --progress --append --inplace -av lucio@193.144.215.86:/comun/alvaro/radauser2/rnaseq_lara_june23 \
/mnt/c/wkdir/Lara/RNAseq/




## 2. Import data on fisso

In [ ]:
%%script bash

rsync --progress --append --inplace -av -e 'ssh -p 6022' /mnt/c/wkdir/Lara/RNAseq/rnaseq_lara_june23 \
lucio@house.mane.st:/media/lucio/bioData1/2.Datasets/Lara/RNAseq

rsync --progress --append --inplace -av -e 'ssh -p 6022' /mnt/c/wkdir/Lara/RNAseq/rnaseq_lara_fastq_0423 \
lucio@house.mane.st:/media/lucio/bioData1/2.Datasets/Lara/RNAseq

rsync --progress --append --inplace -av -e 'ssh -p 6022' /mnt/c/wkdir/Lara/RNAseq/nf-core_samplesheet.csv \
lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4

rsync --progress --append --inplace -av -e 'ssh -p 6022' /mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4/rnaseq_random.config \
lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4_random/



## 3. import environment on fisso

In [ ]:
%%script bash


mkdir /media/lucio/wkssd/bioinfo/Lara/RNAseq_lara_day0_4

cd /media/lucio/wkssd/bioinfo/Lara/RNAseq_lara_day0_4

mkdir git
mkdir nfout



## 4. run analysis on Fisso

### a) standard analysis for DEGs extraction ✅

In [ ]:
%%script bash

cd /media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4

# awk -F'\t' '{
#     # Prendi le prime 8 colonne
#     for (i = 1; i <= 8; i++) {
#         printf("%s\t", $i)
#     }

#     # Combina le colonne dalla 9 alla fine
#     for (i = 9; i <= NF; i++) {
#         if (i == NF) {
#             printf("%s\n", $i)
#         } else {
#             printf("%s ", $i)
#         }
#     }
# }' /media/lucio/bioData1/2.References/gtf/ENSEMBL_102/noheader_ncbi_compatiple_Mus_musculus.GRCm38.102.gtf > /media/lucio/bioData1/2.References/gtf/ENSEMBL_102/fixed_Mus_musculus.GRCm38.102.gtf

sudo su

export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MTgzfS4yYzMxYzc2YzkzMjg5MGUxNzczZTI0ZDBiZjcyNTQyZWUyMTc2NjNj
export NXF_VER=23.10.0

nextflow run /home/lucio/git/nf-core-rnaseq_3.14.0/3_14_0/ \
--input /media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4/nf-core_samplesheet.csv \
--outdir /media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4/nfout \
--fasta /media/lucio/bioData1/2.References/fasta/UCSC_GRCm38/mm10.fa \
--gtf  /media/lucio/bioData1/2.References/gtf/ncbi/mm10_Build38/mm10.refGene.gtf.gz \
--aligner star_rsem \
--skip_umi_extract \
--save_reference \
-profile docker -with-tower -resume




export results

In [ ]:
%%script R

setwd("/media/lucio/wkssd/bioinfo/Lara/RNAseq_lara_day0_4/star_rsem")

#unite unique counts
rawfolder="/media/lucio/wkssd/bioinfo/Lara/RNAseq_lara_day0_4/star_rsem"
files<-list.files(rawfolder, pattern=".genes")


for (i in 1:length(files)){
  intab<-read.table(paste0(rawfolder,"/",files[i]),header=TRUE)
  midtab<-cbind(intab$gene_id,intab$expected_count)
  colnames(midtab)<-c("gene_id",files[i])

  if(i ==1 ){
    outtab <- midtab
  }else{
    outtab <- merge(outtab, midtab, by=1)
  }

}

colnames(outtab)<-gsub(".genes.results","",colnames(outtab))

write.table(outtab, file="/media/lucio/wkssd/bioinfo/Lara/RNAseq_lara_day0_4/star_rsem/rawcounts.txt",
            sep="\t" , col.names=TRUE, row.names=FALSE, quote=FALSE)






### b) random mapping for line visualization

In [ ]:
%%script bash

cd /media/lucio/wkssd_crucial
cp /media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4/nf-core_samplesheet.csv /media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4_random/

cd /media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4_random


sudo su

export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MTgzfS4yYzMxYzc2YzkzMjg5MGUxNzczZTI0ZDBiZjcyNTQyZWUyMTc2NjNj
export NXF_VER=23.10.0

nextflow run ./git/nf-core-rnaseq_3.14.0/3_14_0/ \
--input ./nf-core_samplesheet.csv \
--outdir ./nfout/build_38 \
--fasta /media/lucio/bioData1/2.References/fasta/UCSC_GRCm38/mm10.fa \
--gtf  /media/lucio/bioData1/2.References/gtf/ncbi/mm10_Build38/mm10.refGene.gtf.gz \
--aligner star_rsem \
--skip_umi_extract \
--save_reference \
-profile docker -with-tower -resume






## 4.5 export bigwig

In [ ]:
%%script bash

rsync --progress --append --inplace -av -e 'ssh -p 6022' lucio@house.mane.st:/media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4 \
/mnt/c/wkdir/Lara/RNAseq/rnaseq_lara_june23

rsync --progress --append --inplace -av -e 'ssh' lucio@maindevices.58-11-22-cc-7a-c7@cloud.shellhub.io:/media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4_random \
/mnt/datawk1/analysis/Lara/



In [ ]:
%%script bash

mkdir /mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/bamcoverage

files="/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0DoubleKO_REP1.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0DoubleKO_REP2.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0DoubleKO_REP3.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0Mll1KO_REP1.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0Mll1KO_REP2.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0Mll1KO_REP3.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0Mll2KO_REP1.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0Mll2KO_REP2.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0Mll2KO_REP3.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0WTA_REP1.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0WTA_REP2.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D0WTA_REP3.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4DoubleKO_REP1.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4DoubleKO_REP2.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4DoubleKO_REP3.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4MLL1KO_REP1.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4MLL1KO_REP2.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4MLL1KO_REP3.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4MLL2KO_REP1.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4MLL2KO_REP2.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4MLL2KO_REP3.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4WT_REP1.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4WT_REP2.markdup.sorted.bam \
/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/star_rsem/D4WT_REP3.markdup.sorted.bam"

for i in ${files} ; do prefix=`basename ${i} .bam` ; echo $prefix ; done

for i in ${files} ; do
prefix=`basename ${i} .bam` ; echo $prefix
sudo docker run -v /mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/:/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/ \
quay.io/biocontainers/mulled-v2-eb9e7907c7a753917c1e4d7a64384c047429618a:62d1ebe2d3a2a9d1a7ad31e0b902983fa7c25fa7-0 bamCoverage \
            --bam $i \
            --normalizeUsing 'RPGC' --effectiveGenomeSize 2494787188 \
            --numberOfProcessors 12 \
            --outFileName /mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/bamcoverage/${prefix}.bigWig

done

## TODEL 5.test statistical analysis

In [ ]:
%%script bash

In [ ]:
%%script R

gitwk="C:/gitwk/bioinfoGenerals/RNAseq/bin"
source(paste0(gitwk, "/fun_statistical.R"))
wkdir="C:/wkdir/Lara/RNAseq"

#unite unique counts
rawfolder="C:/wkdir/Lara/RNAseq/star_sem"
files<-list.files(rawfolder, pattern=".genes")


for (i in 1:length(files)){
  intab<-read.table(paste0(rawfolder,"/",files[i]),header=TRUE)
  midtab<-cbind(intab$gene_id,intab$expected_count)
  colnames(midtab)<-c("gene_id",files[i])

  if(i ==1 ){
    outtab <- midtab
  }else{
    outtab <- merge(outtab, midtab, by=1)
  }

}

colnames(outtab)<-gsub(".genes.results","",colnames(outtab))

write.table(outtab, file="C:/wkdir/Lara/RNAseq/star_sem/rawcounts.tsv.csv",
            sep="\t" , col.names=TRUE, row.names=FALSE, quote=FALSE)

# colnames(outtab) <- gsub(".genes.results", "",colnames(outtab))
  annotations.path <- "C:/wkdir/Lara/RNAseq/gencode.vM10.annotation.gtf.gz"
#
   annotations<-read.table(annotations.path, sep="\t", header=FALSE)
#
# annotations<-read.table("C:/wkdir/Lara/RNAseq/gencode.vM10.annotation.gtf.gz", sep="\t", header=FALSE)
#
  annotations<-annotations[which(annotations[,3]=="transcript"),]
#
  gene.id<-grep("gene_id", unlist(strsplit(annotations[,9], "; ")),value=TRUE)
  gene.name<-grep("gene_name", unlist(strsplit(annotations[,9], "; ")),value=TRUE)
#
  anno.2<-data.frame(gene.id,gene.name)
  #colnames(anno.2)<-c("gene_id","gene_name")
  anno.2$gene.id<-gsub("gene_id ", "", anno.2$gene.id)
  anno.2$gene.id<-gsub("\\..*","",anno.2$gene.id)
  anno.2$gene.name<-gsub("gene_name ", "", anno.2$gene.name)
  anno.3<-anno.2[which(duplicated(anno.2)==FALSE),]

  length(unique(anno.3$gene.id))==length(anno.3$gene.id)
  length(unique(anno.3$gene.name))==length(anno.3$gene.name)
  anno.3$gene.name[which(duplicated(anno.3$gene.name)==TRUE)]<-
    paste0("dupgene",which(duplicated(anno.3$gene.name)==TRUE),"_",
         anno.3$gene.name[which(duplicated(anno.3$gene.name)==TRUE)])

  write.table(anno.3, file="C:/wkdir/Lara/RNAseq/gencode_GRCm38_p4_annotations.tsv", row.names = FALSE, col.names = TRUE, quote=FALSE, sep="\t")

differential(prjFolder=wkdir,
             prjName = "Lara_RNAseq_day0_4",
             rawCounts = "C:/wkdir/Lara/RNAseq/star_sem/rawcounts.tsv.csv",
             designTablePath ="C:/wkdir/Lara/RNAseq/SS_RNAseq_Lara_edgeR_SS_D0.tsv",
             annoPath = "C:/wkdir/Lara/RNAseq/gencode_GRCm38_p4_annotations.tsv",
             previous = FALSE,
             accurateFiltering = FALSE
             )

## 6. run statistics analysis

### a) create environment ✅

In [ ]:
%%script bash

mkdir /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4
mkdir /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4/wkdir
mkdir /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4/nfout
mkdir /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4/in
mkdir /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4/git

cd /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4/git

git clone git@github.com:lucidif/bioinfoGenerals.git
git clone git@github.com:nf-core/differentialabundance.git

sudo docker pull lucidif/edger:0.0.1
docker pull nfcore/tools:2.13.1


cd /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4

#cp /mnt/datawk1/references/annotations/ENSEMBL_102/mm10/ncbi_compatiple_Mus_musculus.GRCm38.102.gtf ./

cp /mnt/datawk1/references/annotations/ncbi/mm10_Build38/mm10.refGene.gtf.gz ./in


### b) import data

are present in the assets of git repository

In [ ]:
%%script bash

#rsync --progress --append --inplace -av -e 'ssh -p 6022' \
#lucio@house.mane.st:/media/lucio/bioData1/2.Datasets/Lara/RNAseq/rnaseq_lara_june23 \
#/mnt/datawk1/data/Lara/RNAseq/

#rsync --progress --append --inplace -av -e 'ssh' \
#lucio@maindevices.58-11-22-cc-7a-c7@cloud.shellhub.io:/media/lucio/wkssd/bioinfo/Lara/RNAseq_lara_day0_4/star_rsem/rawcounts.txt \
#/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4/

#rsync --progress --append --inplace -av -e 'ssh' \
#lucio@maindevices.58-11-22-cc-7a-c7@cloud.shellhub.io:/media/lucio/bioData1/2.References/gtf/ENSEMBL_102/fixed_Mus_musculus.GRCm38.102.gtf \
#/mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4/

cd /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4

cp /media/lucio/external.wk/bioinfo/wkdir/Lara/Lara_multiomic_analysis/in/build38_DEseq2_RNAseq/* ./in




### TODEL c) start analysis with edgeR

In [ ]:
%%script bash

sudo docker run -it -v /mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4:/wkdir lucidif/edger:0.0.1

cd ./wkdir

R

source("./git/bioinfoGenerals/RNAseq/bin/fun_statistical.R")

differential(prjFolder="./",
             prjName = "Lara_RNAseq_day0_4",
             rawCounts = "rawcounts.txt",
             designTablePath ="./git/bioinfoGenerals/assets/SS_RNAseq_Lara_edgeR_SS_D0.tsv",
             annoPath = "fixed_Mus_musculus.GRCm38.102.gtf",
             previous = FALSE,
             accurateFiltering = FALSE
             )

quit()

exit


### d) statistical analysis with DEseq2

#### i- format table ✅

In [ ]:
%%R

setwd("/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4")

gitwk="./git/bioinfoGenerals/RNAseq/bin"
source(paste0(gitwk, "/fun_statistical.R"))
wkdir="/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4"

#unite unique counts
rawfolder="/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4/in"
files<-list.files(rawfolder, pattern=".genes")


for (i in 1:length(files)){
  intab<-read.table(paste0(rawfolder,"/",files[i]),header=TRUE)
  midtab<-cbind(intab$gene_id,intab$expected_count)
  colnames(midtab)<-c("gene_id",files[i])

  if(i ==1 ){
    outtab <- midtab
  }else{
    outtab <- merge(outtab, midtab, by=1)
  }

}

colnames(outtab)<-gsub(".genes.results","",colnames(outtab))

write.table(outtab, file=paste0(rawfolder,"/rawcounts.tsv"),
            sep="\t" , col.names=TRUE, row.names=FALSE, quote=FALSE)


#### ii- run statistics

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4

sudo nextflow run ./git/differentialabundance \
     --input SS_RNAseq_Lara_DEseq_input.tsv \
     --contrasts SS_RNAseq_Lara_DEseq_contrasts.tsv \
     --matrix  ./in/rawcounts.tsv \
     --gtf ./in/mm10.refGene.gtf.gz \
     --outdir nfout/build38/differentialabundance \
     -profile rnaseq,docker

sudo nextflow run ./git/differentialabundance \
     --input SS_RNAseq_Lara_DEseq_input.tsv \
     --contrasts SS_RNAseq_Lara_DEseq_ALLcontrasts.tsv \
     --matrix  ./in/rawcounts.tsv \
     --gtf ./in/mm10.refGene.gtf.gz \
     --outdir nfout/build38/differentialabundance_ALLcomp \
     -profile rnaseq,docker



#### make expression bar plot

In [ ]:
%%script R

#load libraries

library(GenomicFeatures)
library(edgeR)
library(ggplot2)

#TODO transform fpkm in function

#parameters

setwd("/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4")

annotations<-"/mnt/datawk1/references/annotations/ncbi/mm10_Build38/mm10.refGene.gtf.gz"

#start analysis

txdb <- makeTxDbFromGFF(annotations)

tx_lengths <- transcriptLengths(txdb, with.cds_len = TRUE)
longest_tx_per_gene <- aggregate(tx_lengths$tx_len, by=list(tx_lengths$gene_id), max)

colnames(longest_tx_per_gene) <- c("GeneID", "Length")

write.csv(longest_tx_per_gene, "/mnt/datawk1/references/annotations/ncbi/mm10_Build38/longest_transcript_lengths.csv", row.names = FALSE)

D0.DKOvsWT<-read.table("nfout/build38/differentialabundance_ALLcomp/tables/differential/D0_DKO_vs_WT.deseq2.results.tsv", sep="\t", header = TRUE)

sign_dodko<-D0.DKOvsWT[which(D0.DKOvsWT$padj<=0.05),]
undiff<-D0.DKOvsWT[which(D0.DKOvsWT$padj>0.05 | is.na(D0.DKOvsWT$padj) ),]

sign_dodko<-sign_dodko[which(sign_dodko$log2FoldChange<=-1 ),]

rawcounts<-read.table("/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4/in/rawcounts.tsv",
                      sep="\t",
                      header=TRUE
)

raw.glength<-merge(longest_tx_per_gene,rawcounts, by.x="GeneID", by.y="gene_id")


samples<-colnames(raw.glength)[3:length(colnames(raw.glength))]

for (i in 1:length(samples)){
  print(i)
  sam<-samples[i]
  counts <- raw.glength[,sam]
  genelengths<-raw.glength[,"Length"]
  dge <- DGEList(counts = counts)
  fpkm <- rpkm(dge, gene.length = genelengths)

  if(i==1){
    fpkm.tab<-data.frame(raw.glength[,1],fpkm)
    colnames(fpkm.tab)<-c("geneID",sam)
  }else{
    fpkm.tab<-cbind(fpkm.tab,fpkm)
    colnames(fpkm.tab)[length(colnames(fpkm.tab))]<-sam
  }

}

sign_dodko

#which(sign_dodko$gene_id %in% fpkm.tab$geneID)
down.genes<-fpkm.tab[which( fpkm.tab$geneID %in% sign_dodko$gene_id),]
undiff.fpkm<-fpkm.tab[which(fpkm.tab$geneID %in% undiff$gene_id),]
all.fpkm<-fpkm.tab

#groups<-c("D0DoubleKO", "D0Mll1KO","D0Mll2KO", "D0WTA")
groups<-c("D0DoubleKO", "D0WTA")

for (i in 1:length(groups)){

  tarval<-data.frame(down.genes[,grep(paste0(groups[i],"_"),colnames(down.genes))])
  tarmean<-data.frame(down.genes[,1],rowMeans(tarval))

  undiff.val<-data.frame(undiff.fpkm[,grep(paste0(groups[i],"_"),colnames(undiff.fpkm))])
  undiff.means<-data.frame(undiff.fpkm[,1],rowMeans(undiff.val))

  all.val <- data.frame(all.fpkm[,grep(paste0(groups[i],"_"),colnames(all.fpkm))])
  all.mean <- data.frame(all.fpkm[,1],rowMeans(all.val))

  out<-rbind(
    data.frame(gene=tarmean[,1],
               fpkm.mean=tarmean[,2],
               type=rep("down",nrow(tarmean))
    ),
    data.frame(gene=all.mean[,1],
               fpkm.mean=all.mean[,2],
               type=rep("all",nrow(all.mean))
    )
  )

  assign(groups[i],out)

}

# ggplot(D0DoubleKO, aes(x=type, y=log(fpkm.mean))) +
#   geom_boxplot(outlier.colour="red", outlier.shape=8,
#                outlier.size=4,
#                )

# ggplot(D0WTA, aes(x = type, y = log2(fpkm.mean), fill=type)) +
#    geom_boxplot(outlier.shape = NA) +  # Nasconde gli outlier
#    ggtitle("D0WTA gene fpkm")


head(D0WTA)

all<-rbind(
  cbind(D0WTA,line=rep("D0WTA",nrow(D0WTA))),
  cbind(D0DoubleKO,line=rep("D0DoubleKO",nrow(D0DoubleKO)))
  )

all$line <- factor(all$line, levels = c("D0WTA", "D0DoubleKO"))

#c064ad
#7879b9

ggplot(all, aes(x = type, y = fpkm.mean, colour=line)) +
  geom_boxplot(outlier.shape = NA, position=position_dodge(1)) +  # Nasconde gli outlier
  ggtitle("fpkm boxplot") +
  coord_cartesian(ylim = c(0, 25)) +
  theme_minimal() +
  scale_colour_manual(values = c("D0WTA" = "#c064ad", "D0DoubleKO" = "#7879b9"))+
  theme(legend.title = element_blank())


# ggplot(df, aes(x = variabile_x, y = variabile_y)) +
#   geom_boxplot() +
#   scale_x_continuous(limits = c(min, max))


#
# ggplot(D0DoubleKO, aes(x = type, y = log2(fpkm.mean+1), fill=type)) +
#   geom_boxplot(outlier.shape = NA) +  # Nasconde gli outlier
#   ggtitle("D0DoubleKO")
#
# ggplot(D0DoubleKO, aes(x = type, y = log2(fpkm.mean), fill=type)) +
#   geom_boxplot(outlier.shape = NA) +  # Nasconde gli outlier
#   ggtitle("D0DoubleKO")
#

ggsave("othouts/fpkm_distribution/plot_fpkm.png", dpi = 300, width = 8, height = 6)

head(D0DoubleKO)

quit()

#### iii - run pca only TODO convert in fun of general bioinfo scripts

In [ ]:
%%R

inrw<-read.table("in/rawcounts.tsv",
                 sep="\t",
                 header=TRUE
                 )

inrw.d0<-inrw[,which(grepl("^D0", colnames(inrw)))]

inrw.d4<-inrw[,which(grepl("^D4", colnames(inrw)))]

library(ggfortify)

pca_tab_d0<-t(log2(inrw.d0 + 1))

pca_res_d0 <- prcomp(t(log2(inrw.d0 + 1)))

data_d0<-cbind(row.names(pca_tab_d0),
            rep(NA,length(row.names(pca_tab_d0))),
            rep(NA,length(row.names(pca_tab_d0)))
            )
row.names(data_d0)<-data_d0[,1]
colnames(data_d0)<-c("sample","colour","group")

data_d0[grep("D0DoubleKO_",data_d0[,1]),2]<-"#ffb00e" #"DKO"
data_d0[grep("D0Mll1KO_",data_d0[,1]),2]<-"#de217d" #"Mll1KO"
data_d0[grep("D0Mll2KO_",data_d0[,1]),2]<-"#ff5f01" #"Mll2KO"
data_d0[grep("D0WTA_",data_d0[,1]),2]<-"#715eee" #"WTA"

data_d0[grep("D0DoubleKO_",data_d0[,1]),3]<-"DKO"
data_d0[grep("D0Mll1KO_",data_d0[,1]),3]<-"Mll1KO"
data_d0[grep("D0Mll2KO_",data_d0[,1]),3]<-"Mll2KO"
data_d0[grep("D0WTA_",data_d0[,1]),3]<-"WTA"


# data2[,3]<-factor(as.character(data2[,3]),levels=c("WTA",
#                                  "Mll1KO",
#                                  "Mll2KO",
#                                  "DKO"
#                                  ))

#data mast have same sort of entry in pca_res$x
# the sort of pca_res$x is by alphabetic letter becouse the factors levels are ordered in this way
autoplot(pca_res_d0, data = data_d0, colour = 'group', size=5) +
        scale_color_manual(values=c("#ffb00e", "#de217d", "#ff5f01", "#715eee" ))+
        theme(legend.title=element_blank())

#autoplot(pca_res, data = ordered_data, colour = 'group', size=5) +
#  scale_color_manual(values=c("#ffb00e", "#de217d", "#ff5f01", "#715eee" ))

        #geom_text(label=row.names(pca_res$x),colour=data[,2])

#scale_fill_manual(values=c("#ffb00e", "#de217d", "#ff5f01", "#715eee" ))

#+
#  geom_text(label=rownames(mtcars))


pca_tab_d4<-t(log2(inrw.d4 + 1))

pca_res_d4 <- prcomp(t(log2(inrw.d4 + 1)))

data_d4<-cbind(row.names(pca_tab_d4),
               rep(NA,length(row.names(pca_tab_d4))),
               rep(NA,length(row.names(pca_tab_d4)))
)
row.names(data_d4)<-data_d4[,1]
colnames(data_d4)<-c("sample","colour","group")

data_d4[grep("D4DoubleKO_",data_d4[,1]),2]<-"#ffb00e" #"DKO"
data_d4[grep("D4MLL1KO_",data_d4[,1]),2]<-"#de217d" #"Mll1KO"
data_d4[grep("D4MLL2KO_",data_d4[,1]),2]<-"#ff5f01" #"Mll2KO"
data_d4[grep("D4WTA_",data_d4[,1]),2]<-"#715eee" #"WTA"

data_d4[grep("D4DoubleKO_",data_d4[,1]),3]<-"DKO"
data_d4[grep("D4MLL1KO_",data_d4[,1]),3]<-"Mll1KO"
data_d4[grep("D4MLL2KO_",data_d4[,1]),3]<-"Mll2KO"
data_d4[grep("D4WT_",data_d4[,1]),3]<-"WTA"

autoplot(pca_res_d4, data = data_d4, colour = 'group', size=5) +
  scale_color_manual(values=c("#ffb00e", "#de217d", "#ff5f01", "#715eee" ))+
  theme(legend.title=element_blank())




#### iv - run volcano only

In [ ]:
%%R

#setwd("/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4")

library("ggplot2")

infile="./nfout/build38/differentialabundance/tables/differential/D0_DKO_vs_WT.deseq2.results.tsv"

degs_dko_wt<-read.table(infile, sep="\t",
                        header=TRUE
)

degs_dko_wt_toplot <- as.data.frame(degs_dko_wt[ , c("gene_id", "log2FoldChange", "padj") ] )

length(which(degs_dko_wt_toplot$padj <= 0.05 & degs_dko_wt_toplot$log2FoldChange <= -1 ))

#degs_dko_wt_toplot<-degs_dko_wt_toplot[which(!is.na(degs_dko_wt_toplot$padj)),]
nona<-degs_dko_wt_toplot[which(!is.na(degs_dko_wt_toplot$padj)),]

signi<-c()

signi[which(degs_dko_wt_toplot$padj<=0.05 & is.na(degs_dko_wt_toplot$padj)==FALSE)]<-"significant"
signi[which(degs_dko_wt_toplot$padj>0.05)]<-"unsignificant"


degs_dko_wt_toplot<-cbind(degs_dko_wt_toplot, signi)

updown<-rep(NA,nrow(degs_dko_wt_toplot))

updown[which(degs_dko_wt_toplot$log2FoldChange<=log2(0.5) &
               degs_dko_wt_toplot$signi == "significant"
)]<-"down"
updown[which(degs_dko_wt_toplot$log2FoldChange>=log2(2) &
               degs_dko_wt_toplot$signi == "significant"
)]<-"up"

degs_dko_wt_toplot<-cbind(degs_dko_wt_toplot,updown)

degs_dko_wt_toplot[which(is.na(degs_dko_wt_toplot$updown)),"updown"]<-"undif"

degs_dko_wt_toplot<-degs_dko_wt_toplot[which(!is.na(degs_dko_wt_toplot$padj)),]


ggplot(degs_dko_wt_toplot,
       aes(x = log2FoldChange,
           y = -log10(padj),
           color = updown
       )
) +
  geom_point() +
  geom_hline(yintercept = -log10(0.05),
             linetype = "dashed") +
  geom_vline(xintercept = c(log2(0.5), log2(2)),
             linetype = "dashed") +
  scale_color_manual(values = c("down" = "#453291", "up" = "#5a90cc", "undif" = "grey"))+
  theme_minimal()
#+
#  scale_y_continuous(limits = c(0, 20))
  ggsave("./othouts/volcano/d0_dko_vs_wt.png", width = 10, height = 6, dpi = 300)


#### D4 preliminary analysis of differential expression results

In [ ]:
%%script R 

source("./git/Lara_MLL_2/bin/D4_prelim_analysis_DE_results.R")

## 7. Gene Onthology

In [ ]:
%%script bash

setwd("/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4")
source("./git/bioinfoGenerals/RNAseq/bin/GOanalysis.R")

log2FC=1
pval=0.05

annotations<-read.table("/mnt/datawk1/references/annotations/ncbi/mm10_Build38/mm10.refGene.gtf.gz", sep="\t",header=FALSE)

allsD0<-read.table("nfout/build38/differentialabundance_ALLcomp/tables/differential/D0_DKO_vs_WT.deseq2.results.tsv", header=TRUE)

degsD0<-allsD0[ which(allsD0$padj <= 0.05 ) , ]
degsD0<-degsD0[ which(degsD0$log2FoldChange <= -1 | degsD0$log2FoldChange >= 1 ) , ]

degsNames<-degsD0$gene_id
degsNames<-cbind(gene_id=degsNames, rep(1,length(degsNames)))

pre.compose<-merge(allsD0,degsNames, all.x=TRUE)
pre.compose$V2[which(is.na(pre.compose$V2))]<-0



dbfile1="/mnt/datawk1/references/annotations/biomart/v102/mart_export_GOterms_extended.txt"
dbfile2="/mnt/datawk1/references/annotations/biomart/v102/mart_export_v102_GRCm38.p6.txt"

#geneLengthFile="/home/lucio/MEGA/bioinformatics/data/QuantSeq/2021/commitments/Imparato/maxlength_byGene.txt"

termReduction=FALSE
filterBy="padj"

library(goseq)
library(rrvgo)

db<-read.delim(dbfile1,sep="\t",header=TRUE)

#db2<-read.delim(dbfile2,sep="\t",header=TRUE)

db<-db[,c("Gene.stable.ID","Gene.name", "Gene.start..bp.", "Gene.end..bp.", "GO.term.accession")]
colnames(db)<-c("Ensembl.Gene.ID","Gene.name","Gene.start","Gene.end","GO.Term.Accession")

db<-db[which(duplicated(db)==FALSE),]

db2<-db[,c("Gene.name","GO.Term.Accession")]

#gl<-read.delim(geneLengthFile,sep="\t",header=TRUE)

#db[which( compose %in% db$Gene.name ),]

#targ<-db[which( db$Gene.name %in% compose ),]

#nrow(targ)
#length(unique(targ$Gene.name))
#length(compose)

#grep("Slc39a8",compose)

# targ.ensembl<-targ[,c("Ensembl.Gene.ID",
#                      "GO.Term.Accession",
#                      "Gene.start",
#                      "Gene.end"
#                      )]

targ.gensyn<-db[,c("Gene.name",
                      "GO.Term.Accession",
                      "Gene.start",
                      "Gene.end"
)]

#=====rimuovi i genesym che hanno diverse coordinate ma stesso nome
#non sono molti mantieni quelli associati a piu' go term ?
duplicated(targ.gensyn$Gene.name)
# united<-paste(targ.gensyn$Gene.name,
#               targ.gensyn$Gene.start,
#               targ.gensyn$Gene.end
#               ,sep="_")
#

unigens<-unique(targ.gensyn$Gene.name)

gene.to.remove<-c()

for (i in 1:length(unigens)){
  print(i)
  targen<-unigens[i]
  #targen<-"Rtl8b"
  #targen<-"1700030C10Rik"

  taranno<-targ.gensyn[which(targ.gensyn$Gene.name==targen),]
  divergent.anno<-taranno[which(duplicated(taranno[,-2])==FALSE),-2]

  if(nrow(divergent.anno)>1){


    #seleziono l'annotazione che ha piu' termini di GO
    anno.counts<-table(paste(taranno$Gene.name,
                             taranno$Gene.start,
                             taranno$Gene.end
                             ,sep="_"))

    if(length(which(anno.counts==max(anno.counts)))>1){
     #mx<-min(which(anno.counts==max(anno.counts)))
     anno.counts[min(which(anno.counts==max(anno.counts)))]<-anno.counts[min(which(anno.counts==max(anno.counts)))]+1
    }

    toremove.anno<-names(anno.counts[which(anno.counts!=max(anno.counts))])
    if(length(toremove.anno)==0){
      toremove.anno<-names(anno.counts[2:length(anno.counts)])
    }

    if(i ==1) {
      gene.to.remove <- toremove.anno
    }else{
      gene.to.remove <- c(gene.to.remove, toremove.anno)
    }
  }

}

targ.gensyn2<-targ.gensyn

#remove the annotations
for(i in 1:length(gene.to.remove)){

  coords<-strsplit(gene.to.remove[i],"_")[[1]]
  targ.gensyn2<-targ.gensyn2[-which(targ.gensyn2$Gene.name==coords[1] &
          targ.gensyn2$Gene.start==as.numeric(coords[2]) &
          targ.gensyn2$Gene.end==as.numeric(coords[3])
        ),]

}

#targ.gensyn[which(targ.gensyn$Gene.name=="Arhgef4"
#      ),]

#=================================



#=================================


gl<-targ.gensyn2[,
                 c("Gene.name","Gene.start","Gene.end")]

#gl<-gl[which(duplicated(gl$Gene.name)==FALSE),]

#gl[grep("Arhgef4", gl$Gene.name),]

#targ.gensyn[grep("Arhgef4", targ.gensyn$Gene.name),]

#degsD0[which(degsD0$gene_id=="Arhgef4"),]

gl<-gl[which(duplicated(gl)==FALSE),]

Gene.length<-gl$Gene.end-gl$Gene.start

gl<-as.data.frame(cbind(Gene.name=gl$Gene.name,Gene.length))
gl$Gene.length<-as.numeric(gl$Gene.length)

pre.compose<-merge(pre.compose,gl,by.x=1,by.y=1)

compose<-as.numeric(pre.compose$V2)
names(compose)<-pre.compose$gene_id

#row.names(gl)<-gl$Gene.name

gnlen<-pre.compose$Gene.length
names(gnlen)<-pre.compose$gene_id

#gl<-gl[compose,]



#gl<-as.data.frame(gl[,-1])
#interested.gl

#1  Fitting the Probability Weighting Function (PWF)

par(mar=c(1,1,1,1))

pwf <- nullp(compose,bias.data=gnlen)

goterms<-goseq(pwf,gene2cat = db2)

goterms.allinfo<-cbind(goterms,p.adjust(goterms$over_represented_pvalue,method = "BH"))

goterms.allinfo<-cbind(goterms,p.adjust(goterms$over_represented_pvalue,method = "BH"))

colnames(goterms.allinfo)[ncol(goterms.allinfo)]<-"padj"

#length(which(goterms$padj<0.05))

#goterms.randomaSampling<-goseq(pwf, "hg38","geneSymbol",use_genes_without_cat=TRUE,method = "Sampling",repcnt=1000)

#Biological Process
#goterms.bp <- goseq(pwf, "hg38","geneSymbol",test.cats="GO:BP")

# Over-represented means that there are more DE genes in the
# category than we would expect given the size of the category and the
# gene length distribution so that would be enriched for DE genes.
# Under-represented means that there are fewer DE genes in the category
# than we would expect by chance. The p-value relates to the probability
# of observing this number of DE genes in the category by chance.

#Molecular Function

#goterms.mf <- goseq(pwf, "hg38","geneSymbol",test.cats="GO:MF")

#Cell Component
#goterms.cc <- goseq(pwf, "hg38","geneSymbol",test.cats="GO:CC")

#goterms.allinfo<-goseq(pwf, "hg38","geneSymbol",use_genes_without_cat=TRUE)


#goterms.allinfo<-rbind(goterms.bp,goterms.mf,goterms.cc)

goterms.sign<-goterms.allinfo[which(goterms.allinfo[,filterBy]<=0.05),]

goterms.sign<-goterms.sign[order(goterms.sign$over_represented_pvalue,by=goterms.sign$padj),]

goterm.final<-goterms.sign[,c("category", "padj","numDEInCat","numInCat","term","ontology")]

goterm.final<-cbind(goterm.final,goterm.final$numDEInCat/goterm.final$numInCat)

colnames(goterm.final)[ncol(goterm.final)]<-"ratio"

goterm.results<-goterm.final

if(length(which(!is.na(goterm.results$term)))>1){
  goterm.results<-goterm.results[which(!is.na(goterm.results$term)),]
}

godot(goterms =  goterm.results)


## 8. veen diagram

set environment

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4


run code

In [ ]:
%%script bash

#setwd("/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4")

#libraries

library(VennDiagram)
library(RColorBrewer)

#params

log2FC=1
pval=0.05

annotations<-read.table("/mnt/datawk1/references/annotations/ncbi/mm10_Build38/mm10.refGene.gtf.gz", sep="\t",header=FALSE)

allsD0_dko_vs_wt<-read.table("nfout/build38/differentialabundance_ALLcomp/tables/differential/D0_DKO_vs_WT.deseq2.results.tsv", header=TRUE)
allsD0_mll1ko_vs_wt<-read.table("nfout/build38/differentialabundance_ALLcomp/tables/differential/D0Mll1KO_vs_WT.deseq2.results.tsv", header=TRUE)
allsD0_mll2ko_vs_wt<-read.table("nfout/build38/differentialabundance_ALLcomp/tables/differential/D0Mll2KO_vs_WT.deseq2.results.tsv", header=TRUE)

degsD0_dko_vs_wt<-allsD0_dko_vs_wt[ which(allsD0_dko_vs_wt$padj <= 0.05 ) , ]
degsD0_dko_vs_wt<-degsD0_dko_vs_wt[ which(degsD0_dko_vs_wt$log2FoldChange <= -1 | degsD0_dko_vs_wt$log2FoldChange >= 1 ) , ]

dim(degsD0_dko_vs_wt)

degsD0_mll1ko_vs_wt<-allsD0_mll1ko_vs_wt[ which(allsD0_mll1ko_vs_wt$padj <= 0.05 ) , ]
degsD0_mll1ko_vs_wt<-degsD0_mll1ko_vs_wt[ which(degsD0_mll1ko_vs_wt$log2FoldChange <= -1 | degsD0_mll1ko_vs_wt$log2FoldChange >= 1 ) , ]

dim(degsD0_mll1ko_vs_wt)

degsD0_mll2ko_vs_wt<-allsD0_mll2ko_vs_wt[ which(allsD0_mll2ko_vs_wt$padj <= 0.05 ) , ]
degsD0_mll2ko_vs_wt<-degsD0_mll2ko_vs_wt[ which(degsD0_mll2ko_vs_wt$log2FoldChange <= -1 | degsD0_mll2ko_vs_wt$log2FoldChange >= 1 ) , ]

dim(degsD0_mll2ko_vs_wt)

myCol <- brewer.pal(3, "Pastel2")

venn.diagram(
        x = list(degsD0_dko_vs_wt$gene_id, degsD0_mll1ko_vs_wt$gene_id, degsD0_mll2ko_vs_wt$gene_id),
        category.names = c("dko vs wt" , "mll1ko vs wt" , "mll2ko vs wt"),
        filename = './othouts/venn/d0_venn.png',
        output=TRUE,

        # Output features
        imagetype="png" ,
        height = 480 ,
        width = 480 ,
        resolution = 300,
        compression = "lzw",

        # Circles
        lwd = 2,
        lty = 'blank',
        fill = myCol,

        # Numbers
        cex = .6,
        fontface = "bold",
        fontfamily = "sans",

        # Set names
        cat.cex = 0.5,
        cat.fontface = "bold",
        cat.default.pos = "outer",
        cat.pos = c(-27, 27, 135),
        cat.dist = c(0.055, 0.055, 0.085),
        cat.fontfamily = "sans",
        rotation = 1
)



#down only


degsD0_dko_vs_wt<-degsD0_dko_vs_wt[ which(degsD0_dko_vs_wt$log2FoldChange <= -1 ) , ]
degsD0_mll1ko_vs_wt<-degsD0_mll1ko_vs_wt[ which(degsD0_mll1ko_vs_wt$log2FoldChange <= -1  ) , ]
degsD0_mll2ko_vs_wt<-degsD0_mll2ko_vs_wt[ which(degsD0_mll2ko_vs_wt$log2FoldChange <= -1  ) , ]

venn.diagram(
        x = list(degsD0_dko_vs_wt$gene_id, degsD0_mll1ko_vs_wt$gene_id, degsD0_mll2ko_vs_wt$gene_id),
        category.names = c("dko vs wt" , "mll1ko vs wt" , "mll2ko vs wt"),
        filename = './othouts/venn/d0_venn_downDEGs.png',
        output=TRUE,

        # Output features
        imagetype="png" ,
        height = 480 ,
        width = 480 ,
        resolution = 300,
        compression = "lzw",

        # Circles
        lwd = 2,
        lty = 'blank',
        fill = myCol,

        # Numbers
        cex = .6,
        fontface = "bold",
        fontfamily = "sans",

        # Set names
        cat.cex = 0.5,
        cat.fontface = "bold",
        cat.default.pos = "outer",
        cat.pos = c(-27, 27, 135),
        cat.dist = c(0.055, 0.055, 0.085),
        cat.fontfamily = "sans",
        rotation = 1
)

DEGs_down_D0<-rbind(
  cbind(degsD0_dko_vs_wt$gene_id,rep("degsD0_dko_vs_wt",length(degsD0_dko_vs_wt$gene_id))),
  cbind(degsD0_mll1ko_vs_wt$gene_id,rep("degsD0_mll1ko_vs_wt",length(degsD0_mll1ko_vs_wt$gene_id))),
  cbind(degsD0_mll2ko_vs_wt$gene_id,rep("degsD0_mll2ko_vs_wt",length(degsD0_mll2ko_vs_wt$gene_id)))
)

colnames(DEGs_down_D0)<-c("gene","category")

write.table(DEGs_down_D0,
            file="./othouts/venn/DEGs_down_D0.tsv",
            sep="\t",
            col.names = TRUE,
            row.names = FALSE,
            quote=FALSE
            )










## 9. TE expression

### Set environment

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4

mkdir othouts/TEtranscripts
mkdir in/RNAseq_lara_day0_4_random
mkdir in/RNAseq_lara_day0_4

#mount ssd 
sudo docker pull mhammelllab/tetranscripts:2.2.3

ssh lucio@maindevices.58-11-22-cc-7a-c7@cloud.shellhub.io

sudo mkdir -p /mnt/micron
sudo mount /dev/sdc1 /mnt/micron

exit

#import bam and bai files

rsync -avz --progress \
  lucio@maindevices.58-11-22-cc-7a-c7@cloud.shellhub.io:/mnt/micron/analysis/RNAseq_lara_day0_4_random/nfout/build_38/star_rsem/*.bam* \
  in/RNAseq_lara_day0_4_random/

#
rsync -avz --progress \
  lucio@maindevices.58-11-22-cc-7a-c7@cloud.shellhub.io:/media/lucio/wkssd_crucial/analysis/RNAseq_lara_day0_4/nfout/build_38/star_rsem/*.bam* \
  in/RNAseq_lara_day0_4


### start analysis

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4/othouts/TEtranscripts

sudo docker run -v `pwd`:`pwd` -w `pwd` -it mhammelllab/tetranscripts:2.2.3 TEtranscripts --help

sudo docker run \
  -v /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4:/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4 \
  -v /mnt/datawk1/references:/mnt/datawk1/references \
  -w `pwd` -u $(id -u):$(id -g) \
  -it mhammelllab/tetranscripts:2.2.3 TEtranscripts \
  --sortByPos --format BAM --mode multi \
  -t ../../in/RNAseq_lara_day0_4_random/D0DoubleKO_REP1.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4_random/D0DoubleKO_REP2.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4_random/D0DoubleKO_REP3.markdup.sorted.bam \
  -c ../../in/RNAseq_lara_day0_4_random/D0WTA_REP1.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4_random/D0WTA_REP2.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4_random/D0WTA_REP3.markdup.sorted.bam \
  --GTF /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
  --TE /mnt/datawk1/references/annotations/TEtranscripts/mm10_rmsk_TE.gtf \
  --project TEexpr_WTAd0_VS_DKOd0

sudo docker run \
  -v /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4:/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4 \
  -v /mnt/datawk1/references:/mnt/datawk1/references \
  -w `pwd` -u $(id -u):$(id -g) \
  -it mhammelllab/tetranscripts:2.2.3 TEtranscripts \
  --sortByPos --format BAM --mode multi \
  -t ../../in/RNAseq_lara_day0_4_random/D4DoubleKO_REP1.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4_random/D4DoubleKO_REP2.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4_random/D4DoubleKO_REP3.markdup.sorted.bam \
  -c ../../in/RNAseq_lara_day0_4_random/D4WT_REP1.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4_random/D4WT_REP2.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4_random/D4WT_REP3.markdup.sorted.bam \
  --GTF /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
  --TE /mnt/datawk1/references/annotations/TEtranscripts/mm10_rmsk_TE.gtf \
  --project TEexpr_WTAd4_VS_DKOd4

#TODO
#unique on unique reads

sudo docker run \
  -v /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4:/mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4 \
  -v /mnt/datawk1/references:/mnt/datawk1/references \
  -w `pwd` -u $(id -u):$(id -g) \
  -it mhammelllab/tetranscripts:2.2.3 TEtranscripts \
  --sortByPos --format BAM --mode uniq \
  -t ../../in/RNAseq_lara_day0_4/D0DoubleKO_REP1.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4/D0DoubleKO_REP2.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4/D0DoubleKO_REP3.markdup.sorted.bam \
  -c ../../in/RNAseq_lara_day0_4/D0WTA_REP1.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4/D0WTA_REP2.markdup.sorted.bam \
  ../../in/RNAseq_lara_day0_4/D0WTA_REP3.markdup.sorted.bam \
  --GTF /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf \
  --TE /mnt/datawk1/references/annotations/TEtranscripts/mm10_rmsk_TE.gtf \
  --project unique_TEexpr_WTAd0_VS_DKOd0

### other elements ecpression

In [ ]:
%%script R

source ("git/Lara_MLL2/bin/TEexpression.R")


# microC



## 1. Download files


reformat in table

In [ ]:
%%R

reformat<-function(intable, outfile){

  #intable="/mnt/datawk1/data/Lara/microC/HN00214039_20samples_md5sum_DownloadLink.txt"
  #File	Size	md5sum	Download_link
  #WT_day4_A_LP2_1.fastq.gz 1,062,036,564 21f745cea5e09b7047bb283ea9a56027  https://data.macrogen.com/~macro3/HiSeq02/20240327/HN00214039/WT_day4_A_LP2_1.fastq.gz
  #WT_day4_A_LP2_2.fastq.gz 1,111,773,662 fe1da9df82b75e9ace14a98f0ed79e90  https://data.macrogen.com/~macro3/HiSeq02/20240327/HN00214039/WT_day4_A_LP2_2.fastq.gz

  #outfile="/mnt/datawk1/data/Lara/microC/SS_microC_HM00214039.tsv"


  dwnlink<-read.table(intable,
           header=TRUE)

  name<-dwnlink[,1]
  name<-gsub("_1.fastq.gz", "", dwnlink[,1])
  name<-gsub("_2.fastq.gz", "", name)

  uname<-unique(name)

  link1<-c()
  link2<-c()
  md5_1<-c()
  md5_2<-c()

  for (j in 1:length(uname)){

    link1[j]<-dwnlink[grep(paste0(uname[j], "_1.fastq.gz"), dwnlink[,4]),4]
    link2[j]<-dwnlink[grep(paste0(uname[j], "_2.fastq.gz"), dwnlink[,4]),4]
    md5_1[j]<-dwnlink[grep(paste0(uname[j], "_1.fastq.gz"), dwnlink[,4]),3]
    md5_2[j]<-dwnlink[grep(paste0(uname[j], "_2.fastq.gz"), dwnlink[,4]),3]

  }

  reformatTable<-cbind(sample=uname,link1,link2,md5sum1=md5_1,md5sum2=md5_2)

  write.table(reformatTable, file=outfile, sep="\t",
              row.names = FALSE,
              col.names=TRUE,
              quote=FALSE)
}


#/mnt/datawk1/data/Lara/2024_05_Lara_microC/

reformat(intable="/mnt/datawk1/data/Lara/microC/HN00214039_20samples_md5sum_DownloadLink.txt",
         outfile="/mnt/datawk1/data/Lara/microC/SS_microC_HM00214039.tsv")

reformat(intable="/mnt/datawk1/data/Lara/2024_05_Lara_microC/2024_05_SS_microC.csv",
         outfile="/mnt/datawk1/data/Lara/2024_05_Lara_microC/StdIn_2024_05_microC.csv")

reformat(intable="/mnt/datawk1/data/Lara/2024_05_Lara_microC/2024_05_SS_batch2.csv",
         outfile="/mnt/datawk1/data/Lara/2024_05_Lara_microC/StdIn_2024_05_batch2.csv")

reformat(intable="/mnt/datawk1/data/Lara/2024_05_Lara_microC/2024_05_SS_batch3.csv",
         outfile="/mnt/datawk1/data/Lara/2024_05_Lara_microC/StdIn_2024_05_batch3.csv")





download and check

In [ ]:
%%script bash

bash /mnt/datawk1/analysis/Lara/2024_05_Lara_microC/git/nf-core-microc/bin/download_from_macrogen.sh \
    /mnt/datawk1/data/Lara/microC/SS_microC_HM00214039.tsv \
    /mnt/datawk1/data/Lara/microC/HN00214039_fastq/

bash /mnt/datawk1/analysis/Lara/2024_05_Lara_microC/git/nf-core-microc/bin/download_from_macrogen.sh \
    /mnt/datawk1/data/Lara/2024_05_Lara_microC/StdIn_2024_05_microC.csv \
    /mnt/datawk1/data/Lara/2024_05_Lara_microC/fastq/

bash /mnt/datawk1/analysis/Lara/2024_05_Lara_microC/git/nf-core-microc/bin/download_from_macrogen.sh \
    /mnt/datawk1/data/Lara/2024_05_Lara_microC/StdIn_2024_05_batch2.csv \
    /mnt/datawk1/data/Lara/2024_05_Lara_microC/fastq_batch2/

bash /mnt/datawk1/analysis/Lara/2024_05_Lara_microC/git/nf-core-microc/bin/download_from_macrogen.sh \
    /mnt/datawk1/data/Lara/2024_05_Lara_microC/StdIn_2024_05_batch3.csv \
    /mnt/datawk1/data/Lara/2024_05_Lara_microC/fastq_batch3/



## 2. test microC analysis

In [ ]:
cd /mnt/datawk1/analysis/Lara/test_microc/

mkdir git

cd ./git

git clone git@github.com:lucidif/nf-core-microc.git

docker tag lucidif/microc:0.0.1 quay.io/lucidif/microc:0.0.1

docker run -itv `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) nfcore/tools modules install


cd ..

sudo nextflow run git/nf-core-microc \
   -profile docker \
   --input toy_samplesheet.csv \
   --fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
   --index /mnt/datawk1/references/UCSC_GRCm38_bwa\
   --outdir /nfout


#test on real sample

sudo nextflow run git/nf-core-microc \
   -profile docker \
   --input realSample_SS.csv \
   --fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
   --index /mnt/datawk1/references/UCSC_GRCm38_bwa\
   --outdir /nfout -resume

#test on proprietary dataset

cd /mnt/datawk1/analysis/Lara/2024_03_Lara_microC

nextflow run git/nf-core-microc -profile docker \
--input SS_onesample_microC_NFSS_2024_03_microC_Lara.csv \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--index /mnt/datawk1/references/UCSC_GRCm38_bwa \
--juicertool_location /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/git/nf-core-microc/bin/juicertools.jar \
--skip_juicer false \
--outdir ./nfout --exclude_lcextrap false -resume



#test in bash

cd /mnt/datawk1/analysis/Lara/2024_03_Lara_microC
sudo docker run -itv `pwd`:`pwd` -w `pwd` lucidif/microc:0.0.1

path="/mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/cut_fastq/KO_day0_A_1.merged.fastq.gz"

sh /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/git/nf-core-microc/bin/std_pipe.sh \
"/mnt/datawk1/analysis/Lara/2024_03_Lara_microC/work/bashtest/UCSC_GRCm38_bwa" \
"/mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/cut_fastq/KO_day4_A_1.merged.fastq.gz" \
"/mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/cut_fastq/KO_day4_A_2.merged.fastq.gz" \
"/mnt/datawk1/analysis/Lara/2024_03_Lara_microC/work/bashtest/resout"




test re microc pipeline

In [ ]:
cd /mnt/datawk1/analysis/Lara/test_microC_RE


export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MTgzfS4yYzMxYzc2YzkzMjg5MGUxNzczZTI0ZDBiZjcyNTQyZWUyMTc2NjNj
export NXF_VER=23.10.0

sudo nextflow run git/nf-core-microc -profile docker \
--input ss_toy_data.csv \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--index /mnt/datawk1/references/UCSC_GRCm38_bwa \
--juicertool_location /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/git/nf-core-microc/bin/juicertools.jar \
--skip_juicer false --fillRE true \
--outdir ./nfout --exclude_lcextrap false -resume






## 2.2. test multimapped L1

## a. multimapping allignment bam

In [ ]:
# on fisso

PIPELINE_PATH="git/nf-core-microc"
#DOCKER_ENV="git/Lara_MLL2/bin/docker_env.sh"

in="sheets/fisso_NFSS_Lara_all_microC.csv"
out="sheets/fisso_NFSS_WT_day4_A_LP123.csv"

awk -F',' 'NR==1 || $1 ~ /^WT_day4_A_LP[123]$/ {print}' "$in" \
| tr -d '\r' > "$out"

export NXF_VER=23.10.0

sudo nextflow run "${PIPELINE_PATH}" -profile docker \
  -work-dir "${NF_WORK}" \
  --input "sheets/fisso_NFSS_WT_day4_A_LP123.csv" \
  --fasta "/home/lucio/wkdir/projects/MLL2_L1_regulation/in/customReferences/chr1_Akt3/genome.fa" \
  --index "/home/lucio/wkdir/projects/MLL2_L1_regulation/in/customReferences/chr1_Akt3" \
  --outdir "${NF_OUT}" \
  --exclude_lcextrap false





## 3. analyze Lara_microC_1

### a. make environment

In [ ]:
%%script bash

mkdir /mnt/datawk1/analysis/Lara/Lara_microC_1

mkdir /mnt/datawk1/analysis/Lara/Lara_microC_1/git
mkdir /mnt/datawk1/analysis/Lara/Lara_microC_1/nfout

cd /mnt/datawk1/analysis/Lara/Lara_microC_1/git

git clone --branch reduce_space_consumption git@github.com:lucidif/nf-core-microc.git

git branch -vv

#* reduce_space_consumption e416c51 [origin/reduce_space_consumption] first working script

cd /mnt/datawk1/analysis/Lara/Lara_microC_1



### b. start analysis


In [ ]:
%%script bash

sudo su

export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MTgzfS4yYzMxYzc2YzkzMjg5MGUxNzczZTI0ZDBiZjcyNTQyZWUyMTc2NjNj
export NXF_VER=23.10.0

nextflow run git/nf-core-microc \
  -profile docker --input SS_Lara_microC_1.csv \
  --fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
  --index /mnt/datawk1/references/UCSC_GRCm38_bwa \
  --outdir ./nfout --exclude_lcextrap false \
  -resume -with-tower


## 3. analyze 2024_03_Lara_microC on ziggy

### a. get env ✅

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/git

git clone git@github.com:lucidif/nf-core-microc.git

### b. start analysis ✅

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/2024_03_Lara_microC

screen -R 2024_03_Lara_microC

sudo su

export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MTgzfS4yYzMxYzc2YzkzMjg5MGUxNzczZTI0ZDBiZjcyNTQyZWUyMTc2NjNj
export NXF_VER=23.10.0

nextflow run git/nf-core-microc \
  -profile docker --input SS_microC_NFSS_2024_03_microC_Lara.csv \
  --fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
  --index /mnt/datawk1/references/UCSC_GRCm38_bwa \
  --outdir ./nfout --exclude_lcextrap false \
  -resume -with-tower

  nextflow run git/nf-core-microc \
  -profile docker --input SS_onesample_microC_NFSS_2024_03_microC_Lara.csv \
  --fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
  --index /mnt/datawk1/references/UCSC_GRCm38_bwa \
  --outdir ./nfout/test_on_sample --exclude_lcextrap false \
  -resume -with-tower


merging library prep

In [ ]:
%%script bash

sudo su

export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MTgzfS4yYzMxYzc2YzkzMjg5MGUxNzczZTI0ZDBiZjcyNTQyZWUyMTc2NjNj
export NXF_VER=23.10.0

nextflow run git/nf-core-microc -profile docker \
--input SS_mergeLP_NFSS_2024_03_microC_Lara.csv \
--fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
--index /mnt/datawk1/references/UCSC_GRCm38_bwa \
--juicertool_location /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/git/nf-core-microc/bin/juicertools.jar \
--skip_juicer false \
--outdir ./nfout/mergeLP --exclude_lcextrap false -resume -with-tower

## 4. start analysis 2024_05_Lara_microC


### get environment

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/2024_05_Lara_microC

cd ./git/nf-core-microc
git pull






### start analysis

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/2024_05_Lara_microC

screen -R 2024_05_Lara_microC

sudo su

export TOWER_ACCESS_TOKEN=eyJ0aWQiOiA5MTgzfS4yYzMxYzc2YzkzMjg5MGUxNzczZTI0ZDBiZjcyNTQyZWUyMTc2NjNj
export NXF_VER=23.10.0

# nextflow run git/nf-core-microc \
#   -profile docker --input b1_NFSS_2024_05_microC_Lara.csv \
#   --fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
#   --index /mnt/datawk1/references/UCSC_GRCm38_bwa \
#   --outdir ./nfout --exclude_lcextrap false \
#   -resume -with-tower

nextflow run git/nf-core-microc \
  -profile docker --input KO_D0_A.csv \
  --fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
  --index /mnt/datawk1/references/UCSC_GRCm38_bwa \
  --outdir ./nfout --exclude_lcextrap false \
  -resume -with-tower

nextflow run git/nf-core-microc \
  -profile docker --input KO_D0_B.csv \
  --fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
  --index /mnt/datawk1/references/UCSC_GRCm38_bwa \
  --outdir ./nfout/KO_D0_B --exclude_lcextrap false \
  -resume -with-tower

nextflow run git/nf-core-microc \
  -profile docker --input KO_D4_A.csv \
  --fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
  --index /mnt/datawk1/references/UCSC_GRCm38_bwa \
  --outdir ./nfout/KO_D4_A --exclude_lcextrap false \
  -resume -with-tower

nextflow run git/nf-core-microc \
  -profile docker --input WT_D4_B.csv \
  --fasta /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa \
  --index /mnt/datawk1/references/UCSC_GRCm38_bwa \
  --outdir ./nfout/WT_D4_B --exclude_lcextrap false \
  -resume -with-tower


## 5. merge pairsam and test resolution




### create environment

In [ ]:
%%script bash

mkdir /mnt/datawk1/analysis/Lara/2024_10_Lara_microC_downstream

cd /mnt/datawk1/analysis/Lara/2024_10_Lara_microC_downstream

mkdir git
mkdir in
mkdir out
mkdir ./out/pairtools_merge

cd ./git

git clone git@github.com:lucidif/nf-core-microc.git

cd ..

rsync -av /media/lucio/Backup1/Lucio/Analysis/Lara/Lara_microC_1/nfout/pairtools/dedup/ ./in

rsync -av /media/lucio/Backup1/Lucio/Analysis/Lara/2024_05_Lara_microC/nfout/WT_D0_B/pairtools/dedup/ ./in

rsync -av /media/lucio/Backup1/Lucio/Analysis/Lara/2024_05_Lara_microC/nfout/KO_day0_A/pairtools/dedup/ ./in

rsync -av /media/lucio/Backup1/Lucio/Analysis/Lara/2024_05_Lara_microC/nfout/KO_D0_B/pairtools/dedup/ ./in

rsync -av /media/lucio/Backup1/Lucio/Analysis/Lara/2024_05_Lara_microC/nfout/KO_D4_A/pairtools/dedup/ ./in








### merge samples

In [ ]:
%%script bash


source "git/Lara_MLL2/bin/docker_env.sh"
sudo docker run --rm "${DOCKER_ARGS[@]}" -it lucidif/microc:0.0.1

cd outs/2024_10_Lara_microC_downstream

pairtools merge -o ./out/pairtools_merge/aLp_WT_day0_B.Dd.pairs.gz ./in/WT_day0_B_LP1.Dd.pairs.gz ./in/WT_day0_B_LP2.Dd.pairs.gz ./in/WT_day0_B_LP3.Dd.pairs.gz

pairtools merge -o ./out/pairtools_merge/aLp_WT_day0.Dd.pairs.gz ./in/WT_day0_A.Dd.pairs.gz ./out/pairtools_merge/aLp_WT_day0_B.Dd.pairs.gz

pairtools merge -o ./out/pairtools_merge/aLp_KO_day0.Dd.pairs.gz ./in/KO_day0_A_LP1.Dd.pairs.gz ./in/KO_day0_A_LP2.Dd.pairs.gz ./in/KO_day0_A_LP3.Dd.pairs.gz ./in/KO_day0_B_LP1.Dd.pairs.gz ./in/KO_day0_B_LP2.Dd.pairs.gz ./in/KO_day0_B_LP3.Dd.pairs.gz

pairtools merge -o ./out/pairtools_merge/aLp_KO_day0.Dd.pairs.gz ./in/KO_day0_A_LP1.Dd.pairs.gz ./in/KO_day0_A_LP2.Dd.pairs.gz ./in/KO_day0_A_LP3.Dd.pairs.gz 

pairtools merge -o ./out/pairtools_merge/aLp_KO_day0_B.Dd.pairs.gz ./in/KO_day0_B_LP1.Dd.pairs.gz ./in/KO_day0_B_LP2.Dd.pairs.gz ./in/KO_day0_B_LP3.Dd.pairs.gz

pairtools merge -o ./out/pairtools_merge/aLp_KO_day0_A.Dd.pairs.gz ./in/KO_day0_A_LP1.Dd.pairs.gz ./in/KO_day0_A_LP2.Dd.pairs.gz ./in/KO_day0_A_LP3.Dd.pairs.gz

exit

##day 4

screen -R mergeMicroC


sudo docker run -it -v /media/lucio/Backup1:/media/lucio/Backup1 -v /mnt/datawk1:/mnt/datawk1 -w `pwd` -u $(id -u):$(id -g) lucidif/microc:0.0.1

cd /mnt/datawk1/analysis/Lara/2024_10_Lara_microC_downstream

pairsam_path="/media/lucio/Backup1/Lucio/Analysis/Lara/2024_05_Lara_microC/nfout/"

subfolder="/pairtools/dedup/"

samplesA="KO_D4_A"
samplesB="KO_D4_B"
nameSamA="KO_day4_A"
nameSamB="KO_day4_B"

#merge tecnical replicate

echo "pairtools merge -o ./out/pairtools_merge/aLp_KO_day4_A.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP1.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP2.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP3.Dd.pairs.gz"

pairtools merge -o ./out/pairtools_merge/aLp_KO_day4_A.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP1.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP2.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP3.Dd.pairs.gz

echo "pairtools merge -o ./out/pairtools_merge/aLp_KO_day4_B.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP1.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP2.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP3.Dd.pairs.gz"

pairtools merge -o ./out/pairtools_merge/aLp_KO_day4_B.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP1.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP2.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP3.Dd.pairs.gz

pairtools merge -o ./out/pairtools_merge/aLp_KO_day4.Dd.pairs.gz ./out/pairtools_merge/aLp_KO_day4_A.Dd.pairs.gz ./out/pairtools_merge/aLp_KO_day4_B.Dd.pairs.gz


exit

screen -R merge_microC_WT

sudo docker run -it -v /media/lucio/Backup1:/media/lucio/Backup1 -v /mnt/datawk1:/mnt/datawk1 -w `pwd` -u $(id -u):$(id -g) lucidif/microc:0.0.1

cd /mnt/datawk1/analysis/Lara/2024_10_Lara_microC_downstream

samples="WT_D4_A  WT_D4_B"

pairsam_path="/media/lucio/Backup1/Lucio/Analysis/Lara/2024_05_Lara_microC/nfout/"

subfolder="/pairtools/dedup/"

samplesA="WT_D4_A"
samplesB="WT_D4_B"
nameSamA="WT_day4_A"
nameSamB="WT_day4_B"

echo "pairtools merge -o ./out/pairtools_merge/aLp_WT_day4_A.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP1.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP2.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP3.Dd.pairs.gz"

pairtools merge -o ./out/pairtools_merge/aLp_WT_day4_A.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP1.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP2.Dd.pairs.gz ${pairsam_path}${samplesA}${subfolder}${nameSamA}_LP3.Dd.pairs.gz

echo "pairtools merge -o ./out/pairtools_merge/aLp_WT_day4_B.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP1.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP2.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP3.Dd.pairs.gz"

pairtools merge -o ./out/pairtools_merge/aLp_WT_day4_B.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP1.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP2.Dd.pairs.gz ${pairsam_path}${samplesB}${subfolder}${nameSamB}_LP3.Dd.pairs.gz

pairtools merge -o ./out/pairtools_merge/aLp_WT_day4.Dd.pairs.gz ./out/pairtools_merge/aLp_WT_day4_A.Dd.pairs.gz ./out/pairtools_merge/aLp_WT_day4_B.Dd.pairs.gz


### make .hic contact matrix

In [ ]:
%%script bash

# make matrix by replicate

#prj_folder="/home/lucio/wkdir/projects/MLL2_L1_regulation" 

mkdir ./outs/2024_10_Lara_microC_downstream/out/pairtools_merge/hic_by_replicate

ls ${prj_folder}/outs/2024_10_Lara_microC_downstream/in/mm10.sizes

source "git/Lara_MLL2/bin/docker_env.sh"
sudo docker run --rm "${DOCKER_ARGS[@]}" -it lucidif/microc:0.0.1

main_prj=`pwd`
cd ./outs/2024_10_Lara_microC_downstream/out/pairtools_merge/hic_by_replicate

java -Xms512m -Xmx32g -jar ${main_prj}/git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ../../../in/WT_day0_A.Dd.pairs.gz aLp_WT_day0_A.Dd.hic ${main_prj}/outs/2024_10_Lara_microC_downstream/in/mm10.sizes
java -Xms512m -Xmx32g -jar ${main_prj}/git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ../aLp_WT_day0_B.Dd.pairs.gz aLp_WT_day0_B.Dd.hic ${main_prj}/outs/2024_10_Lara_microC_downstream/in/mm10.sizes
java -Xms512m -Xmx32g -jar ${main_prj}/git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ../aLp_KO_day0_A.Dd.pairs.gz aLp_KO_day0_A.Dd.hic ${main_prj}/outs/2024_10_Lara_microC_downstream/in/mm10.sizes
java -Xms512m -Xmx32g -jar ${main_prj}/git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ../aLp_KO_day0_B.Dd.pairs.gz aLp_KO_day0_B.Dd.hic ${main_prj}/outs/2024_10_Lara_microC_downstream/in/mm10.sizes
java -Xms512m -Xmx32g -jar ${main_prj}/git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ../aLp_WT_day4_A.Dd.pairs.gz aLp_WT_day4_A.Dd.hic ${main_prj}/outs/2024_10_Lara_microC_downstream/in/mm10.sizes
java -Xms512m -Xmx32g -jar ${main_prj}/git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ../aLp_WT_day4_B.Dd.pairs.gz aLp_WT_day4_B.Dd.hic ${main_prj}/outs/2024_10_Lara_microC_downstream/in/mm10.sizes
java -Xms512m -Xmx32g -jar ${main_prj}/git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ../aLp_KO_day4_A.Dd.pairs.gz aLp_KO_day4_A.Dd.hic ${main_prj}/outs/2024_10_Lara_microC_downstream/in/mm10.sizes
java -Xms512m -Xmx32g -jar ${main_prj}/git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ../aLp_KO_day4_B.Dd.pairs.gz aLp_KO_day4_B.Dd.hic ${main_prj}/outs/2024_10_Lara_microC_downstream/in/mm10.sizes

java -Xms512m -Xmx32g -jar ${main_prj}/git/nf-core-microc/bin/juicertools.jar validate aLp_WT_day4_A.Dd.hic


#end TODO

# make matrix of pulled samples

cd /mnt/datawk1/analysis/Lara/2024_05_Lara_microC/tmp/

sudo docker run -itv /mnt/datawk1:/mnt/datawk1 -w `pwd` -u $(id -u):$(id -g) lucidif/microc:0.0.1

cd /mnt/datawk1/analysis/Lara/2024_05_Lara_microC

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ./out/pairtools_merge/aLp_WT_day0.Dd.pairs.gz ./out/pairtools_merge/aLp_WT_day0.Dd.hic ./in/mm10.sizes

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ./out/pairtools_merge/aLp_KO_day0.Dd.pairs.gz ./out/pairtools_merge/aLp_KO_day0.Dd.hic ./in/mm10.sizes

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 10000 --threads 12 ./out/pairtools_merge/aLp_WT_day0.Dd.pairs.gz ./out/pairtools_merge/10kb_aLp_WT_day0.Dd.hic ./in/mm10.sizes

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 10000 --threads 12 ./out/pairtools_merge/aLp_KO_day0.Dd.pairs.gz ./out/pairtools_merge/10kb_aLp_KO_day0.Dd.hic ./in/mm10.sizes

#test single replicate KO

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ./in/KO_day0_A_LP1.Dd.pairs.gz ./out/juicertools_pre/KO_day0_A_LP1.Dd.hic ./in/bck.mm10.sizes

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ./in/KO_day0_A_LP2.Dd.pairs.gz ./out/juicertools_pre/KO_day0_A_LP2.Dd.hic ./in/bck.mm10.sizes

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ./in/KO_day0_A_LP3.Dd.pairs.gz ./out/juicertools_pre/KO_day0_A_LP3.Dd.hic ./in/bck.mm10.sizes

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 5000 --threads 12 ./in/KO_day0_B_LP1.Dd.pairs.gz ./out/juicertools_pre/KO_day0_B_LP1.Dd.hic ./in/bck.mm10.sizes

# day 4

sudo docker run -itv /mnt/datawk1:/mnt/datawk1 -w `pwd` -u $(id -u):$(id -g) lucidif/microc:0.0.1

cd /mnt/datawk1/analysis/Lara/2024_10_Lara_microC_downstream

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 15000,5000 --threads 12 ./out/pairtools_merge/aLp_WT_day4.Dd.pairs.gz ./out/pairtools_merge/15k_5k_aLp_WT_day4.Dd.hic ./in/mm10.sizes

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 15000,5000 --threads 12 ./out/pairtools_merge/aLp_KO_day4.Dd.pairs.gz ./out/pairtools_merge/15k_5k_aLp_KO_day4.Dd.hic ./in/mm10.sizes



### make cooler matrix

#### environments

In [ ]:
%%script bash

cd  /mnt/datawk1/analysis/Lara/2024_10_Lara_microC_downstream/

cp /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/in/ucsc/mm10.sizes ./in

#/media/lucio/Backup1/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/in/KO_day0_A_LP1.Dd.pairs.gz

cp /media/lucio/Backup1/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/in/KO_day0_A_LP1.Dd.pairs.gz ./out/pairtools_merge/

cp /media/lucio/Backup1/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/pairtools_merge/aLp_WT_day0.Dd.pairs.gz ./out/pairtools_merge/

#mv ./Lara_multiomic_analysis/in/2024_10_Lara_microC_downstream/KO_day0_A_LP1.Dd.pairs.gz ./2024_10_Lara_microC_downstream/in/



#### generate matrix pulled samples

In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it quay.io/biocontainers/pairix:0.3.7--py36h30a8e3e_3

cd /mnt/datawk1/analysis/Lara/2024_10_Lara_microC_downstream/out

#D0

# pairix \
#      \
#     ./pairtools_merge/KO_day0_A_LP1.Dd.pairs.gz

pairix \
    \
    ./pairtools_merge/aLp_KO_day0.Dd.pairs.gz

pairix \
    \
    ../in/WT_day0_A.Dd.pairs.gz

pairix \
    \
    ./aLp_WT_day0_B.Dd.pairs.gz

pairix \
     \
    ./pairtools_merge/aLp_KO_day4.Dd.pairs.gz

pairix \
     \
    ./pairtools_merge/aLp_WT_day4.Dd.pairs.gz


exit


sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0

cd /mnt/datawk1/analysis/Lara/2024_10_Lara_microC_downstream/

#chromsizes="./in/mm10.sizes"
#cool_bin=5000
#pairs="./in/KO_day0_A_LP1.Dd.pairs.gz"
#prefix="5kb_KO_day0_A_LP1"
#nproc=8

# echo "cooler cload pairs $nproc ${chromsizes}:${cool_bin} $pairs ./out/${prefix}.cool"
# cooler cload pairs $nproc ${chromsizes}:${cool_bin} $pairs ./out/${prefix}.cool

# cooler cload \
#     pairix \
#     --nproc 8 \
#     ./in/mm10.sizes:5000 \
#     ./in/KO_day0_A_LP1.Dd.pairs.gz \
#     ./out/5kb_KO_day4_A_LP1.cool

mkdir ./out/cooler_cload_balance

cooler cload \
    pairix \
    --nproc 12 \
    ./in/mm10.sizes:5000 \
    ./out/pairtools_merge/aLp_KO_day0.Dd.pairs.gz \
    ./out/cooler_cload_balance/5kb_aLp_KO_day0.Dd.cool

cooler cload \
    pairix \
    --nproc 12 \
    ./in/mm10.sizes:5000 \
    ./out/pairtools_merge/aLp_WT_day0.Dd.pairs.gz \
    ./out/cooler_cload_balance/5kb_aLp_WT_day0.Dd.cool

cooler cload \
    pairix \
    --nproc 12 \
    ./in/mm10.sizes:15000 \
    ./out/pairtools_merge/aLp_WT_day0.Dd.pairs.gz \
    ./out/cooler_cload_balance/15kb_aLp_WT_day0.Dd.cool

cooler cload \
    pairix \
    --nproc 12 \
    ./in/mm10.sizes:15000 \
    ./out/pairtools_merge/aLp_KO_day0.Dd.pairs.gz \
    ./out/cooler_cload_balance/15kb_aLp_KO_day0.Dd.cool

exit

cp ./out/cooler_cload_balance/5kb_aLp_KO_day0.Dd.cool ./out/cooler_cload_balance/balanced_5kb_aLp_KO_day0.Dd.cool
rm ./out/cooler_cload_balance/5kb_aLp_KO_day0.Dd.cool

sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/5kb_aLp_KO_day0.Dd.cool

cooler dump --balanced -c chrom1,start1,end1,chrom2,start2,end2,count,balanced ./out/cooler_cload_balance/balanced_5kb_aLp_KO_day0.Dd.cool | head

cp ./out/cooler_cload_balance/5kb_aLp_WT_day0.Dd.cool ./out/cooler_cload_balance/balanced_5kb_aLp_WT_day0.Dd.cool


rm ./out/cooler_cload_balance/5kb_aLp_WT_day0.Dd.cool
sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/balanced_5kb_aLp_WT_day0.Dd.cool

cooler dump --balanced -c chrom1,start1,end1,chrom2,start2,end2,count,balanced ./out/cooler_cload_balance/balanced_5kb_aLp_WT_day0.Dd.cool | head

#cooler balance ./out/cooler_cload_balance/15kb_aLp_KO_day0.Dd.cool

#cooler balance ./out/cooler_cload_balance/15kb_aLp_WT_day0.Dd.cool

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it quay.io/biocontainers/pairix:0.3.7--py36h30a8e3e_3
#D4

cooler cload \
    pairix \
    --nproc 12 \
    ./in/mm10.sizes:5000 \
    ./out/pairtools_merge/aLp_WT_day4.Dd.pairs.gz \
    ./out/cooler_cload_balance/5kb_aLp_WT_day4.Dd.cool

cooler cload \
    pairix \
    --nproc 12 \
    ./in/mm10.sizes:15000 \
    ./out/pairtools_merge/aLp_WT_day4.Dd.pairs.gz \
    ./out/cooler_cload_balance/15kb_aLp_WT_day4.Dd.cool

cooler cload \
    pairix \
    --nproc 12 \
    ./in/mm10.sizes:5000 \
    ./out/pairtools_merge/aLp_KO_day4.Dd.pairs.gz \
    ./out/cooler_cload_balance/5kb_aLp_KO_day4.Dd.cool

cooler cload \
    pairix \
    --nproc 12 \
    ./in/mm10.sizes:15000 \
    ./out/pairtools_merge/aLp_KO_day4.Dd.pairs.gz \
    ./out/cooler_cload_balance/15kb_aLp_KO_day4.Dd.cool


exit

#cooler balance ./out/cooler_cload_balance/5kb_aLp_WT_day4.Dd.cool
cp ./out/cooler_cload_balance/5kb_aLp_WT_day4.Dd.cool ./out/cooler_cload_balance/balanced_5kb_aLp_WT_day4.Dd.cool
sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/5kb_aLp_WT_day4.Dd.cool
cooler dump --balanced -c chrom1,start1,end1,chrom2,start2,end2,count,balanced ./out/cooler_cload_balance/5kb_aLp_WT_day4.Dd.cool | head

#cooler balance ./out/cooler_cload_balance/15kb_aLp_WT_day4.Dd.cool


cp ./out/cooler_cload_balance/5kb_aLp_KO_day4.Dd.cool ./out/cooler_cload_balance/balanced_5kb_aLp_KO_day4.Dd.cool
rm ./out/cooler_cload_balance/5kb_aLp_KO_day4.Dd.cool
sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/balanced_5kb_aLp_KO_day4.Dd.cool
cooler dump --balanced -c chrom1,start1,end1,chrom2,start2,end2,count,balanced ./out/cooler_cload_balance/balanced_5kb_aLp_WT_day4.Dd.cool | head

#cooler balance ./out/cooler_cload_balance/15kb_aLp_KO_day4.Dd.cool


#generate mcool of 5kb

samples="aLp_WT_day0.Dd aLp_KO_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"

for i in ${samples} ; do cooler zoomify ./out/cooler_cload_balance/balanced_5kb_${i}.cool ; done


#generete hic for genome browser


sudo docker run -it -v `pwd`:`pwd` -w `pwd` lucidif/microc:0.0.1

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 15000 --threads 12 ./out/pairtools_merge/aLp_WT_day0.Dd.pairs.gz ./out/pairtools_merge/norm_15k_aLp_WT_day0.Dd.hic ./in/mm10.sizes

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 15000 --threads 12 ./out/pairtools_merge/aLp_KO_day0.Dd.pairs.gz ./out/pairtools_merge/norm_15k_aLp_WT_day0.Dd.hic ./in/mm10.sizes

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 15000 --threads 12 ./out/pairtools_merge/aLp_WT_day4.Dd.pairs.gz ./out/pairtools_merge/norm_15k_aLp_WT_day0.Dd.hic ./in/mm10.sizes

java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 15000 --threads 12 ./out/pairtools_merge/aLp_KO_day4.Dd.pairs.gz ./out/pairtools_merge/norm_15k_aLp_WT_day0.Dd.hic ./in/mm10.sizes

#### generate matrix by replicate

In [ ]:
%%script bash

screen -R rebuild_cooler

source "git/Lara_MLL2/bin/initialize.sh"
sudo docker run --rm "${DOCKER_ARGS[@]}" -it quay.io/biocontainers/pairix:0.3.7--py36h30a8e3e_3

cd outs/2024_10_Lara_microC_downstream/out

#D0

# pairix \
#      \
#     ./pairtools_merge/KO_day0_A_LP1.Dd.pairs.gz

pairix \
     \
    ./pairtools_merge/aLp_KO_day0.Dd.pairs.gz

pairix \
     \
    ./pairtools_merge/aLp_WT_day0.Dd.pairs.gz


# D4

pairix \
     \
    ./pairtools_merge/aLp_KO_day4_A.Dd.pairs.gz

pairix \
     \
    ./pairtools_merge/aLp_KO_day4_B.Dd.pairs.gz    

pairix \
     \
    ./pairtools_merge/aLp_WT_day4_A.Dd.pairs.gz

pairix \
     \
    ./pairtools_merge/aLp_WT_day4_B.Dd.pairs.gz

exit




## fan c compartments

### make contact heatmap in cool format

In [ ]:
samples="aLp_KO_day0.Dd aLp_WT_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler cload pairix ./in/mm10.sizes:1000000 ./out/pairtools_merge/${i}.pairs.gz ./out/pairtools_merge/1M_${i}.cool ; done

for i in ${samples} ; do cp ./out/pairtools_merge/1M_${i}.cool ./out/cooler_cload_balance/balanced_1M_${i}.cool ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/balanced_1M_${i}.cool ; done
    

#350kb

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler cload pairix ./in/mm10.sizes:350000 ./out/pairtools_merge/${i}.pairs.gz ./out/pairtools_merge/350kb_${i}.cool ; done

for i in ${samples} ; do cp ./out/pairtools_merge/350kb_${i}.cool ./out/cooler_cload_balance/balanced_350kb_${i}.cool ; done
    
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/balanced_350kb_${i}.cool ; done  

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler zoomify ./out/cooler_cload_balance/balanced_350kb_${i}.cool ; done
    
# samples="aLp_WT_day4.Dd aLp_KO_day4.Dd"

# for i in ${samples} ; do cp ./out/pairtools_merge/350kb_${i}.cool ./out/cooler_cload_balance/TT_balanced_350kb_${i}.cool ; done
# for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/TT_balanced_350kb_${i}.cool ; done 


#250kb 
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler cload pairix ./in/mm10.sizes:250000 ./out/pairtools_merge/${i}.pairs.gz ./out/pairtools_merge/250kb_${i}.cool ; done

for i in ${samples} ; do cp ./out/pairtools_merge/250kb_${i}.cool ./out/cooler_cload_balance/balanced_250kb_${i}.cool ; done
    
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/balanced_250kb_${i}.cool ; done  

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler zoomify ./out/cooler_cload_balance/balanced_250kb_${i}.cool ; done




one="aLp_KO_day0.Dd"
cp ./out/pairtools_merge/250kb_${one}.cool ./out/cooler_cload_balance/balanced_250kb_${one}.cool
sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/balanced_250kb_${one}.cool
sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler zoomify ./out/cooler_cload_balance/balanced_250kb_${one}.cool



### extract compartments genome wide coparmetization TODEL we used by chromosomes approach

In [ ]:
#pwd /mnt/datawk1/analysis/Lara/2024_10_Lara_microC_downstream/




#=========================================

# wide normalization

#PC1
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest /fanc_env/bin/fanc compartments ./out/cooler_cload_balance/balanced_1M_${i}.cool  out/fanc_compartments/wide_1M_${i}.ab -v ./out/fanc_compartments/1M_${i}.bed -g in/mm10.fa -w ; done

for i in ${samples} ; do awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/1M_${i}.bed > ./out/fanc_compartments/1M_${i}.bedgraph ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/1M_${i}.bedgraph ./in/mm10.sizes ./out/fanc_compartments/1M_${i}.bw ; done 

#PC2
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest /fanc_env/bin/fanc compartments ./out/cooler_cload_balance/balanced_1M_${i}.cool  out/fanc_compartments/PC2_1M_${i}.ab -i 2 -v ./out/fanc_compartments/PC2_1M_${i}.bed -g in/mm10.fa -w ; done

for i in ${samples} ; do awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/PC2_1M_${i}.bed > ./out/fanc_compartments/PC2_1M_${i}.bedgraph ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/PC2_1M_${i}.bedgraph ./in/mm10.sizes ./out/fanc_compartments/PC2_1M_${i}.bw ; done 

#PC3
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest /fanc_env/bin/fanc compartments ./out/cooler_cload_balance/balanced_1M_${i}.cool  out/fanc_compartments/PC3_wide_1M_${i}.ab -i 3 -v ./out/fanc_compartments/PC3_1M_${i}.bed -g in/mm10.fa -w ; done

for i in ${samples} ; do awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/PC3_1M_${i}.bed > ./out/fanc_compartments/PC3_1M_${i}.bedgraph ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/PC3_1M_${i}.bedgraph ./in/mm10.sizes ./out/fanc_compartments/PC3_1M_${i}.bw ; done 

#========================================



### 1M extract compartments genome by chromosome 

In [ ]:
samples="aLp_KO_day0.Dd aLp_WT_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"

# for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler cload pairix ./in/mm10.sizes:1000000 ./out/pairtools_merge/${i}.pairs.gz ./out/pairtools_merge/1M_${i}.cool ; done

# for i in ${samples} ; do cp ./out/pairtools_merge/1M_${i}.cool ./out/cooler_cload_balance/balanced_1M_${i}.cool ; done

# for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/balanced_1M_${i}.cool ; done

#=========================================

# wide normalization

#PC1
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest /fanc_env/bin/fanc compartments ./out/cooler_cload_balance/balanced_1M_${i}.cool  out/fanc_compartments/bychr_1M_${i}.ab -v ./out/fanc_compartments/bychr_1M_${i}.bed -g in/mm10.fa --compartment-strength ./out/fanc_compartments/t_bychr_1M_${i}.txt ; done

for i in ${samples} ; do awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/bychr_1M_${i}.bed > ./out/fanc_compartments/bychr_1M_${i}.bedgraph ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/bychr_1M_${i}.bedgraph ./in/mm10.sizes ./out/fanc_compartments/bychr_1M_${i}.bw ; done 




#PC2
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest /fanc_env/bin/fanc compartments ./out/cooler_cload_balance/balanced_1M_${i}.cool  out/fanc_compartments/bychr_PC2_1M_${i}.ab -i 2 -v ./out/fanc_compartments/bychr_PC2_1M_${i}.bed -g in/mm10.fa ; done

for i in ${samples} ; do awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/bychr_PC2_1M_${i}.bed > ./out/fanc_compartments/bychr_PC2_1M_${i}.bedgraph ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/bychr_PC2_1M_${i}.bedgraph ./in/mm10.sizes ./out/fanc_compartments/bychr_PC2_1M_${i}.bw ; done 

# #PC3
# for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest /fanc_env/bin/fanc compartments ./out/cooler_cload_balance/balanced_1M_${i}.cool  out/fanc_compartments/PC3_wide_1M_${i}.ab -i 3 -v ./out/fanc_compartments/PC3_1M_${i}.bed -g in/mm10.fa -w ; done

# for i in ${samples} ; do awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/PC3_1M_${i}.bed > ./out/fanc_compartments/PC3_1M_${i}.bedgraph ; done

# for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/PC3_1M_${i}.bedgraph ./in/mm10.sizes ./out/fanc_compartments/PC3_1M_${i}.bw ; done 

#========================================

sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g)  quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 bigwigCompare --help
sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g)  quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 bigwigCompare -b1 bychr_PC2_1M_aLp_KO_day0.Dd.bw -b2 --region chr10



### 250kb extract compartments



In [ ]:
#samples="aLp_KO_day0.Dd aLp_WT_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"
#samples="aLp_KO_day0.Dd"

samples="aLp_WT_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"

#PC1
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest /fanc_env/bin/fanc compartments ./out/cooler_cload_balance/balanced_250kb_${i}.cool  out/fanc_compartments/bychr_250kb_${i}.ab -v ./out/fanc_compartments/bychr_250kb_${i}.bed -g in/mm10.fa --compartment-strength ./out/fanc_compartments/t_bychr_250kb_${i}.txt ; done

for i in ${samples} ; do awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/bychr_250kb_${i}.bed > ./out/fanc_compartments/bychr_250kb_${i}.bedgraph ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/bychr_250kb_${i}.bedgraph ./in/mm10.sizes ./out/fanc_compartments/bychr_250kb_${i}.bw ; done 




#PC2
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest /fanc_env/bin/fanc compartments ./out/cooler_cload_balance/balanced_250kb_${i}.cool  out/fanc_compartments/bychr_PC2_250kb_${i}.ab -i 2 -v ./out/fanc_compartments/bychr_PC2_250kb_${i}.bed -g in/mm10.fa ; done

for i in ${samples} ; do awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/bychr_PC2_250kb_${i}.bed > ./out/fanc_compartments/bychr_PC2_250kb_${i}.bedgraph ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/bychr_PC2_250kb_${i}.bedgraph ./in/mm10.sizes ./out/fanc_compartments/bychr_PC2_250kb_${i}.bw ; done 



### 350kbn extract compartments

In [ ]:
#samples="aLp_KO_day0.Dd aLp_WT_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"
#samples="aLp_KO_day0.Dd"

samples="aLp_WT_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"

#PC1
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest /fanc_env/bin/fanc compartments ./out/cooler_cload_balance/balanced_350kb_${i}.cool  out/fanc_compartments/bychr_350kb_${i}.ab -v ./out/fanc_compartments/bychr_350kb_${i}.bed -g in/mm10.fa --compartment-strength ./out/fanc_compartments/t_bychr_350kb_${i}.txt ; done

for i in ${samples} ; do awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/bychr_350kb_${i}.bed > ./out/fanc_compartments/bychr_350kb_${i}.bedgraph ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/bychr_350kb_${i}.bedgraph ./in/mm10.sizes ./out/fanc_compartments/bychr_350kb_${i}.bw ; done 



#PC2
for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest /fanc_env/bin/fanc compartments ./out/cooler_cload_balance/balanced_350kb_${i}.cool  out/fanc_compartments/bychr_PC2_350kb_${i}.ab -i 2 -v ./out/fanc_compartments/bychr_PC2_350kb_${i}.bed -g in/mm10.fa ; done

for i in ${samples} ; do awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/bychr_PC2_350kb_${i}.bed > ./out/fanc_compartments/bychr_PC2_350kb_${i}.bedgraph ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/bychr_PC2_350kb_${i}.bedgraph ./in/mm10.sizes ./out/fanc_compartments/bychr_PC2_350kb_${i}.bw ; done 



### compartment statistics

In [ ]:
cp /mnt/datawk1/references/annotations/UCSC_mm10.refGene/mm10.refGene.gtf in/

Rscript ./git/Lara_MLL2/compartments_stats.R

awk '{print $1"\t"$2"\t"$3"\t"$5}' ./out/fanc_compartments/dchange_wt_d4.bed > ./out/fanc_compartments/dchange_wt_d4.bedgraph

sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/ucsc-bedgraphtobigwig:377--h446ed27_1 bedGraphToBigWig ./out/fanc_compartments/dchange_wt_d4.bedgraph ./in/mm10.sizes ./out/fanc_compartments/dchange_wt_d4.bw



### fanc plot

In [ ]:
fancplot chr10:8mb-12mb --tick-locations 8mb 10mb -p triangular output/hic/binned/fanc_example_50kb.hic

### upload in higlass

In [ ]:


sudo docker run --detach \
           --publish 8989:80 \
           --volume /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream:/data \
           --volume /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/tmp:/tmp \
           --name higlass-container \
         higlass/higlass-docker:v0.6.1

sudo docker start higlass-container

sudo docker exec -it higlass-container python higlass-server/manage.py migrate

sudo docker exec -it higlass-container higlass-server/manage.py createsuperuser

# reference

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/in/mm10.sizes \
  --filetype chromsizes-tsv \
  --datatype chromsizes \
  --coordSystem Dd

sudo docker exec higlass-container python higlass-server/manage.py ingest_tileset \
    --filename /data/in/mm10.sizes \
    --filetype chromsizes-tsv \
    --datatype chromsizes \
    --name Lara_mm10 \
    --coordSystem Dd



# matrix

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/pairtools_merge/1M_aLp_WT_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/pairtools_merge/1M_aLp_KO_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/pairtools_merge/1M_aLp_WT_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/pairtools_merge/1M_aLp_KO_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix 


#350kb

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_350kb_aLp_WT_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_350kb_aLp_KO_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_350kb_aLp_WT_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_350kb_aLp_KO_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

#250kb

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_250kb_aLp_WT_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_250kb_aLp_KO_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_250kb_aLp_KO_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_250kb_aLp_WT_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix


# bigwig


#D0

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/1M_aLp_WT_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC1ev_WT_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/1M_aLp_KO_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC1ev_KO_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/PC2_1M_aLp_WT_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC2ev_WT_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/PC2_1M_aLp_KO_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC2ev_KO_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/PC3_1M_aLp_WT_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC3ev_WT_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/PC3_1M_aLp_KO_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC3ev_KO_day0 \
  --coordSystem Dd

#D4

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/1M_aLp_WT_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC1ev_WT_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/1M_aLp_KO_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC1ev_KO_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/PC2_1M_aLp_WT_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC2ev_WT_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/PC2_1M_aLp_KO_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC2ev_KO_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/PC3_1M_aLp_WT_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC3ev_WT_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/PC3_1M_aLp_KO_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC3ev_KO_day4 \
  --coordSystem Dd


#/media/lucio/DATI/wkdir/bioinfo/Lara_multiomic_analysis/
# sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
#   --filename /data/dchange_wt_d4.bw \
#   --filetype bigwig \
#   --datatype vector \
#   --name dchange_wt_d4 \
#   --coordSystem Dd




sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/PC3_1M_aLp_KO_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name PC3ev_KO_day4 \
  --coordSystem Dd


#single chormosome compartments tracks

#d0

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_1M_aLp_WT_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name bychr_PC1ev_WT_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_1M_aLp_KO_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name bychr_PC1ev_KO_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_1M_aLp_WT_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name bychr_PC2ev_WT_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_1M_aLp_KO_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name bychr_PC2ev_KO_day0 \
  --coordSystem Dd

#d4

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_1M_aLp_WT_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name bychr_PC1ev_WT_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_1M_aLp_KO_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name bychr_PC1ev_KO_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_1M_aLp_WT_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name bychr_PC2ev_WT_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_1M_aLp_KO_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name bychr_PC2ev_KO_day4 \
  --coordSystem Dd


#350kb

#d0

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_350kb_aLp_WT_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 350kb_bychr_PC1ev_WT_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_350kb_aLp_KO_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 350kb_bychr_PC1ev_KO_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_350kb_aLp_WT_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 350kb_bychr_PC2ev_WT_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_350kb_aLp_KO_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 350kb_bychr_PC2ev_KO_day0 \
  --coordSystem Dd

#d4

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_350kb_aLp_WT_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 350kb_bychr_PC1ev_WT_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_350kb_aLp_KO_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 350kb_bychr_PC1ev_KO_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_350kb_aLp_WT_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 350kb_bychr_PC2ev_WT_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_350kb_aLp_KO_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 350kb_bychr_PC2ev_KO_day4 \
  --coordSystem Dd


#250kb


#d0

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_250kb_aLp_WT_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 250kb_bychr_PC1ev_WT_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_250kb_aLp_WT_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 250kb_bychr_PC2ev_WT_day0 \
  --coordSystem Dd


sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_250kb_aLp_KO_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 250kb_bychr_PC1ev_KO_day0 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_250kb_aLp_KO_day0.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 250kb_bychr_PC2ev_KO_day0 \
  --coordSystem Dd



#d4

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_250kb_aLp_WT_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 250kb_bychr_PC1ev_WT_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_250kb_aLp_WT_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 250kb_bychr_PC2ev_WT_day4 \
  --coordSystem Dd


sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_250kb_aLp_KO_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 250kb_bychr_PC1ev_KO_day4 \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_compartments/bychr_PC2_250kb_aLp_KO_day4.Dd.bw \
  --filetype bigwig \
  --datatype vector \
  --name 250kb_bychr_PC2ev_KO_day4 \
  --coordSystem Dd



### json load

In [ ]:
curl -X POST \
  -H "Content-Type: application/json" \
    --data-binary @mod_viewconf.json \
  http://localhost:8989/api/v1/viewconfs/


jq '{viewconf: .}' viewconf.json > wrapped_viewconf.json

curl -X POST \
  -H "Content-Type: application/json" \
  --data-binary @wrapped_viewconf.json \
  http://localhost:8989/api/v1/viewconfs/

http://localhost:8989/app/?config=cxrZfpgWR62aaV6oRIgwTQ

jq '{viewconf: .}'  KO_d0.json > wrapped_KO_d0.json
jq '{viewconf: .}'  WT_d0.json > wrapped_WT_d0.json

http://localhost:8989/app/?config=EFZc7iouTpCrHkVZBEz42A

curl -X POST \
  -H "Content-Type: application/json" \
  --data-binary @wrapped_WT_d0.json \
  http://localhost:8989/api/v1/viewconfs/

http://localhost:8989/app/?config=GjbX05dfT7qvV9kEZk-FWg

curl -X POST \
  -H "Content-Type: application/json" \
  --data-binary @wrapped_KO_d0.json \
  http://localhost:8989/api/v1/viewconfs/

http://localhost:8989/app/?config=QEinBowOTMW-y2KRgS-HPQ


## fanc insulation score

In [ ]:
## code

samples="aLp_KO_day0.Dd aLp_WT_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"

#for i in $samples ; do java -Xms512m -Xmx32g -jar ./git/nf-core-microc/bin/juicertools.jar pre -r 50000 --threads 12 ./out/pairtools_merge/${i}.pairs.gz ./out/pairtools_merge/50k_${i}.hic ./in/mm10.sizes ; done

sudo docker run -v `pwd`:`pwd` -w `pwd` -it -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0
samples="aLp_KO_day0.Dd aLp_WT_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"
for i in ${samples} ; do cooler cload pairix ./in/mm10.sizes:50000 ./out/pairtools_merge/${i}.pairs.gz ./out/pairtools_merge/50k_${i}.cool ; done
exit

#for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler cload pairix ./in/mm10.sizes:15000 ./out/pairtools_merge/${i}.pairs.gz ./out/pairtools_merge/15k_${i}.cool ; done


for i in ${samples} ; do cp ./out/pairtools_merge/15k_${i}.cool ./out/cooler_cload_balance/balanced_15k_${i}.cool ; done
    

#cooler dump -t pixels -b --balanced ./out/cooler_cload_balance/balanced_15k_aLp_WT_day0.Dd.cool > ./out/cooler_cload_balance/dump_balanced_15k_aLp_WT_day0.Dd.txt

#python3 /home/lucio/tools/cool2hic/3D-genome-tools/cool2hic.py -i ./out/cooler_cload_balance/balanced_15k_aLp_WT_day0.Dd.cool -r 15000


for i in ${samples} ; do cp ./out/pairtools_merge/50k_${i}.cool ./out/cooler_cload_balance/balanced_50k_${i}.cool ; done

for i in ${samples} ; do sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0 cooler balance ./out/cooler_cload_balance/balanced_1M_${i}.cool ; done



sudo docker run -v `pwd`:`pwd` -w `pwd` -it -u $(id -u):$(id -g) quay.io/biocontainers/cooler:0.9.2--pyh7cba7a3_0

samples="aLp_KO_day0.Dd aLp_WT_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"

for i in ${samples} ; do cooler balance ./out/cooler_cload_balance/balanced_15k_${i}.cool ; done    
for i in ${samples} ; do cooler balance ./out/cooler_cload_balance/balanced_50k_${i}.cool ; done

for i in ${samples} ; do cooler dump --join ./out/cooler_cload_balance/balanced_15k_${i}.cool > ./out/cooler_cload_balance/balanced_15k_${i}.txt ; done
for i in ${samples} ; do cooler dump --join ./out/cooler_cload_balance/balanced_50k_${i}.cool > ./out/cooler_cload_balance/balanced_50k_${i}.txt ; done

for i in ${samples} ; do cooler zoomify ./out/cooler_cload_balance/balanced_50k_${i}.cool ; done
for i in ${samples} ; do cooler zoomify ./out/cooler_cload_balance/balanced_15k_${i}.cool ; done

exit

# fanc insulation [-h] [-o OUTPUT_FORMAT]
#                        [-w WINDOW_SIZES [WINDOW_SIZES ...]] [-r REGION] [-i]
#                        [--offset OFFSET] [-L] [-N]
#                        [--normalisation-window NORMALISATION_WINDOW] [-s] [-g]
#                        [--trim-mean TRIM_MEAN] [-tmp]
#                        input [output]


#fanc insulation -o “bigwig” -w 100kb 150kb 200kb 250kb 300kb 350kb 400kb --offset 2 --log --geom-mean  

sudo docker run -it -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) lucidif/fanc:latest
samples="aLp_KO_day0.Dd aLp_WT_day0.Dd aLp_WT_day4.Dd aLp_KO_day4.Dd"

for i in ${samples}; do /fanc_env/bin/fanc insulation ./out/cooler_cload_balance/balanced_50k_${i}.cool -o bigwig -w 100kb 150kb 200kb 250kb 300kb 350kb 400kb --offset 2 --geom-mean ; done

#cp ./out/cooler_cload_balance/*.bigwig ./out/fanc_insulation/



for i in ${samples}; do /fanc_env/bin/fanc insulation ./out/cooler_cload_balance/balanced_15k_${i}.cool -o bigwig -w 100kb --offset 2 --geom-mean ; done

/fanc_env/bin/fanc insulation ./out/cooler_cload_balance/balanced_15k_aLp_WT_day0.Dd.cool -o bigwig -w 100kb --offset 2 --geom-mean

cp ./out/cooler_cload_balance/*.bigwig ./out/fanc_insulation/

rm ./out/cooler_cload_balance/*.bigwig

exit



## TADCompare

In [ ]:
%%script bash

DOCKER_ENV="git/Lara_MLL2/bin/docker_env.sh"
source "${DOCKER_ENV}"

# sudo docker run "${DOCKER_ARGS[@]}" \
#     -u $(id -u):$(id -g) \
#     lucidif/tadcompare:1.18.0

sudo docker run "${DOCKER_ARGS[@]}" \
    -u $(id -u):$(id -g) -it lucidif/tadcompare:1.18.0

In [ ]:
%%script R


source ("git/Lara_MLL2/bin/TADcompare_WT_DKO_d0_d4.R")




## TODEL higlass import

In [ ]:
%%script bash

sudo docker run --detach \
           --publish 8989:80 \
           --volume /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream:/data \
           --volume /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/tmp:/tmp \
           --name higlass-container \
         higlass/higlass-docker:v0.6.1

# sudo docker run --detach \
#            --publish 8989:80 \
#            --volume /media/lucio/DATI/wkdir/bioinfo/Lara_microC:/data \
#            --volume /media/lucio/DATI/wkdir/bioinfo/Lara_microC/tmp:/tmp \
#            --name higlass-container \
#          higlass/higlass-docker:v0.6.1

sudo docker start higlass-container

sudo docker exec -it higlass-container python higlass-server/manage.py migrate

sudo docker exec -it higlass-container higlass-server/manage.py createsuperuser

# reference

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/in/mm10.sizes \
  --filetype chromsizes-tsv \
  --datatype chromsizes \
  --coordSystem Dd

sudo docker exec higlass-container python \
    higlass-server/manage.py ingest_tileset \
    --filename /data/in/mm10.sizes \
    --filetype chromsizes-tsv \
    --datatype chromsizes \
    --name Lara_mm10 \
    --coordSystem Dd

# matrix 5k

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_5kb_aLp_WT_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_5kb_aLp_KO_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_5kb_aLp_WT_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_5kb_aLp_KO_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix 


# matrix 50k

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_50k_aLp_WT_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_50k_aLp_KO_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_50k_aLp_WT_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_50k_aLp_KO_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

# matrix 15k

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_15k_aLp_WT_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_15k_aLp_KO_day0.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_15k_aLp_WT_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix 

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/cooler_cload_balance/balanced_15k_aLp_KO_day4.Dd.mcool \
  --filetype cooler \
  --datatype matrix 


# bigwig

#D0

#WT

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_15k_aLp_WT_day0.Dd.cool_100kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 15k_WT_day0_100kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_WT_day0.Dd.cool_100kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_WT_day0_100kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_WT_day0.Dd.cool_150kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_WT_day0_150kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_WT_day0.Dd.cool_250kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_WT_day0_250kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_WT_day0.Dd.cool_350kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_WT_day0_350kb \
  --coordSystem Dd

#KO

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_15k_aLp_KO_day0.Dd.cool_100kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 15k_KO_day0_100kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_KO_day0.Dd.cool_100kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_KO_day0_100kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_KO_day0.Dd.cool_150kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_KO_day0_150kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_KO_day0.Dd.cool_250kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_KO_day0_250kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_KO_day0.Dd.cool_350kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_KO_day0_350kb \
  --coordSystem Dd

#wt d4

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_15k_aLp_WT_day0.Dd.cool_100kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 15k_WT_day4_100kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_WT_day0.Dd.cool_100kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_WT_day4_100kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_WT_day0.Dd.cool_150kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_WT_day4_150kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_WT_day0.Dd.cool_250kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_WT_day4_250kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_WT_day0.Dd.cool_350kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_WT_day4_350kb \
  --coordSystem Dd

#ko d4

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_15k_aLp_KO_day0.Dd.cool_100kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 15k_KO_day4_100kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_KO_day0.Dd.cool_100kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_KO_day4_100kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_KO_day0.Dd.cool_150kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_KO_day4_150kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_KO_day0.Dd.cool_250kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_KO_day4_250kb \
  --coordSystem Dd

sudo docker exec -it higlass-container python higlass-server/manage.py ingest_tileset \
  --filename /data/out/fanc_insulation/balanced_50k_aLp_KO_day0.Dd.cool_350kb.bigwig \
  --filetype bigwig \
  --datatype vector \
  --name 50k_KO_day4_350kb \
  --coordSystem Dd



#balanced_50k_aLp_KO_day0.Dd.cool_150kb.bigwig


## Higlass Compartments

In [ ]:
%%script bash

sh git/Lara_MLL2/bin/HiGlass_PC_TADs.sh

## Virtual 4C

In [ ]:
#sh git/Lara_MLL2/bin/Virtual_4C_analysis.sh

#sh git/Lara_MLL2/bin/Virtual_4C_extract_bigwig.sh

sh git/Lara_MLL2/bin/virtual4C_chr10_Rfx6_WTd4.sh



## save on storage

### raw data

In [ ]:
cd /mnt/datawk1/data/Lara/microC/HN00214039_fastq
md5sum *.fastq.gz > hash.txt

scp -r /mnt/datawk1/data/Lara/microC/HN00214039_fastq lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Data/flid/

ssh lucio@193.144.215.241

cd /volume1/Data_Just_In_NAS/Lucio/Data/flid/HN00214039_fastq

nrow=$(wc -l < hash.txt)
for ((i = 1; i <= nrow; i++)); do
    ##echo $i
    tarhash=`cut -d ' ' -f 1 hash.txt | head -n$i | tail -n1`
    trname=`cut -d ' ' -f 3 hash.txt | head -n$i | tail -n1`
    echo $flname
    echo $trname
    examin_hash=`md5sum $trname | cut -d ' ' -f 1`
    if [ "$tarhash" = "$examin_hash" ]; then
      checkout="ok"
    else
      checkout="ERROR"
    fi
    echo "${trname}: ${tarhash} = ${examin_hash}; ${checkout}"
done

cd /mnt/datawk1/data/Lara/2024_05_Lara_microC/fastq/

md5sum *.fastq.gz > hash.txt

scp -r /mnt/datawk1/data/Lara/2024_05_Lara_microC lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Data/flid/

ssh lucio@193.144.215.241

cd /volume1/Data_Just_In_NAS/Lucio/Data/flid/2024_05_Lara_microC/fastq

nrow=$(wc -l < hash.txt)
for ((i = 1; i <= nrow; i++)); do
    ##echo $i
    tarhash=`cut -d ' ' -f 1 hash.txt | head -n$i | tail -n1`
    trname=`cut -d ' ' -f 3 hash.txt | head -n$i | tail -n1`
    echo $flname
    echo $trname
    examin_hash=`md5sum $trname | cut -d ' ' -f 1`
    if [ "$tarhash" = "$examin_hash" ]; then
      checkout="ok"
    else
      checkout="ERROR"
    fi
    echo "${trname}: ${tarhash} = ${examin_hash}; ${checkout}"
done


cd /mnt/datawk1/data/Lara/2024_05_Lara_microC/fastq_batch2/

md5sum *.fastq.gz > hash.txt
scp -r /mnt/datawk1/data/Lara/2024_05_Lara_microC/fastq_batch2 lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Data/flid/2024_05_Lara_microC/

cd /mnt/datawk1/data/Lara/2024_05_Lara_microC/fastq_batch3/

md5sum *.fastq.gz > hash.txt

scp -r /mnt/datawk1/data/Lara/2024_05_Lara_microC/fastq_batch3 lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Data/flid/2024_05_Lara_microC/

scp /mnt/datawk1/data/Lara/2024_05_Lara_microC/StdIn_2024_05_batch2.csv lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Data/flid/2024_05_Lara_microC/
scp /mnt/datawk1/data/Lara/2024_05_Lara_microC/StdIn_2024_05_batch3.csv lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Data/flid/2024_05_Lara_microC/

ssh lucio@193.144.215.241

cd /volume1/Data_Just_In_NAS/Lucio/Data/flid/2024_05_Lara_microC/fastq_batch2

nrow=$(wc -l < hash.txt)
for ((i = 1; i <= nrow; i++)); do
    ##echo $i
    tarhash=`cut -d ' ' -f 1 hash.txt | head -n$i | tail -n1`
    trname=`cut -d ' ' -f 3 hash.txt | head -n$i | tail -n1`
    echo $flname
    echo $trname
    examin_hash=`md5sum $trname | cut -d ' ' -f 1`
    if [ "$tarhash" = "$examin_hash" ]; then
      checkout="ok"
    else
      checkout="ERROR"
    fi
    echo "${trname}: ${tarhash} = ${examin_hash}; ${checkout}"
done


cd /volume1/Data_Just_In_NAS/Lucio/Data/flid/2024_05_Lara_microC/fastq_batch3

nrow=$(wc -l < hash.txt)
for ((i = 1; i <= nrow; i++)); do
    ##echo $i
    tarhash=`cut -d ' ' -f 1 hash.txt | head -n$i | tail -n1`
    trname=`cut -d ' ' -f 3 hash.txt | head -n$i | tail -n1`
    echo $flname
    echo $trname
    examin_hash=`md5sum $trname | cut -d ' ' -f 1`
    if [ "$tarhash" = "$examin_hash" ]; then
      checkout="ok"
    else
      checkout="ERROR"
    fi
    echo "${trname}: ${tarhash} = ${examin_hash}; ${checkout}"
done





### analysis data

2024_03_Lara_microC storage

In [ ]:
%%script bash

sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/work
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/bwamem
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/chromsizes
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/juicer
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/cooler
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/pairix
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/pairtools/*.pairs.gz
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/pairtools/dedup/*.pairs.gz
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/pairtools/parse/*.pairsam.gz
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/samtools/*.bam
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/samtools/*.bai
sudo rm /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/pairtools/unsorted.bam

sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/mergeLP/bwamem
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/mergeLP/cut_fastq
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/mergeLP/pairix
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/mergeLP/chromsizes
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/mergeLP/pairtools/*.pairs.gz
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/mergeLP/pairtools/dedup/*.pairs.gz
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/mergeLP/pairtools/parse/*.pairsam.gz
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/mergeLP/samtools/*.bam
sudo rm -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/mergeLP/samtools/*.bai
sudo rm /mnt/datawk1/analysis/Lara/2024_03_Lara_microC/nfout/mergeLP/pairtools/unsorted.bam


scp -r /mnt/datawk1/analysis/Lara/2024_03_Lara_microC lucio@193.144.215.241:/volume1/Data_Just_In_NAS/Lucio/Analysis/Lara/






# scRNA

In [ ]:
%%R

source("./git/base/bin/reformat_macrogen.R")

reformat_macrogen ("/media/lucio/easystore/bck_data/Lara/scRNA/20260512_HN00274378/20260512_HN00274378_TRR_Report/assets/spgs/HN00274378_4samples_md5sum_DownloadLink.txt",
                   "/media/lucio/easystore/bck_data/Lara/scRNA/20260512_HN00274378/SS_20260512_HN00274378_Lara_scRNA.tsv")






In [ ]:
%%script bash

cd /media/lucio/easystore/bck_data/Lara/scRNA//20260512_HN00274378/fastq

ss="/media/lucio/easystore/bck_data/Lara/scRNA/20260512_HN00274378/SS_20260512_HN00274378_Lara_scRNA.tsv"

nrow=`wc -l $ss | cut -d' ' -f1`

for (( i=2; i<=nrow; i++ )); do
  samplename=`cut -f1 ${ss} | head -n $i | tail -n 1`
  url1=`cut -f2 ${ss} | head -n $i | tail -n 1`
  url2=`cut -f3 ${ss} | head -n $i | tail -n 1`
  echo $samplename
  echo $url1
  echo $url2
  wget -O "${samplename}_1.fastq.gz" "$url1"
  wget -O "${samplename}_2.fastq.gz" "$url2"
done

#md5 check
for (( i=2; i<=nrow; i++ )); do
  samplename=`cut -f1 ${ss} | head -n $i | tail -n 1`
  prevmd5_1=$(cut -f4 ${ss} | head -n $i | tail -n 1)
  curmd5_1=`md5sum ./${samplename}_1.fastq.gz | cut -d" " -f1`
  if [ "$curmd5_1" == "$prevmd5_1" ]; then
    echo "$samplename R1:${prevmd5_1} => ${curmd5_1} MD5 OK"
  else
    echo "$samplename R1:${prevmd5_1} => ${curmd5_1} MD5 WRONG"
  fi

  prevmd5_2=$(cut -f5 ${ss} | head -n $i | tail -n 1)
  curmd5_2=`md5sum ./${samplename}_2.fastq.gz | cut -d" " -f1`
  if [ "$curmd5_2" == "$prevmd5_2" ]; then
    echo "$samplename R2:${prevmd5_2} => ${curmd5_2} MD5 OK"
  else
    echo "$samplename R2:${prevmd5_2} => ${curmd5_2} MD5 WRONG"
  fi
done



# multi-omic

## make environment

In [ ]:
%%script bash

mkdir /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis

mkdir git

cd git

git clone git@github.com:lucidif/downstream_multiomic.git

cp /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/in/test_chipseq_downstream/macs_broadpeaks/

cp /mnt/datawk1/analysis/Lara/RNAseq_lara_day0_4_random/bamcoverage/* in/build38_DEseq2_RNAseq/bw/

##copy chip bed tracks 

cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/tmp/*.bw ./in/test_chipseq_downstream/deeptools_heatmap_tmp/ #TODO replace with origina source folder
cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig/*.bw ./in/test_chipseq_downstream/parallel_averageBigwig

/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO/in




## heatmap bivalent DEGg

### prepare environment

In [ ]:
%%script bash

#cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream

#mkdir ./otherouts
#cd ./otherouts

#mkdir deeptools_heatmaps
#cd deeptools_heatmaps
#mkdir tmp

macs_peaks="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/broad_0_5/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.threshold_3.broadPeak"
sed 's/ /\t/g' ${macs_peaks} | cut -f 1-6 > ./coordinate.bed

#outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO"
#inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO"

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/

cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO/input/* ./in/chip_bw/


### make degs genes bed

In [ ]:
D0degs<-read.table("in/build38_DEseq2_RNAseq/D0_DKO_vs_WT.deseq2.results.tsv",sep="\t", header=TRUE)

annotations<-read.table("/mnt/datawk1/references/annotations/ncbi/mm10_Build38/mm10.refGene.gtf.gz", sep="\t",header=FALSE)

window<-5000



tranno<-annotations[which(annotations$V3=="transcript"),]

gene_id<-c()
#chr<-c()
#transcript_start<-c()
#transcript_end<-c()
for(i in 1:nrow(tranno)){
  print(i)
  attr <- strsplit (tranno[i,9], "; ")
  attr <- gsub (";","", attr[[1]])
  gn <- attr[grep("gene_id", attr)]
  gn <- gsub( "gene_id ", "", gn )
  gene_id[i]<-gn
}

tranno<-cbind(gene_id=gene_id,tranno)

tss<-c()
for(i in 1:nrow(tranno)){
  if(tranno[i,"V7"]=="+"){
    tss[i]<-tranno[i,"V4"]
  }else{
    if(tranno[i,"V7"]=="-"){
      tss[i]<-tranno[i,"V5"]
    }else{ #no orientation
      tss[i]<-NA
    }
  }
}

tranno<-cbind(tranno,tss)

bed<-cbind(chr=tranno[,"V1"],
           start=as.numeric(tranno[,"tss"]) - window,
           end=as.numeric(tranno[,"tss"]) + window,
           gene=tranno[,"gene_id"]
           )


#dbfile1="/mnt/datawk1/references/annotations/biomart/v102/mart_export_GOterms_extended.txt"

#db<-read.delim(dbfile1,sep="\t",header=TRUE)

#db<-db[,c("Gene.name", "Gene.start..bp.", "Gene.end..bp.")]

#db<-db[which(duplicated(db)==FALSE),]

down<-D0degs[which(D0degs$padj<=0.05 & D0degs$log2FoldChange <= -1 ),]


subset.bed <- merge (bed, down, by.x=4, by.y=1)

subed<-subset.bed[,c(2,3,4,1)]
subed<-subed[which(duplicated(subed)==FALSE),]

write.table(subed,
            file="outs/downGenesHeatmap/D0dkoVSwt.downgenes.bed",
            col.names = FALSE,
            row.names = FALSE,
            sep="\t",
            quote = FALSE
            )




### make plot

In [ ]:
TODO : sostituisci con average

cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps

cp ../bedtools_window/proximal_CpG_plus_unique.bed ./
cp ../bedtools_window/proximal_CpG_minus.bed ./
cp ../bedtools_window/distal_CpG_plus_unique.bed ./
cp ../bedtools_window/distal_CpG_minus.bed ./

outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"
inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"
wkdir="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream"

plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"
macs_peaks="./tmp/proximal_CpG_plus_unique.bed ./tmp/proximal_CpG_minus.bed ./tmp/distal_CpG_plus_unique.bed ./tmp/distal_CpG_minus.bed"


panel3_A="./tmp/Anti-GFP_average.bw \
./tmp/Anti-Menin_average.bw \
./tmp/Anti-Mll1_average.bw \
./tmp/Mll2_KO_Mll1_average.bw \
./tmp/Anti-RbBp5_average.bw \
./tmp/Double_KO_RbBP5_average.bw \
../parallel_averageBigwig/D0_WT_H3K27ac__average.bw \
../parallel_averageBigwig/D0_Mll2KO_H3K27ac__average.bw \
../parallel_averageBigwig/D0_Mll1KO_H3K27ac__average.bw \
../parallel_averageBigwig/D0_DoubleKO_H3K27ac__average.bw \
./tmp/F_F_K27me3_average.bw \
./tmp/FC_FC_K27me3_average.bw \
./tmp/Mll1-KO_K27me3_average.bw \
./tmp/Double_KO_K27me3_average.bw \
./tmp/F_F_K4me3_average.bw \
./tmp/FC_FC_K4me3_average.bw \
./tmp/Mll1-KO_K4me3_average.bw \
./tmp/Double_KO_K4me3_average.bw \
./tmp/F_F_K4me2_average.bw \
./tmp/FC_FC_K4me2_average.bw \
./tmp/Mll1-KO_K4me2_average.bw \
./tmp/Double_KO_K4me2_average.bw \
./tmp/F_F_K4me1_average.bw \
./tmp/FC_FC_K4me1_average.bw \
./tmp/Mll1-KO_K4me1_average.bw \
./tmp/Double_KO_K4me1_average.bw \
"

panel3_B="./tmp/Anti-GFP_average.bw \
./tmp/Anti-Menin_average.bw \
./tmp/Anti-Mll1_average.bw \
./tmp/Mll2_KO_Mll1_average.bw \
./tmp/Anti-RbBp5_average.bw \
./tmp/Double_KO_RbBP5_average.bw \
../parallel_averageBigwig/D0_WT_H3K27ac__average.bw \
../parallel_averageBigwig/D0_WT_RING1B__average.bw \
../parallel_averageBigwig/D0_KO_RING1B__average.bw \
../parallel_averageBigwig/D0_WT_H3K9me3__average.bw \
../parallel_averageBigwig/D0_KO_H3K9me3__average.bw \
"


In [ ]:
%%script bash

#all track

outname="alltracks"

outpath="/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis"

#/outs/downGenesHeatmap/

samples="./in/chip_bw/Anti-GFP_average.bw ./in/chip_bw/Anti-Menin_average.bw ./in/chip_bw/Anti-Mll1_average.bw ./in/chip_bw/Mll2_KO_Mll1_average.bw ./in/chip_bw/Anti-RbBp5_average.bw ./in/chip_bw/Double_KO_RbBP5_average.bw \
./in/chip_bw/WT_H3K27ac_C.bigWig ./in/chip_bw/Mll2KO_H3K27ac_C.bigWig ./in/chip_bw/Mll1KO_H3K27ac_C.bigWig ./in/chip_bw/DoubleKO_H3K27ac_C.bigWig \
./in/chip_bw/F_F_K27me3_average.bw ./in/chip_bw/FC_FC_K27me3_average.bw ./in/chip_bw/Mll1-KO_K27me3_average.bw ./in/chip_bw/Double_KO_K27me3_average.bw \
./in/chip_bw/F_F_K4me3_average.bw ./in/chip_bw/FC_FC_K4me3_average.bw ./in/chip_bw/Mll1-KO_K4me3_average.bw ./in/chip_bw/Double_KO_K4me3_average.bw \
./in/chip_bw/F_F_K4me2_average.bw ./in/chip_bw/FC_FC_K4me2_average.bw ./in/chip_bw/Mll1-KO_K4me2_average.bw ./in/chip_bw/Double_KO_K4me2_average.bw \
./in/chip_bw/F_F_K4me1_average.bw ./in/chip_bw/FC_FC_K4me1_average.bw ./in/chip_bw/Mll1-KO_K4me1_average.bw ./in/chip_bw/Double_KO_K4me1_average.bw \
"

slabels="Anti-GFP_av Anti-Menin_av Anti-Mll1_av Mll2_KO_Mll1_av Anti-RbBp5_av Double_KO_RbBP5_av \
WT_H3K27ac_av Mll2KO_H3K27ac_av Mll1KO_H3K27ac_av DoubleKO_H3K27ac_av \
F_F_K27me3_av FC_FC_K27me3_av Mll1-KO_K27me3_av Double_KO_K27me3_av \
F_F_K4me3_av FC_FC_K4me3_av Mll1-KO_K4me3_av Double_KO_K4me3_av \
F_F_K4me2_av FC_FC_K4me2_av Mll1-KO_K4me2_av Double_KO_K4me2_av \
F_F_K4me1_av FC_FC_K4me1_av Mll1-KO_K4me1_av Double_KO_K4me1_av \
"

thrsholds="10 10 10 10 15 15 10 10 10 10 5 5 5 5 20 20 20 20 30 30 30 30 10 10 10 10"

sudo docker run -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -R outs/downGenesHeatmap/D0dkoVSwt.downgenes.bed -S $samples -b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 --averageTypeBins "median" --outFileName outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName $outpath/${outname}_plotHeatmap.pdf --sortUsingSamples 7 --samplesLabel ${slabels} --regionsLabel ${plabels} --outFileNameMatrix outs/downGenesHeatmap/${outname}_plotHeatmap.mat.tab

# panel B

thrsholds_B="10 10 10 10 15 15 10 20 20 3 3"
outname_B="additional_tracks"
slabels_B="Anti-GFP_av Anti-Menin_av Anti-Mll1_av Mll2_KO_Mll1_av Anti-RbBp5_av Double_KO_RbBP5_av \
WT_H3K27ac_av \
WT_RING1B_av \
KO_RING1B_av \
WT_H3K9me3_av \
KO_H3K9me3_av \
"

sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S $inpath/${panel3_B} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname_B}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $wkdir:$wkdir -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile $outpath/${outname_B}_deeptools_matrix.gzip \
--zMax ${thrsholds_B} --outFileName $outpath/${outname_B}_plotHeatmap.pdf --sortUsingSamples 7 \
--samplesLabel ${slabels_B} --regionsLabel ${plabels} \
--outFileNameMatrix $outpath/${outname_B}_plotHeatmap.mat.tab



## line distribution evaluation ✅

### distance filters evaluation

In [ ]:
%%script R

source("git/downstream_multiomic/bin/Lara_CHiP_postprocessing_line1_dist.R")

#test line families inside peaks bed by distance

files<-c("distfilter_DKO_K4me3_dcm.l1.bed",
"distfilter1000_DKO_K4me3_dcm.l1.bed",
"distfilter500_DKO_K4me3_dcm.l1.bed",
"distfilter400_DKO_K4me3_dcm.l1.bed",
"distfilter300_DKO_K4me3_dcm.l1.bed",
"distfilter200_DKO_K4me3_dcm.l1.bed",
"distfilter100_DKO_K4me3_dcm.l1.bed"
)

fam1val<-c()
fam2val<-c()
fam3val<-c()
fam4val<-c()
nm1<-c()
nm2<-c()
nm3<-c()
nm4<-c()

for (i in 1:length(files)){

  print("#=======================================")

  bd<-read.table(paste0("/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/",files[i]))

  arr<-bd[,4]

  transformed_array <- gsub("_\\d+$", "", arr)

  sror<-table(transformed_array)

  tb<-sror[order(sror,decreasing = TRUE)]

  print(paste0(files[i]))
  print(tb[1:4])
  print(paste0("other lines",sum(tb[5:length(tb)])))

  print("ratio top4 vs others")
  print(sum(tb[1:4])/sum(tb[5:length(tb)]))

    fam1val[i]<-tb[1]
    fam2val[i]<-tb[2]
    fam3val[i]<-tb[3]
    fam4val[i]<-tb[4]
    nm1[i]<-names(tb[1])
    nm2[i]<-names(tb[2])
    nm3[i]<-names(tb[3])
    nm4[i]<-names(tb[4])

  if(i!=1){

    nmpre<-c(nm1[i-1],nm2[i-1],nm3[i-1],nm4[i-1])
    valpre<-c(fam1val[i-1],fam2val[i-1],fam3val[i-1],fam4val[i-1])

    preval1<-valpre[grep(nm1[i],nmpre)]
    preval2<-valpre[grep(nm2[i],nmpre)]
    preval3<-valpre[grep(nm3[i],nmpre)]
    preval4<-valpre[grep(nm4[i],nmpre)]

    print(paste0(nm1[i]," ratio with previous value ",fam1val[i]/preval1))
    print(paste0(nm2[i]," ratio with previous value ",fam2val[i]/preval2))
    print(paste0(nm3[i]," ratio with previous value ",fam3val[i]/preval3))
    print(paste0(nm4[i]," ratio with previous value ",fam4val[i]/preval4))

  }

  print("#=======================================")


}

# [1] "#======================================="
# [1] "distfilter_DKO_K4me3_dcm.l1.bed"
# transformed_array
#  L1Md_T  L1Md_A L1Md_F2 L1Md_Gf
#    1672     603     166     129
# [1] "other lines1268"
# [1] "ratio top4 vs others"
# [1] 2.026814
# [1] "#======================================="
# [1] "#======================================="
# [1] "distfilter1000_DKO_K4me3_dcm.l1.bed"
# transformed_array
#  L1Md_T  L1Md_A L1Md_F2 L1Md_Gf
#    1665     597     132     128
# [1] "other lines633"
# [1] "ratio top4 vs others"
# [1] 3.984202
# [1] "L1Md_T ratio with previous value 0.995813397129187"
# [1] "L1Md_A ratio with previous value 0.990049751243781"
# [1] "L1Md_F2 ratio with previous value 0.795180722891566"
# [1] "L1Md_Gf ratio with previous value 0.992248062015504"
# [1] "#======================================="
# [1] "#======================================="
# [1] "distfilter500_DKO_K4me3_dcm.l1.bed"
# transformed_array
#  L1Md_T  L1Md_A L1Md_Gf L1Md_F2
#    1597     555     122     120
# [1] "other lines369"
# [1] "ratio top4 vs others"
# [1] 6.487805
# [1] "L1Md_T ratio with previous value 0.959159159159159"
# [1] "L1Md_A ratio with previous value 0.92964824120603"
# [1] "L1Md_Gf ratio with previous value 0.953125"
# [1] "L1Md_F2 ratio with previous value 0.909090909090909"
# [1] "#======================================="
# [1] "#======================================="
# [1] "distfilter400_DKO_K4me3_dcm.l1.bed"
# transformed_array
#  L1Md_T  L1Md_A L1Md_Gf L1Md_F2
#    1499     495     117     106
# [1] "other lines322"
# [1] "ratio top4 vs others"
# [1] 6.885093
# [1] "L1Md_T ratio with previous value 0.938634940513463"
# [1] "L1Md_A ratio with previous value 0.891891891891892"
# [1] "L1Md_Gf ratio with previous value 0.959016393442623"
# [1] "L1Md_F2 ratio with previous value 0.883333333333333"
# [1] "#======================================="
# [1] "#======================================="
# [1] "distfilter300_DKO_K4me3_dcm.l1.bed"
# transformed_array
#  L1Md_T  L1Md_A L1Md_Gf L1Md_F2
#    1319     396     108      81
# [1] "other lines263"
# [1] "ratio top4 vs others"
# [1] 7.239544
# [1] "L1Md_T ratio with previous value 0.879919946631087"
# [1] "L1Md_A ratio with previous value 0.8"
# [1] "L1Md_Gf ratio with previous value 0.923076923076923"
# [1] "L1Md_F2 ratio with previous value 0.764150943396226"
# [1] "#======================================="
# [1] "#======================================="
# [1] "distfilter200_DKO_K4me3_dcm.l1.bed"
# transformed_array
#  L1Md_T  L1Md_A L1Md_Gf L1Md_F2
#     981     257      75      55
# [1] "other lines184"
# [1] "ratio top4 vs others"
# [1] 7.434783
# [1] "L1Md_T ratio with previous value 0.743745261561789"
# [1] "L1Md_A ratio with previous value 0.648989898989899"
# [1] "L1Md_Gf ratio with previous value 0.694444444444444"
# [1] "L1Md_F2 ratio with previous value 0.679012345679012"
# [1] "#======================================="
# [1] "#======================================="
# [1] "distfilter100_DKO_K4me3_dcm.l1.bed"
# transformed_array
#  L1Md_T  L1Md_A L1Md_Gf L1Md_F2
#     533     117      43      19
# [1] "other lines108"
# [1] "ratio top4 vs others"
# [1] 6.592593
# [1] "L1Md_T ratio with previous value 0.543323139653415"
# [1] "L1Md_A ratio with previous value 0.455252918287938"
# [1] "L1Md_Gf ratio with previous value 0.573333333333333"
# [1] "L1Md_F2 ratio with previous value 0.345454545454545"
# [1] "#======================================="



### filter 500bp TSS distance by line ✅


In [ ]:
%%R

#if not devined the environment
#setwd("/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/")

source("./git/Lara_MLL2/bin/500bp_Lara_distance_by_nearest_DEGs.R")

#TODO the source don't save png of plot add this in script


## GC content evaluation

### tss centering

In [ ]:
%%script R

read.table("outs/CHiP_postprocessing_line1_dist/distfilter500_DKO_K4me3_dcm.l1.bed", sep="\t", )


### make heatmap

In [ ]:
%%script bash

#d0 tracks and d0 + d4 tracks in CpG content heatmap

git/Lara_MLL2/bin/heatmaps_CpG_content.sh

#### cumulative distibution

In [ ]:
%%script R

source("git/Lara_MLL2/bin/l1_d4_cumulative_dist.R")


### profiles

In [ ]:
%%script bash



inpath="/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis"
outpath="/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/gc_content_heatmap/"

## plot profile separately
## --perGroup make one image per BED file instead of per bigWig file
## you must split the in matrix by modification

#all preaks file

#=============H3K27ac=================

outname="DKO_K4me3_dcm_l1_H3K27ac_only"
macs_peaks="outs/gc_content_heatmap/recentered_distfil500_DKO_K4me3_dcm.l1.bed"
#plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"
#allpeaks="input/fold2_th0_05_K4me2_Double_KO_vs_F_F_DBA_DESEQ2_DOWN.bed"

samples="./in/test_chipseq_downstream/parallel_averageBigwig/D0_WT_H3K27ac__average.bw ./in/test_chipseq_downstream/parallel_averageBigwig/D0_Mll1KO_H3K27ac__average.bw ./in/test_chipseq_downstream/parallel_averageBigwig/D0_Mll2KO_H3K27ac__average.bw ./in/test_chipseq_downstream/parallel_averageBigwig/D0_DoubleKO_H3K27ac__average.bw"

slabels="WT Mll1KO Mll2KO DKO"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $inpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${samples} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotProfile -m $outpath/${outname}_deeptools_matrix.gzip -out ${outpath}/${outname}_Profile.png --perGroup --colors "#715eee" "#de217d" "#ff5f01" "#ffb00e" --plotTitle "line1 near H3K27ac loss K4me3 in dko vs wt" --samplesLabel ${slabels}


# data_d0[grep("D0DoubleKO_",data_d0[,1]),2]<-"#ffb00e" #"DKO"
# data_d0[grep("D0Mll1KO_",data_d0[,1]),2]<-"#de217d" #"Mll1KO"
# data_d0[grep("D0Mll2KO_",data_d0[,1]),2]<-"#ff5f01" #"Mll2KO"
# data_d0[grep("D0WTA_",data_d0[,1]),2]<-"#715eee" #"WTA"

#=================

#=============H3K27me3=================

title="H3K27me3"
outname="DKO_K4me3_dcm_l1_${title}_only"
macs_peaks="outs/gc_content_heatmap/recentered_distfil500_DKO_K4me3_dcm.l1.bed"
#plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"

samples="./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K27me3_average.bw ./in/test_chipseq_downstream/deeptools_heatmap_tmp/Mll1-KO_K27me3_average.bw ./in/test_chipseq_downstream/deeptools_heatmap_tmp/FC_FC_K27me3_average.bw ./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K27me3_average.bw"

slabels="WT Mll1KO Mll2KO DKO"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $inpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${samples} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotProfile -m $outpath/${outname}_deeptools_matrix.gzip -out ${outpath}/${outname}_Profile.png --perGroup --colors "#715eee" "#de217d" "#ff5f01" "#ffb00e" --plotTitle "line1 near H3K27ac loss K4me3 in dko vs wt" --samplesLabel ${slabels}


#=============K4me3=================

title="K4me3"
outname="K4me3_DOWN_Double_KO_vs_F_F_${title}_only"
macs_peaks="outs/gc_content_heatmap/recentered_distfil500_DKO_K4me3_dcm.l1.bed"
#plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"

samples="./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K4me3_average.bw ./in/test_chipseq_downstream/deeptools_heatmap_tmp/Mll1-KO_K4me3_average.bw ./in/test_chipseq_downstream/deeptools_heatmap_tmp/FC_FC_K4me3_average.bw ./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K4me3_average.bw"

slabels="WT Mll1KO Mll2KO DKO"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $inpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${samples} -R $macs_peaks \
-b 1000 --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 \
--averageTypeBins "median" --outFileName $outpath/${outname}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $inpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotProfile -m $outpath/${outname}_deeptools_matrix.gzip -out ${outpath}/${outname}_Profile.png --perGroup --colors "#715eee" "#de217d" "#ff5f01" "#ffb00e" --plotTitle "${title} profile l1" --samplesLabel ${slabels} 

sh git/Lara_MLL2/bin/profiles_terget_lines.sh



## hypergeometric test ✅

### TODEL distance cutoff 2500bp

#### environment

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis

cd ./git

git clone git@github.com:lucidif/downstream_multiomic.git

mkdir ./in/build38_DEseq2_RNAseq

cp /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4/nfout/build38/differentialabundance/tables/differential/* ./in/build38_DEseq2_RNAseq/

cp /mnt/datawk1/analysis/Lara/DE_RNAseq_lara_day0_4/nfout/build38/differentialabundance/tables/annotation/mm10.anno.tsv ./in/build38_DEseq2_RNAseq/



#### make docker

In [ ]:
%%script bash

sudo docker pull r-base:4.4.2

sudo docker run -it r-base:4.4.2


inside the docker run

In [ ]:
%%R

install.packages("VennDiagram")




#### run R script

In [ ]:
%%R

setwd("/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis")

source("./git/downstream_multiomic/bin/Lara_overlap.R")


### cutoff 500bp ✅

#### environment

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis

mkdir ./outs/overlap_cutoff500bp

cd ./git

git clone git@github.com:lucidif/downstream_multiomic.git



#### run script

In [ ]:
%%R

setwd("/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis")

source("./git/downstream_multiomic/bin/Lara_overlap_filter500.R")

## Distance between DEG and D+/D- comparing to expected 

##### make environment

In [ ]:
%%script bash

md5sum /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/in/test_chipseq_downstream/deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed

# 205f249a856ccc157639136533f08525

md5sum /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed

# 205f249a856ccc157639136533f08525

/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/great/Double_KO_vs_F_F/basal/dcp/20250210-public-4.0.4-X2dyH7-mm10-all-gene.txt


##### make analysis

In [ ]:
%%script R

source("./git/Lara_MLL2/bin/D0D4_dist_DEG_dcp.R")


## microC pileup


### environment

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis

mkdir ./outs/coolpup
mkdir ./outs/coolpup/500bp

mkdir ./outs/coolpup/500bp/D0
mkdir ./outs/coolpup/500bp/D4

mkdir ./outs/coolpup/500bp/D0/15kb
mkdir ./outs/coolpup/500bp/D0/5kb
mkdir ./outs/coolpup/500bp/D4/15kb
mkdir ./outs/coolpup/500bp/D4/5kb




### TODEL dist filter 2500 ✅

#### anchor 1 : all peaks dcm differential down genes in DKO/WT comparison (peaks) near (distance : 2500) from line 1

TODO format original chipseq data (see filter by distance from LINE) by adding the coordinate of the peak and the cordinate of l1 TSS in a bedpe format


In [ ]:
%%R

setwd("/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis")

lines.file<-read.table("./outs/distfilter_DKO_K4me3_dcm.l1.bed",sep="\t")
peaks.file<-read.table("./outs/distfilter_DKO_K4me3_dcm.target.peaks.bed",sep="\t")
row.names(peaks.file)<-peaks.file$V4
great.basal.dcm.genes<-read.table("./outs/great/filtered_by_distance/basal/dcm/20240719-public-4.0.4-nxvS8V-mm10-all-gene.txt", sep="\t")
anno<-read.table("./in/GREATv4/GREATv4.genes.mm10.tsv", sep="\t")


#i<-813
for (i in 1:nrow(great.basal.dcm.genes)){
  tar.great<-great.basal.dcm.genes[i,]
  tarpeaks<-strsplit(great.basal.dcm.genes$V2[i],",")
  tarpeaks.arr<-unlist(strsplit(unlist(tarpeaks), " "))
  tarpeaks.arr<-tarpeaks.arr[grep("peak",tarpeaks.arr)]

  tarpeaks.arr<-cbind(tarpeaks.arr, rep(great.basal.dcm.genes[i,1], length(tarpeaks.arr)))

  if(i==1){
    ttfinal<-tarpeaks.arr
  }else{
    ttfinal<-rbind(ttfinal,tarpeaks.arr)
  }

}
colnames(ttfinal)<-c("peak","gene")
#row.names(anno)<-anno[,5]

gene.coords<-merge(ttfinal,anno, by.x="gene", by.y="V5")
colnames(gene.coords)<-c("gene","peaks","ensembl","gene.chr","gene.tss","gene.strand")

gene.peaks.coords<-merge(gene.coords, peaks.file, by.x="peaks", by.y="V4")

colnames(gene.peaks.coords)[7:11]<-c("peaks.chr","peak.start","peak.end","peak.value","peak.strand")

bedpe1<-cbind(
              gene.peaks.coords$peaks.chr,
              gene.peaks.coords$peak.start,
              gene.peaks.coords$peak.end,
              gene.peaks.coords$peak.strand ,
              gene.peaks.coords$gene.chr,
              gene.peaks.coords$gene.tss-1,
              gene.peaks.coords$gene.tss+1,
              gene.peaks.coords$gene.strand
              )

write.table(bedpe1,
            file="outs/anchors1.bedpe",
            quote=FALSE,
            sep="\t",
            col.names = FALSE,
            row.names = FALSE
            )


bedpe1ext<-cbind(
  gene.peaks.coords$peaks.chr,
  gene.peaks.coords$peak.start,
  gene.peaks.coords$peak.end,
  gene.peaks.coords$peak.strand ,
  gene.peaks.coords$gene.chr,
  gene.peaks.coords$gene.tss-1,
  gene.peaks.coords$gene.tss+1,
  gene.peaks.coords$gene.strand,
  gene.peaks.coords$peaks,
  gene.peaks.coords$gene
)

write.table(bedpe1ext,
            file="outs/anchors1_extended.bedpe",
            quote=FALSE,
            sep="\t",
            col.names = FALSE,
            row.names = FALSE
            )



bed<-cbind(gene.peaks.coords$peaks.chr,
              gene.peaks.coords$peak.start,
              gene.peaks.coords$peak.end,
              gene.peaks.coords$gene.strand
           )

write.table(bed,
            file="outs/anchors1.bed",
            quote=FALSE,
            sep="\t",
            col.names = FALSE,
            row.names = FALSE)




#### anchor 2 D0: great taget genes obtaines by the previous peaks

In [ ]:
%%R

setwd("/media/lucio/external.wk/bioinfo/wkdir/Lara/Lara_multiomic_analysis")
#setwd("/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/")

# lines.file<-read.table("./outs/distfilter_DKO_K4me3_dcm.l1.bed",sep="\t")
# peaks.file<-read.table("./outs/distfilter_DKO_K4me3_dcm.target.peaks.bed",sep="\t")
# row.names(peaks.file)<-peaks.file$V4
# great.basal.dcm.genes<-read.table("./outs/great/filtered_by_distance/basal/dcm/20240719-public-4.0.4-nxvS8V-mm10-all-gene.txt", sep="\t")
# anno<-read.table("./in/GREATv4/GREATv4.genes.mm10.tsv", sep="\t")

anch1<-read.table("outs/anchors1_extended.bedpe",
                  sep="\t",
                  header=FALSE
                  )

d0.down.degs<-read.table("outs/overlap/D0.down.tsv",
                         sep="\t",
                         header=TRUE
                         )
anch1.d0.down<-merge(anch1, d0.down.degs, by.x=10, by.y=1)

anch2<-anch1.d0.down[,2:9]
anch2<-anch2[,c(1,2,3,5,6,7,4,8)]

anch2bed<-anch2[,c(1:3)]

peak.center<-anch2[,2]+round((anch2[,3]-anch2[,2])/2,0)

anch2.centered.peaks<-anch2

anch2.centered.peaks[,2]<-peak.center-1
anch2.centered.peaks[,3]<-peak.center+1

anch2.peaksOnL.id <- c()
anch2.peaksOnR.id <- c()
anch2.peaksOnL.progressor<-1
anch2.peaksOnR.progressor<-1

anch2.peaksOnR.progressor[]

for (i in 1:nrow(anch2.centered.peaks)){

  peak.coord <- (anch2.centered.peaks[,"V2"] + 1 )[i]
  tss.coord <-  (anch2.centered.peaks[,"V6"] + 1 )[i]

  if(peak.coord <= tss.coord ){
    anch2.peaksOnL.id[anch2.peaksOnL.progressor]<-i
    anch2.peaksOnL.progressor<-anch2.peaksOnL.progressor+1
  }else{
    anch2.peaksOnR.id[anch2.peaksOnR.progressor]<-i
    anch2.peaksOnR.progressor<-anch2.peaksOnR.progressor+1
  }


}

sorted.anchs<-rbind(anch2.centered.peaks[anch2.peaksOnL.id,],
                    anch2.centered.peaks[anch2.peaksOnR.id,c("V5","V6","V7","V1","V2","V3","V4","V8")]
                    )



write.table(anch2,"outs/anchors2.bedpe",
            quote=FALSE,
            sep="\t",
            col.names = FALSE,
            row.names = FALSE
            )


write.table(anch2bed,"outs/anchors2.bed",
            quote=FALSE,
            sep="\t",
            col.names = FALSE,
            row.names = FALSE
            )

write.table(
            anch2.centered.peaks[anch2.peaksOnL.id,],
            "outs/anchors2_center_peaks.bedpe",
            quote=FALSE,
            sep="\t",
            col.names = FALSE,
            row.names = FALSE
            )

write.table(
          sorted.anchs,
          "outs/anchors2_sorted.bedpe",
          quote=FALSE,
          sep="\t",
          col.names = FALSE,
          row.names = FALSE
)



#### anchor 2 : D4

In [ ]:
%%R

setwd("/media/lucio/external.wk/bioinfo/wkdir/Lara/Lara_multiomic_analysis")

anch1<-read.table("outs/anchors1_extended.bedpe",
                  sep="\t",
                  header=FALSE
                  )

d4.down.degs<-read.table("outs/overlap/D4.down.tsv",
                         sep="\t",
                         header=TRUE
                         )
anch1.d4.down<-merge(anch1, d4.down.degs, by.x=10, by.y=1)

anch2<-anch1.d4.down[,2:9]
anch2<-anch2[,c(1,2,3,5,6,7,4,8)]

anch2bed<-anch2[,c(1:3)]

write.table(anch2,"outs/anchors2_d4.bedpe",
            quote=FALSE,
            sep="\t",
            col.names = FALSE,
            row.names = FALSE
            )


write.table(anch2bed,"outs/anchors2_d4.bed",
            quote=FALSE,
            sep="\t",
            col.names = FALSE,
            row.names = FALSE
            )






#### anchor 3 : great target genes overlapped with RNAseq founded genes

In [ ]:
%%R

setwd("/media/lucio/external.wk/bioinfo/wkdir/Lara/Lara_multiomic_analysis")


anch1<-read.table("outs/anchors1_extended.bedpe",
                  sep="\t",
                  header=FALSE
)

proximal.peaks<-read.table("outs/great/DoubleKO_vs_F_F/basal/proximal/20241114-public-4.0.4-8PRA6m-mm10-all-gene.txt",
                           sep="\t",
                           header=FALSE)

anch3<-merge(anch1,proximal.peaks, by.x=10, by.y=1)
anch3<-cbind(anch3, anch3$V10)
anch3<-anch3[,c(-1,-10)]
anch3<-anch3[,c(1,2,3,5,6,7,4,8)]

write.table(anch3,"outs/anchors3.bedpe",
            quote=FALSE,
            sep="\t",
            col.names = FALSE,
            row.names = FALSE
)



### dist filter 500bp

#### anchor1 ✅

In [ ]:
%%R

#=========================
#        paramters
#=========================

outpath="outs/coolpup/500bp/"
windowing=TRUE
windows<-c(1,500,1000)

#==========================
#         pathfile
#==========================

lines.file<-read.table("./outs/CHiP_postprocessing_line1_dist/distfilter500_DKO_K4me3_dcm.l1.bed",sep="\t")
peaks.file<-read.table("./outs/CHiP_postprocessing_line1_dist/distfilter500_DKO_K4me3_dcm.target.peaks.bed",sep="\t")
row.names(peaks.file)<-peaks.file$V4
great.basal.dcm.genes<-read.table("./outs/great/Double_KO_vs_F_F/basal/filtered_by_distance_500bp/dcm/20241126-public-4.0.4-hpxvmj-mm10-all-gene.txt", sep="\t")
anno<-read.table("./in/GREATv4/GREATv4.genes.mm10.tsv", sep="\t")

#i<-813
for (i in 1:nrow(great.basal.dcm.genes)){
  tar.great<-great.basal.dcm.genes[i,]
  tarpeaks<-strsplit(great.basal.dcm.genes$V2[i],",")
  tarpeaks.arr<-unlist(strsplit(unlist(tarpeaks), " "))
  tarpeaks.arr<-tarpeaks.arr[grep("peak",tarpeaks.arr)]

  tarpeaks.arr<-cbind(tarpeaks.arr, rep(great.basal.dcm.genes[i,1], length(tarpeaks.arr)))

  if(i==1){
    ttfinal<-tarpeaks.arr
  }else{
    ttfinal<-rbind(ttfinal,tarpeaks.arr)
  }

}
colnames(ttfinal)<-c("peak","gene")
#row.names(anno)<-anno[,5]

gene.coords<-merge(ttfinal,anno, by.x="gene", by.y="V5")
colnames(gene.coords)<-c("gene","peaks","ensembl","gene.chr","gene.tss","gene.strand")

gene.peaks.coords<-merge(gene.coords, peaks.file, by.x="peaks", by.y="V4")

colnames(gene.peaks.coords)[7:11]<-c("peaks.chr","peak.start","peak.end","peak.value","peak.strand")

bedpe1_unsort<-data.frame(
  chrA=gene.peaks.coords$peaks.chr,
  startA=gene.peaks.coords$peak.start,
  endA=gene.peaks.coords$peak.end,
  #gene.peaks.coords$peak.strand ,
  chrB=gene.peaks.coords$gene.chr,
  startB=gene.peaks.coords$gene.tss-1,
  endB=gene.peaks.coords$gene.tss+1
  #,gene.peaks.coords$gene.strand
  ,gene.peaks.coords$peaks
  ,gene.peaks.coords$gene
)

bedpe1<-bedpe1_unsort

for (i in 1:nrow(bedpe1_unsort)){

  meanA <- round((bedpe1_unsort$startA[i] + ((bedpe1_unsort$endA[i] - bedpe1_unsort$startA [i] )/2)),0)
  meanB <- round((bedpe1_unsort$startB[i] + ((bedpe1_unsort$endB[i] - bedpe1_unsort$startB [i] )/2)),0)

  if(meanB<meanA){
    bedpe1[i,1]<-bedpe1_unsort[i,4]
    bedpe1[i,2]<-bedpe1_unsort[i,5]
    bedpe1[i,3]<-bedpe1_unsort[i,6]
    bedpe1[i,4]<-bedpe1_unsort[i,1]
    bedpe1[i,5]<-bedpe1_unsort[i,2]
    bedpe1[i,6]<-bedpe1_unsort[i,3]
  }


}


if(windowing==TRUE){

  #bedpe2
  for (i in 1:length(windows)){

    tarwin<-windows[i]

    meanA<-round((bedpe1$startA + ((bedpe1$endA - bedpe1$startA)/2)),0)
    meanB<-round((bedpe1$startB + ((bedpe1$endB - bedpe1$startB)/2)),0)

    windowed_bendpe<-data.frame(
      chrA=bedpe1$chrA,
      startA=meanA-tarwin,
      endA=meanA+tarwin,
      chrB=bedpe1$chrA,
      startB=meanB-tarwin,
      endB=meanB+tarwin,
      peak.name=bedpe1$gene.peaks.coords.peaks,
      gene=bedpe1$gene.peaks.coords.gene
    )

    write.table(windowed_bendpe,
                file=paste0(outpath,"/win",tarwin,"_anchors1.bedpe"),
                quote=FALSE,
                sep="\t",
                col.names = FALSE,
                row.names = FALSE
    )

  }

} else {
  write.table(bedpe1,
              file=paste0(outpath,"500_anchors1.bedpe"),
              quote=FALSE,
              sep="\t",
              col.names = FALSE,
              row.names = FALSE
  )
}


# write.table(bedpe1ext,
#             file="outs/500_anchors1_extended.bedpe",
#             quote=FALSE,
#             sep="\t",
#             col.names = FALSE,
#             row.names = FALSE
# )

# bed<-cbind(gene.peaks.coords$peaks.chr,
#            gene.peaks.coords$peak.start,
#            gene.peaks.coords$peak.end,
#            gene.peaks.coords$gene.strand
# )

# write.table(bed,
#             paste0(outpath,"anchors1.bed"),
#             quote=FALSE,
#             sep="\t",
#             col.names = FALSE,
#             row.names = FALSE)





#### anchor2 ✅

In [ ]:
%%R

outpath="outs/coolpup/500bp/"

anch1<-read.table(paste0(outpath,"win500_anchors1.bedpe"),
                  sep="\t",
                  header=FALSE
)

d0.down.degs<-read.table("outs/overlap/D0.down.tsv",
                         sep="\t",
                         header=TRUE
)
anch1.d0.down<-merge(anch1, d0.down.degs, by.x=8, by.y=1)

anch2_unsort<-anch1.d0.down[,2:9]
anch2_unsort<-anch2[,c(1,2,3,4,5,6)]

anch2bed<-anch2_unsort[,c(1:3)]

colnames(anch2_unsort)<-c("chrA","startA","endA","chrB","startB","endB")

anch2<-anch2_unsort

for (i in 1:nrow(anch2_unsort)){

    meanA <- round((anch2_unsort$startA[i] + ((anch2_unsort$endA[i] - anch2_unsort$startA [i] )/2)),0)
    meanB <- round((anch2_unsort$startB[i] + ((anch2_unsort$endB[i] - anch2_unsort$startB [i] )/2)),0)

    if(meanB<meanA){
      anch2[i,1]<-anch2_unsort[i,4]
      anch2[i,2]<-anch2_unsort[i,5]
      anch2[i,3]<-anch2_unsort[i,6]
      anch2[i,4]<-anch2_unsort[i,1]
      anch2[i,5]<-anch2_unsort[i,2]
      anch2[i,6]<-anch2_unsort[i,3]
    }

}

window <- 500

meansA <- round((anch2$startA + ((anch2$endA - anch2$startA )/2)),0)
meansB <- round((anch2$startB + ((anch2$endB - anch2$startB )/2)),0)

anch2$startA<-meansA-window
anch2$endA<-meansA+window

anch2$startB<-meansB-window
anch2$endB<-meansB+window

# make.anch(outfile = "outs/coolpup/500bp/win500_anchors3.bedpe",
#           anch1.path = "outs/coolpup/500bp/win500_anchors1.bedpe",
#           peaks.path = "outs/great/Double_KO_vs_F_F/basal/proximal/20241126-public-4.0.4-xCT3F1-mm10-all-gene.txt"
#           )

# make.anch(outfile = "outs/coolpup/500bp/win1_anchors3.bedpe",
#           anch1.path = "outs/coolpup/500bp/win1_anchors1.bedpe",
#           peaks.path = "outs/great/Double_KO_vs_F_F/basal/proximal/20241126-public-4.0.4-xCT3F1-mm10-all-gene.txt"
# )

# make.anch(outfile = "outs/coolpup/500bp/win1000_anchors3.bedpe",
#           anch1.path = "outs/coolpup/500bp/win1000_anchors1.bedpe",
#           peaks.path = "outs/great/Double_KO_vs_F_F/basal/proximal/20241126-public-4.0.4-xCT3F1-mm10-all-gene.txt"
# )

write.table(anch2,
              file=paste0(outpath,"/win",window,"_anchors2.bedpe"),
              quote=FALSE,
              sep="\t",
              col.names = FALSE,
              row.names = FALSE
)


#### anchor 3 ✅

In [ ]:
%%R

#setwd("/media/lucio/external.wk/bioinfo/wkdir/Lara/Lara_multiomic_analysis")
#setwd("/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis")

make.anch<-function(outfile,
                    anch1.path,
                    peaks.path){

  anch1<-read.table(anch1.path,
                    sep="\t",
                    header=FALSE
  )

  proximal.peaks<-read.table(peaks.path,
                             sep="\t",
                             header=FALSE)

  anch3_unsort<-merge(anch1,
                      proximal.peaks,
                      by.x=8,
                      by.y=1)

  #anch3<-cbind(anch3, anch3$V10)
  anch3_unsort<-anch3_unsort[,c(-1,-8,-9)]
  #anch3<-anch3[,c(1,2,3,5,6,7,4,8)]

  colnames(anch3_unsort)<-c("chrA","startA","endA","chrB","startB","endB")

  anch3<-anch3_unsort

  for (i in 1:nrow(anch3_unsort)){

    meanA <- round((anch3_unsort$startA[i] + ((anch3_unsort$endA[i] - anch3_unsort$startA [i] )/2)),0)
    meanB <- round((anch3_unsort$startB[i] + ((anch3_unsort$endB[i] - anch3_unsort$startB [i] )/2)),0)

    if(meanB<meanA){
      anch3[i,1]<-anch3_unsort[i,4]
      anch3[i,2]<-anch3_unsort[i,5]
      anch3[i,3]<-anch3_unsort[i,6]
      anch3[i,4]<-anch3_unsort[i,1]
      anch3[i,5]<-anch3_unsort[i,2]
      anch3[i,6]<-anch3_unsort[i,3]
    }


  }

  write.table(anch3,
              outfile,
              quote=FALSE,
              sep="\t",
              col.names = FALSE,
              row.names = FALSE
  )


}


make.anch(outfile = "outs/coolpup/500bp/win500_anchors3.bedpe",
          anch1.path = "outs/coolpup/500bp/win500_anchors1.bedpe",
          peaks.path = "outs/great/Double_KO_vs_F_F/basal/proximal/20241126-public-4.0.4-xCT3F1-mm10-all-gene.txt"
          )

make.anch(outfile = "outs/coolpup/500bp/win1_anchors3.bedpe",
          anch1.path = "outs/coolpup/500bp/win1_anchors1.bedpe",
          peaks.path = "outs/great/Double_KO_vs_F_F/basal/proximal/20241126-public-4.0.4-xCT3F1-mm10-all-gene.txt"
)

make.anch(outfile = "outs/coolpup/500bp/win1000_anchors3.bedpe",
          anch1.path = "outs/coolpup/500bp/win1000_anchors1.bedpe",
          peaks.path = "outs/great/Double_KO_vs_F_F/basal/proximal/20241126-public-4.0.4-xCT3F1-mm10-all-gene.txt"
)


#head outs/Lara_multiomic_analysis/out/outs/coolpup/500bp/win500_anchors3.bedpe

### Make pileup plot

#### set environment

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis

mkdir ./out/coolpoppy_pileup
mkdir ./in/2024_05_Lara_microC/
mkdir ./in/2024_05_Lara_microC/cooler
mkdir ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/Backup1/Lucio/Analysis/Lara/2024_05_Lara_microC/nfout/KO_day0_A/cooler/25000/KO_day0_A_LP3.cool ./in/2024_05_Lara_microC/cooler/

cp /media/lucio/Backup1/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/in/KO_day0_A_LP1.Dd.pairs.gz ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/Backup1/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/KO_day0_A_LP1.Dd.pairs.gz ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/pairtools_merge/5k_aLp_KO_day0.Dd.cool ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/pairtools_merge/15k_aLp_KO_day0.Dd.cool ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/pairtools_merge/15k_aLp_WT_day0.Dd.cool ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/pairtools_merge/5k_aLp_WT_day0.Dd.cool ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/pairtools_merge/5k_aLp_WT_day4.Dd.cool ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/pairtools_merge/15k_aLp_WT_day4.Dd.cool ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/pairtools_merge/5k_aLp_KO_day4.Dd.cool ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/pairtools_merge/15k_aLp_KO_day4.Dd.cool ./in/2024_10_Lara_microC_downstream/

cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_5kb_aLp_WT_day0.Dd.cool ./in/2024_10_Lara_microC_downstream/
cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_5kb_aLp_KO_day0.Dd.cool ./in/2024_10_Lara_microC_downstream/
cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_5kb_aLp_WT_day4.Dd.cool ./in/2024_10_Lara_microC_downstream/

cooler dump --balanced -c chrom1,start1,end1,chrom2,start2,end2,count,balanced /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_15k_aLp_KO_day4.Dd.cool | head
cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_15k_aLp_KO_day4.Dd.cool ./in/2024_10_Lara_microC_downstream/

cooler dump --balanced -c chrom1,start1,end1,chrom2,start2,end2,count,balanced /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_15k_aLp_WT_day4.Dd.cool | head
cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_15k_aLp_WT_day4.Dd.cool ./in/2024_10_Lara_microC_downstream/

cooler dump --balanced -c chrom1,start1,end1,chrom2,start2,end2,count,balanced /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_15k_aLp_WT_day0.Dd.cool | head
cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_15k_aLp_WT_day0.Dd.cool ./in/2024_10_Lara_microC_downstream/

cooler dump --balanced -c chrom1,start1,end1,chrom2,start2,end2,count,balanced /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_15k_aLp_KO_day0.Dd.cool | head
cp /media/lucio/easystore/Lucio/Analysis/Lara/2024_10_Lara_microC_downstream/out/cooler_cload_balance/balanced_15k_aLp_KO_day0.Dd.cool ./in/2024_10_Lara_microC_downstream/



#### execute analysis

##### TODEL anchor2 pileup

##### 5kb

###### D0

In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

# day 0

#WT

#wt
coolpup.py --features_format "bedpe" -o 500bp/D0/5kb/win500_anch2_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_WT_day0.Dd.cool 500bp/win500_anchors2.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 2 \
    --input_pups 500bp/D0/5kb/win500_anch2_aLp_WT_day0.Dd.clpy \
    --output 500bp/D0/5kb/win500_anch2_aLp_WT_day0.Dd.png

#ko
coolpup.py --features_format "bedpe" -o 500bp/D0/5kb/win500_anch2_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_KO_day0.Dd.cool 500bp/win500_anchors2.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 2 \
    --input_pups 500bp/D0/5kb/win500_anch2_aLp_KO_day0.Dd.clpy \
    --output 500bp/D0/5kb/win500_anch2_aLp_KO_day0.Dd.png




###### D4

In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

# day 0

#WT

#wt
coolpup.py --features_format "bedpe" -o 500bp/D4/5kb/win500_anch2_aLp_WT_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_WT_day0.Dd.cool 500bp/win500_anchors2.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 2 \
    --input_pups 500bp/D4/5kb/win500_anch2_aLp_WT_day4.Dd.clpy \
    --output 500bp/D4/5kb/win500_anch2_aLp_WT_day4.Dd.png

#ko
coolpup.py --features_format "bedpe" -o 500bp/D4/5kb/win500_anch2_aLp_KO_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_KO_day4.Dd.cool 500bp/win500_anchors2.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 2 \
    --input_pups 500bp/D4/5kb/win500_anch2_aLp_KO_day4.Dd.clpy \
    --output 500bp/D4/5kb/win500_anch2_aLp_KO_day4.Dd.png



#### anchor3 pileup : bedpe

#####  15kb

###### D0 - WT

In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

# day 0

#WT

#win1
# coolpup.py --features_format "bedpe" -o 500bp/D0/15kb/win1_anch3_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/balanced_15kb_aLp_WT_day0.Dd.cool 500bp/win1_anchors3.bedpe

# plotpup.py \
#     --plot_ticks \
#     --not_symmetric \
#     --vmin 1 --vmax 1.3 \
#     --center 3 \
#     --input_pups 500bp/D0/15kb/win1_anch3_aLp_WT_day0.Dd.clpy \
#     --output 500bp/D0/15kb/win1_anch3_aLp_WT_day0.Dd.png

#win500
coolpup.py --features_format "bedpe" -o 500bp/D0/15kb/win500_anch3_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/balanced_15k_aLp_WT_day0.Dd.cool 500bp/win500_anchors3.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 3 \
    --input_pups 500bp/D0/15kb/win500_anch3_aLp_WT_day0.Dd.clpy \
    --output 500bp/D0/15kb/win500_anch3_aLp_WT_day0.Dd.png

#win1000
# coolpup.py --features_format "bedpe" -o 500bp/D0/15kb/win1000_anch3_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day0.Dd.cool 500bp/win1000_anchors3.bedpe

# plotpup.py \
#     --plot_ticks \
#     --not_symmetric \
#     --vmin 1 --vmax 1.3 \
#     --center 3 \
#     --input_pups 500bp/D0/15kb/win1000_anch3_aLp_WT_day0.Dd.clpy \
#     --output 500bp/D0/15kb/win1000_anch3_aLp_WT_day0.Dd.png


###### D0 - KO

In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

# day 0

#win1
# coolpup.py --features_format "bedpe" -o 500bp/D0/15kb/win1_anch3_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/balanced_15k_aLp_KO_day0.Dd.cool 500bp/win1_anchors3.bedpe

# plotpup.py \
#     --plot_ticks \
#     --not_symmetric \
#     --vmin 1 --vmax 1.3 \
#     --center 3 \
#     --input_pups 500bp/D0/15kb/win1_anch3_aLp_KO_day0.Dd.clpy \
#     --output 500bp/D0/15kb/win1_anch3_aLp_KO_day0.Dd.png

#win500
coolpup.py --features_format "bedpe" -o 500bp/D0/15kb/win500_anch3_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/balanced_15k_aLp_KO_day0.Dd.cool 500bp/win500_anchors3.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 3 \
    --input_pups 500bp/D0/15kb/win500_anch3_aLp_KO_day0.Dd.clpy \
    --output 500bp/D0/15kb/win500_anch3_aLp_KO_day0.Dd.png

#win1000
# coolpup.py --features_format "bedpe" -o 500bp/D0/15kb/win1000_anch3_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_KO_day0.Dd.cool 500bp/win1000_anchors3.bedpe

# plotpup.py \
#     --plot_ticks \
#     --not_symmetric \
#     --vmin 1 --vmax 1.3 \
#     --center 3 \
#     --input_pups 500bp/D0/15kb/win1000_anch3_aLp_KO_day0.Dd.clpy \
#     --output 500bp/D0/15kb/win1000_anch3_aLp_KO_day0.Dd.png


###### D4 - WT

In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

# day 0

#WT

#win500
coolpup.py --features_format "bedpe" -o 500bp/D4/15kb/win500_anch3_aLp_WT_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/balanced_15k_aLp_WT_day4.Dd.cool 500bp/win500_anchors3.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 2 \
    --input_pups 500bp/D4/15kb/win500_anch3_aLp_WT_day4.Dd.clpy \
    --output 500bp/D4/15kb/win500_anch3_aLp_WT_day4.Dd.png


###### D4 - KO

In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

# day 4

#WT

#win500
coolpup.py --features_format "bedpe" -o 500bp/D4/15kb/win500_anch3_aLp_KO_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/balanced_15k_aLp_KO_day4.Dd.cool 500bp/win500_anchors3.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 2 \
    --input_pups 500bp/D4/15kb/win500_anch3_aLp_KO_day4.Dd.clpy \
    --output 500bp/D4/15kb/win500_anch3_aLp_KO_day4.Dd.png


##### 5kb

###### D0 WT

In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

# day 0

#WT

#win500
coolpup.py --features_format "bedpe" -o 500bp/D0/5kb/win500_anch3_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/balanced_5kb_aLp_WT_day0.Dd.cool 500bp/win500_anchors3.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmax 1.4 \
    --vmin 0.9 \
    --center 5 \
    --input_pups 500bp/D0/5kb/win500_anch3_aLp_WT_day0.Dd.clpy \
    --output 500bp/D0/5kb/win500_anch3_aLp_WT_day0.Dd.png

#make aggregate file to load in higlass





###### D0 KO

In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

# day 0

#WT

#win500
coolpup.py --features_format "bedpe" -o 500bp/D0/5kb/win500_anch3_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/balanced_5kb_aLp_KO_day0.Dd.cool 500bp/win500_anchors3.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmax 1.4 \
    --vmin 0.9 \
    --center 5 \
    --input_pups 500bp/D0/5kb/win500_anch3_aLp_KO_day0.Dd.clpy \
    --output 500bp/D0/5kb/win500_anch3_aLp_KO_day0.Dd.png

#test TODEL

plotpup.py \
  --scale linear \
  --vmin 0.9 --vmax 1.4 \
  --plot_ticks --not_symmetric --center 5 \
  --input_pups outs/Lara_multiomic_analysis/outs/coolpup/500bp/D0/5kb/win500_anch3_aLp_KO_day0.Dd.clpy \
  --output outs/Lara_multiomic_analysis/outs/coolpup/500bp/D0/5kb/TODEL_win500_anch3_aLp_KO_day0.linear.png





###### D4 WT

In [ ]:
sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

# day 4

#WT

#win500
coolpup.py --features_format "bedpe" -o 500bp/D4/5kb/win500_anch3_aLp_WT_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/balanced_5kb_aLp_WT_day4.Dd.cool 500bp/win500_anchors3.bedpe

# plotpup.py \
#     --plot_ticks \
#     --not_symmetric \
#     --vmin 1 --vmax 1.3 \
#     --center 5 \
#     --input_pups 500bp/D4/5kb/win500_anch3_aLp_WT_day4.Dd.clpy \
#     --output 500bp/D4/5kb/win500_anch3_aLp_WT_day4.Dd.png


plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 5 \
    --input_pups 500bp/D4/5kb/win500_anch3_aLp_WT_day4.Dd.clpy \
    --output 500bp/D4/5kb/win500_anch3_aLp_WT_day4.Dd.png

    # --vmax 1.5 \
    # --vmin 0.9 \
#--vmin 1 --vmax 1.3 \


###### D4 KO

In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

# day 4

#WT

#win500
coolpup.py --features_format "bedpe" -o 500bp/D4/5kb/win500_anch3_aLp_KO_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/balanced_5kb_aLp_KO_day4.Dd.cool 500bp/win500_anchors3.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 5 \
    --input_pups 500bp/D4/5kb/win500_anch3_aLp_KO_day4.Dd.clpy \
    --output 500bp/D4/5kb/win500_anch3_aLp_KO_day4.Dd.png



#### higlass session 

In [ ]:
%%script bash

#prepare anchors files

sh git/Lara_MLL2/bin/HiGlass_anchor3_visualization.sh

sh git/Lara_MLL2/bin/HiGlass_check_target_genes.sh

#### MAplot pileup

In [ ]:
%%script bash
#bash git/Lara_MLL2/bin/MAplot.sh

git/Lara_MLL2/bin/MAplot_pileup_microC.sh

#### HiCAggR pileup

In [ ]:
%%script bash

source("git/Lara_MLL2/bin/HicAggR_microC_analysis.R")

#### TODEL

In [ ]:
%%script bash

#

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/coolpup

#cooltools expected-cis --view hg38_arms.bed -p 2 -o test_expected_cis.tsv test.mcool::resolutions/10000

#test_expected_cis.tsv

#coolpup.py --by_distance --features_format "bedpe" ../../in/2024_10_Lara_microC_downstream/5kb_aLp_KO_day0.Dd.cool ../anchors1.bedpe

#plotpup.py \
#    --input_pups 5kb_aLp_KO_day0.Dd.cool-5.0K_over_anchors1_10-shifts.clpy \
#    --output bydistance.png

#============================================================
#===========coolpup KO
#============================================================

# anchor1 5kb

coolpup.py --flank 55000 --features_format "bedpe" ../../in/2024_10_Lara_microC_downstream/5kb_aLp_KO_day0.Dd.cool ../anchors1.bedpe

# anchor1 15kb

coolpup.py --features_format "bedpe" ../../in/2024_10_Lara_microC_downstream/15kb_aLp_KO_day0.Dd.cool ../anchors1.bedpe


# anchor2 5kb

coolpup.py --features_format "bedpe" -o anch2_5kb_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_KO_day0.Dd.cool ../anchors2.bedpe

# anchor2 15kb

coolpup.py --flank 200000 --features_format "bedpe" -o anch2_15kb_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_KO_day0.Dd.cool ../anchors2.bedpe

# anchor2 sorted 15kb

coolpup.py --flank 200000 --features_format "bedpe" -o anch2sorted_15kb_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_KO_day0.Dd.cool ../anchors2_sorted.bedpe

plotpup.py --plot_ticks --not_symmetric --vmin 1 --vmax 1.5 --center 1 --input_pups anch2sorted_15kb_aLp_KO_day0.Dd.clpy --output anch2sort_15kb_aLp_KO_day0_stdplot.png

# anchor2 sorted 15kb filter 500

coolpup.py --flank 100000 --features_format "bedpe" -o 500_anch2sorted_15kb_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_KO_day0.Dd.cool ../500_anchors2_sorted.bedpe

plotpup.py --plot_ticks --not_symmetric --vmin 1 --vmax 1.5 --center 1 --input_pups 500_anch2sorted_15kb_aLp_KO_day0.Dd.clpy --output 500_anch2sort_15kb_aLp_KO_day0_stdplot.png

# anchor3 5kb

coolpup.py --flank 55000 --features_format "bedpe" -o anch3_5kb_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_KO_day0.Dd.cool ../anchors3.bedpe


# anchor3 15kb

coolpup.py --features_format "bedpe" -o anch3_15kb_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_KO_day0.Dd.cool ../anchors3.bedpe



# anchor 3 sorted disfilter 500bp

coolpup.py --features_format "bedpe" -o anch3sort_15kb_aLp_KO_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_KO_day0.Dd.cool ../500_anchors3_sorted.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 1 \
    --input_pups anch3sort_15kb_aLp_KO_day0.Dd.clpy \
    --output anch3sort_15kb_aLp_KO_day0.Dd.png

#=============
# KO D4
#=============

# anchor2 15kb

coolpup.py --flank 200000 --features_format "bedpe" -o anch2_15kb_aLp_KO_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_KO_day4.Dd.cool ../anchors2.bedpe

# anchor2 sorted 15kb

coolpup.py --flank 200000 --features_format "bedpe" -o anch2sorted_15kb_aLp_KO_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_KO_day4.Dd.cool ../anchors2_sorted.bedpe

plotpup.py --plot_ticks --not_symmetric --vmin 1 --vmax 1.5 --center 1 --input_pups anch2sorted_15kb_aLp_KO_day4.Dd.clpy --output anch2sort_15kb_aLp_KO_day4_stdplot.png

# anchor2 sorted 15kb 500bp dist cutoff

coolpup.py --flank 100000 --features_format "bedpe" -o 500_anch2sorted_15kb_aLp_KO_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_KO_day4.Dd.cool ../500_anchors2_sorted.bedpe

plotpup.py --plot_ticks --not_symmetric --vmin 1 --vmax 1.5 --center 1 --input_pups 500_anch2sorted_15kb_aLp_KO_day4.Dd.clpy --output 500_anch2sort_15kb_aLp_KO_day4_stdplot.png

# anchor2 d4 5kb

coolpup.py --flank 55000 --features_format "bedpe" -o anch2d4_5kb_aLp_KO_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_KO_day4.Dd.cool ../anchors2_d4.bedpe

# anchor3 5kb

coolpup.py --flank 55000 --features_format "bedpe" -o anch3_5kb_aLp_KO_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_KO_day4.Dd.cool ../anchors3.bedpe


#======================================================================
#======coolpup WT
#======================================================================


#anchor1 5kb

coolpup.py --store_stripes --flank 55000 --features_format "bedpe" ../../in/2024_10_Lara_microC_downstream/5kb_aLp_WT_day0.Dd.cool ../anchors1.bedpe

#anchors1 15kb

coolpup.py --store_stripes --features_format "bedpe" ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day0.Dd.cool ../anchors1.bedpe


#anchor2 5kb

coolpup.py --features_format "bedpe" -o anch2_5kb_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_WT_day0.Dd.cool ../anchors2.bedpe

#anchor2 15kb

coolpup.py --flank 200000 --features_format "bedpe" -o anch2_15kb_aLp_WT_day0.Dd.clpy --log 'INFO' ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day0.Dd.cool ../anchors2.bedpe

#anchor2 sorted 15kb
coolpup.py --flank 200000 --features_format "bedpe" -o anch2sorted_15kb_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day0.Dd.cool ../anchors2_sorted.bedpe

plotpup.py --plot_ticks --not_symmetric --vmin 1 --vmax 1.5 --center 1 --input_pups anch2sorted_15kb_aLp_WT_day0.Dd.clpy --output anch2sort_15kb_aLp_WT_day0_stdplot.png

#anchor2 sorted 15kb 500bp dist
coolpup.py --flank 100000 --features_format "bedpe" -o 500_anch2sorted_15kb_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day0.Dd.cool ../500_anchors2_sorted.bedpe

plotpup.py --plot_ticks --not_symmetric --vmin 1 --vmax 1.5 --center 1 --input_pups 500_anch2sorted_15kb_aLp_WT_day0.Dd.clpy --output 500_anch2sort_15kb_aLp_WT_day0_stdplot.png

#anchor3 5kb

coolpup.py --store_stripes --flank 55000 --features_format "bedpe" -o anch3_5kb_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_WT_day0.Dd.cool ../anchors3.bedpe

#anchor3 15kb

coolpup.py --store_stripes --features_format "bedpe" -o anch3_15kb_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day0.Dd.cool ../anchors3.bedpe

# anchor 3 sorted disfilter 500bp

coolpup.py --features_format "bedpe" -o anch3sort_15kb_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day0.Dd.cool ../500_anchors3_sorted.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 1 \
    --input_pups anch3sort_15kb_aLp_WT_day0.Dd.clpy \
    --output anch3sort_15kb_aLp_WT_day0.Dd.png

# anchor 3 bed file , by window 500bp

coolpup.py --features_format "bedpe" -o anch3sort_15kb_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day0.Dd.cool ../500_anchors3_sorted.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 1 \
    --input_pups anch3sort_15kb_aLp_WT_day0.Dd.clpy \
    --output anch3sort_15kb_aLp_WT_day0.Dd.png




#==============
# WT D4
#==============

#anchor2 D4 5kb
coolpup.py --flank 55000 --features_format "bedpe" -o anch2d4_5kb_aLp_WT_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_WT_day4.Dd.cool ../anchors2_d4.bedpe

#anchor2 D4 15kb

coolpup.py --flank 200000 --features_format "bedpe" -o anch2_15kb_aLp_WT_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day4.Dd.cool ../anchors2.bedpe

#anchor2 sorted D4 15kb

coolpup.py --flank 200000 --features_format "bedpe" -o anch2sorted_15kb_aLp_WT_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day4.Dd.cool ../anchors2_sorted.bedpe

plotpup.py --plot_ticks --not_symmetric --vmin 1 --vmax 1.5 --center 1 --input_pups anch2sorted_15kb_aLp_WT_day4.Dd.clpy --output anch2sort_15kb_aLp_WT_day4_stdplot.png

#anchor2 sorted D4 15kb 500bp dist

coolpup.py --flank 100000 --features_format "bedpe" -o 500_anch2sorted_15kb_aLp_WT_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day4.Dd.cool ../500_anchors2_sorted.bedpe

plotpup.py --plot_ticks --not_symmetric --vmin 1 --vmax 1.5 --center 1 --input_pups 500_anch2sorted_15kb_aLp_WT_day4.Dd.clpy --output 500_anch2sort_15kb_aLp_WT_day4_stdplot.png



#anchor3 D4 5kb

coolpup.py --flank 55000 --features_format "bedpe" -o anch3_5kb_aLp_WT_day4.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_WT_day4.Dd.cool ../anchors3.bedpe




#==================
# by distance pileup
#==================

#coolpup.py --flank 200000 --by_distance --features_format "bed" -o distance_anch2_15kb_aLp_WT_day0.Dd.clpy --log 'INFO' ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day0.Dd.cool ../anchors2.bedpe

coolpup.py --flank 200000 --by_distance --features_format "bed" -o distance_anch2_15kb_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/15kb_aLp_WT_day0.Dd.cool ../anchors2.bed

#=================
# draw plots
#=================


plotpup.py \
    --not_symmetric \
    --plot_ticks \
    --input_pups 5kb_aLp_WT_day0.Dd.cool-5.0K_over_anchors1_10-shifts.clpy \
    --vmin 0.98 --vmax 1.2 \
    --output WT_stdplot.png

plotpup.py \
    --not_symmetric \
    --plot_ticks \
    --vmin 0.98 --vmax 1.2 \
    --input_pups 5kb_aLp_KO_day0.Dd.cool-5.0K_over_anchors1_10-shifts.clpy \
    --output KO_stdplot.png

plotpup.py \
    --not_symmetric \
    --vmin 0.98 --vmax 1.2 \
    --plot_ticks \
    --input_pups 15kb_aLp_WT_day0.Dd.cool-15.0K_over_anchors1_10-shifts.clpy \
    --output 15kb_WT_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.2 \
    --input_pups 15kb_aLp_KO_day0.Dd.cool-15.0K_over_anchors1_10-shifts.clpy \
    --output 15kb_KO_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.5 \
    --input_pups anch2_5kb_aLp_KO_day0.Dd.clpy \
    --output anch2_5kb_aLp_KO_day0_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.5 \
    --input_pups anch2_15kb_aLp_KO_day0.Dd.clpy \
    --output anch2_15kb_aLp_KO_day0_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.5 \
    --input_pups anch2_5kb_aLp_WT_day0.Dd.clpy \
    --output anch2_5kb_aLp_WT_day0_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.5 \
    --input_pups anch2_15kb_aLp_WT_day0.Dd.clpy \
    --output anch2_15kb_aLp_WT_day0_stdplot.png

#==========================
#==========================


plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.4 \
    --input_pups anch3_5kb_aLp_KO_day0.Dd.clpy \
    --output anch3_5kb_aLp_KO_day0_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.2 \
    --input_pups anch3_15kb_aLp_KO_day0.Dd.clpy \
    --output anch3_15kb_aLp_KO_day0_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.4 \
    --input_pups anch3_5kb_aLp_WT_day0.Dd.clpy \
    --output anch3_5kb_aLp_WT_day0_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.2 \
    --input_pups anch3_15kb_aLp_WT_day0.Dd.clpy \
    --output anch3_15kb_aLp_WT_day0_stdplot.png


#========================
# D4
#========================

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 1 \
    --input_pups anch3_5kb_aLp_KO_day4.Dd.clpy \
    --output anch3_5kb_aLp_KO_day4_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 1.3 \
    --center 1 \
    --input_pups anch3_5kb_aLp_WT_day4.Dd.clpy \
    --output anch3_5kb_aLp_WT_day4_stdplot.png

# plotpup.py \
#     --plot_ticks \
#     --not_symmetric \
#     --vmin 0.98 --vmax 1.2 \
#     --input_pups anch3_15kb_aLp_WT_day4.Dd.clpy \
#     --output anch3_15kb_aLp_WT_day4_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.3 \
    --input_pups anch2_15kb_aLp_WT_day4.Dd.clpy \
    --output anch2_15kb_aLp_WT_day4_stdplot.png


plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.3 \
    --input_pups anch2_15kb_aLp_KO_day4.Dd.clpy \
    --output anch2_15kb_aLp_KO_day4_stdplot.png


#sorted anchors

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.3 \
    --input_pups anch2sorted_15kb_aLp_WT_day4.Dd.clpy \
    --output anch2sorted_15kb_aLp_WT_day4_stdplot.png

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 0.98 --vmax 1.3 \
    --input_pups anch2sorted_15kb_aLp_KO_day4.Dd.clpy \
    --output anch2sorted_15kb_aLp_KO_day4_stdplot.png



#=====expr genes







#### pileup broad genes

In [ ]:
%%script bash

#make bedpe of down genes

#outs/downGenesHeatmap/D0dkoVSwt.broad.downgenes.bed 

#ref_regions="outs/downGenesHeatmap/D0dkoVSwt.broad.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.narrow.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.negative.downgenes.bed"


In [ ]:
%%script bash

sudo docker run -v /mnt/datawk1/analysis:/mnt/datawk1/analysis -it lucidif/coolpoppy:0.0.1

cd /media/lucio/easystore/Lucio/Analysis/Lara/Lara_multiomic_analysis/outs/coolpup/broad_genes_coolpup

#coolpup.py --by_distance 0 50000 1000000 2000000 4000000 8000000 --features_format "bed" -o WTd0_D0dkoVSwt.broad.downgenes.clpy  ../../../in/2024_10_Lara_microC_downstream/balanced_5kb_aLp_WT_day0.Dd.cool ../../downGenesHeatmap/D0dkoVSwt.negative.downgenes.bed

coolpup.py --flank 100000 --mindist 0 --maxdist 2000000 --features_format "bed" -o WTd0_broad_all_genes.clpy  ../../../in/2024_10_Lara_microC_downstream/balanced_15k_aLp_WT_day0.Dd.cool ../../../outs/broad_all_genes.bed

# coolpup.py --by_distance 0 2000000 --flank 50000 --features_format "bed" -o WTd0_broad_all_genes.clpy  ../../../in/2024_10_Lara_microC_downstream/balanced_5kb_aLp_WT_day0.Dd.cool ../../../outs/broad_all_genes.bed


#coolpup.py --by_distance --features_format "bed" -o WTd0_D0dkoVSwt.broad.downgenes.clpy  ../../../in/2024_10_Lara_microC_downstream/balanced_5kb_aLp_WT_day0.Dd.cool ../../downGenesHeatmap/D0dkoVSwt.negative.downgenes.bed

#coolpup.py --by_distance 0 50000 1000000 2000000 4000000 8000000 --features_format "bed" -o KOd0_D0dkoVSwt.broad.downgenes.clpy ../../../in/2024_10_Lara_microC_downstream/balanced_5kb_aLp_KO_day0.Dd.cool ../../downGenesHeatmap/D0dkoVSwt.negative.downgenes.bed

#wt
#coolpup.py --features_format "bedpe" -o 500bp/D0/5kb/win500_anch2_aLp_WT_day0.Dd.clpy ../../in/2024_10_Lara_microC_downstream/5kb_aLp_WT_day0.Dd.cool #500bp/win500_anchors2.bedpe

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --center 2 \
    --input_pups WTd0_broad_all_genes.clpy \
    --output WTd0_broad_all_genes.png


plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --vmin 1 --vmax 3 \
    --center 2 \
    --input_pups WTd0_broad_all_genes.clpy \
    --output WTd0_broad_all_genes.png

#=================================
# generat bed6 format
#=================================

awk '{print $0"\t1"}' ../../../outs/broad_all_genes.bed > ../../../outs/broad_all_genes_score1.bed


awk 'NR==FNR {strand[$5]=$4; next} {print $1,$2,$3,$4,$5,strand[$4]}' \
  /media/lucio/easystore/Lucio/Analysis/Lara/Lara_multiomic_analysis/in/GREATv4/GREATv4.genes.mm10.tsv \
  ../../../outs/broad_all_genes_score1.bed \
  OFS='\t' > ../../../outs/broad_all_genes_with_strand.bed

awk '
BEGIN {miss=0; tot=0; kept=0}
NR==FNR {strand[$5]=$4; next}
{
  tot++
  s=strand[$4]
  if(s=="") {
    miss++
    print $4 > "missing_strand_genes.txt"
    next
  }
  kept++
  print $1,$2,$3,$4,$5,s
}
END {
  printf("Totale geni: %d\n", tot)
  printf("Senza orientamento (rimossi): %d (%.2f%%)\n", miss, (miss/tot)*100)
  printf("Con orientamento (mantenuti): %d (%.2f%%)\n", kept, (kept/tot)*100)
}' \
/media/lucio/easystore/Lucio/Analysis/Lara/Lara_multiomic_analysis/in/GREATv4/GREATv4.genes.mm10.tsv \
../../../outs/broad_all_genes_score1.bed \
OFS='\t' > ../../../outs/broad_all_genes_with_strand_filtered.bed

grep -v "^Totale" ../../../outs/broad_all_genes_with_strand_filtered.bed | \
grep -v "^Senza" | \
grep -v "^Con" > ../../../outs/broad_all_genes_with_strand_filtered_clean.bed

wc -l ../../../outs/broad_all_genes_score1.bed
wc -l ../../../outs/broad_all_genes_with_strand.bed
wc -l ../../../outs/broad_all_genes_with_strand_filtered.bed
cat -T ../../../outs/broad_all_genes_with_strand_filtered.bed
awk 'BEGIN{OFS="\t"} {$1=$1; print}' \
  ../../../outs/broad_all_genes_with_strand_filtered_clean.bed \
  > ../../../outs/broad_all_genes_with_strand_filtered_tabs.bed

head /media/lucio/easystore/Lucio/Analysis/Lara/Lara_multiomic_analysis/in/GREATv4/GREATv4.genes.mm10.tsv
head ../../../outs/broad_all_genes_with_strand.bed

#=========================
# plot by stand
#=========================


coolpup.py --flank 300000 --mindist 0 --maxdist 2000000 --by_strand --rescale -o stranded_WTd0_broad_all_genes.clpy  ../../../in/2024_10_Lara_microC_downstream/balanced_15k_aLp_WT_day0.Dd.cool ../../../outs/broad_all_genes_with_strand_filtered_tabs.bed

plotpup.py \
    --plot_ticks \
    --not_symmetric \
    --center 2 \
    --input_pups stranded_WTd0_broad_all_genes.clpy  \
    --output stranded_WTd0_broad_all_genes.png

#python -c "from coolpuppy import plotpup; plotpup.plot('stranded_WTd0_broad_all_genes.clpy', groupby='orientation', scale='log', cmap='coolwarm', vmax=3, sym=False, savefig='stranded_WTd0_broad_all_genes_orientation.png')"

python3 - <<'EOF'
from coolpuppy.lib import io
from coolpuppy import plotpup

# carica il file .clpy
pups = io.load_pileup_df('stranded_WTd0_broad_all_genes.clpy')

# crea il plot (senza savefig)
fg = plotpup.plot(
    pups,
    rows='orientation',
    score=False,
    cmap='coolwarm',
    scale='log',
    sym=True,
    vmax=3,
    height=3
)

# salva manualmente la figura
fg.savefig('stranded_WTd0_broad_all_genes_orientation.png')
EOF





## line expression

In [ ]:
%% script bash

cd /home/lucio/wkdir/analysis/Lara/TEseq

#docker commit 4e0e816afb17 teseq:0.0.1
mkdir Line_expr


sudo docker run -it -v /mnt/datawk1/references:/mnt/datawk1/references -v /home/lucio/wkdir/data:/home/lucio/wkdir/data -v `pwd`:`pwd` -w `pwd` teseq:0.0.1 /bin/bash

cd Line_expr

conda active

git clone -b stable https://github.com/maxfieldk/TE-Seq.git

cd TE-Seq


cp -r workflow/conf_example conf

cd ../../

mkdir -p srna_rawdata

mkdir genome_files

cd genome_files

wget https://hgdownload.soe.ucsc.edu/goldenPath/mm39/bigZips/mm39.fa.gz

cp /mnt/datawk1/references/annotations/ncbi/GCF_000001635_26_GRCm38_p6/GCF_000001635.26_GRCm38.p6_genomic.gtf.gz ./
#gff3 ?
cp /mnt/datawk1/references/annotations/ncbi/GCF_000001635_26_GRCm38_p6/GCF_000001635.26_GRCm38.p6_genomic.gff.gz ./
#fasta 
cp /mnt/datawk1/references/fasta/UCSC_GRCm38/mm10.fa ./

mv mm10.fa reference.ucsc.fa
gunzip GCF_000001635.26_GRCm38.p6_genomic.gtf.gz; mv GCF_000001635.26_GRCm38.p6_genomic.gtf refseq.gtf
gunzip GCF_000001635.26_GRCm38.p6_genomic.gff.gz; mv GCF_000001635.26_GRCm38.p6_genomic.gff refseq.gff3
gunzip mm39.out.gz; mv mm39.out repeatmasker.ucsc.out

sudo chown lucio:lucio .

sort -k1,1V -k4,4n -k5,5n refseq.gtf > refseq.sorted.gtf
sort -k1,1V -k4,4n -k5,5n refseq.gff3 > refseq.sorted.gff3

wget http://hgdownload.soe.ucsc.edu/goldenPath/mm10/database/chromAlias.txt.gz

cd ..

mv ./genome_files ./Line_expr/

cd ./Line_expr/genome_files

chmod +x ../TE-Seq/resources/programs/chromToUcsc

../TE-Seq/resources/programs/chromToUcsc -i refseq.sorted.gtf -a chromAlias.txt > refseq.sorted.ncbi.gtf
../TE-Seq/resources/programs/chromToUcsc -i refseq.sorted.gff3 -a chromAlias.txt > refseq.sorted.ncbi.gff3

rm refseq.gtf; rm refseq.sorted.gtf; rm refseq.gff3; rm refseq.sorted.gff3

# paper figures

## Figure 1

### environment : elitebook

In [ ]:
%%script bash

#cd /mnt/d/bioinfo/wkdir/Lara/Lara_multiomic_analysis/
cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/

mkdir ./git_elite

cd ./git_elite

git clone git@github.com:lucidif/downstream_multiomic.git

cd ..



### start analysis

In [ ]:
%%script R

setwd("D:/bioinfo/wkdir/Lara/Lara_multiomic_analysis")


#### TODO (maybe) J plotProfile

In [ ]:
%%script bash

plotProfile -m matrix.mat.gz \
     -out ExampleProfile2.png \
     --plotType=fill \ # add color between the x axis and the lines
     --perGroup \ # make one image per BED file instead of per bigWig file
     --colors red yellow blue \
     --plotTitle "Test data profile"


#### J : NMI length of DEG vs ALL developmental genes


environment

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/

cp


run analysis

In [ ]:
%%script R

source("./bin/D0_NMIlen_DEG_vs_ALL_deve_genes.R")


tmp to rem

In [ ]:
%%script R



compose=D0.down

row.names(compose)<-compose$gene_id

dbfile="/mnt/datawk1/references/annotations/biomart/v102/mart_export_GOterms.txt"
geneLengthFile="/mnt/datawk1/references/annotations/biomart/v102/mart_export_GOterms.txt"
termReduction=FALSE
filterBy="padj"

db<-read.delim(dbfile,sep="\t",header=TRUE)

sudb<-db[!duplicated(db),]

db<-cbind(Ensembl.Gene.ID=sudb$Gene.stable.ID,
          GO.Term.Accession=sudb$GO.term.accession)

#gl<-read.delim(geneLengthFile,sep="\t",header=TRUE)

gl<-cbind( Ensembl.Gene.ID=sudb$Gene.stable.ID,
           Transcript.length=sudb$Transcript.length..including.UTRs.and.CDS.

)

gl<-gl[!duplicated(gl),]

compose


selgenes<-intersect(names(compose),gl$Ensembl.Gene.ID




####  Heatmap ChIP-seq of down regulated genes (DEG as bivalent genes)


prima di tutto prendere i proximal peaks e vedere tutti quei picchi vicini ai geni differentilamente espressi, dopo di che utilizzare il subset dei picchi per il tornado plot.




##### prepare environment


In [ ]:
%%script bash


#cd /mnt/datawk1/analysis/Lara/test_chipseq_dowstream

#mkdir ./otherouts
#cd ./otherouts

#mkdir deeptools_heatmaps
#cd deeptools_heatmaps
#mkdir tmp

macs_peaks="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/nfout/broad_0_5/star/mergedLibrary/macs2/broadPeak/Anti-GFP.mLb.mkD.sorted_peaks.threshold_3.broadPeak"
sed 's/ /\t/g' ${macs_peaks} | cut -f 1-6 > ./coordinate.bed

#outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO"
#inpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO"

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/

cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO/input/* ./in/chip_bw/

cp /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/H3K27me3_broad_narrow/downGenes_peaksType_distribution.tsv ./outs/H3K27me3_broad_narrow/


chiptracks_path="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"

panel3_B_files="./tmp/Anti-GFP_average.bw \
./tmp/Anti-Menin_average.bw \
./tmp/Anti-Mll1_average.bw \
./tmp/Mll2_KO_Mll1_average.bw \
./tmp/Anti-RbBp5_average.bw \
./tmp/Double_KO_RbBP5_average.bw \
../parallel_averageBigwig/D0_WT_H3K27ac__average.bw \
../parallel_averageBigwig/D0_WT_RING1B__average.bw \
../parallel_averageBigwig/D0_KO_RING1B__average.bw \
../parallel_averageBigwig/D0_WT_H3K9me3__average.bw \
../parallel_averageBigwig/D0_KO_H3K9me3__average.bw \
"

for i in $panel3_B_files ; do cd ${chiptracks_path} ; echo "cp '$i' /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/in/test_chipseq_downstream/deeptools_heatmaps/"; cp "$i" /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/in/test_chipseq_downstream/deeptools_heatmaps/ ; cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/   ; done

chiptracks_path="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO/input/"

samples="Anti-GFP_average.bw \
Anti-Menin_average.bw \
Anti-Mll1_average.bw \
Mll2_KO_Mll1_average.bw \
Anti-RbBp5_average.bw \
Double_KO_RbBP5_average.bw \
D0_WT_H3K27ac__average.bw \
D0_Mll1KO_H3K27ac__average.bw \
D0_Mll2KO_H3K27ac__average.bw \
D0_DoubleKO_H3K27ac__average.bw \
F_F_K27me3_average.bw \
Mll1-KO_K27me3_average.bw \
FC_FC_K27me3_average.bw \
Double_KO_K27me3_average.bw \
F_F_K4me3_average.bw \
Mll1-KO_K4me3_average.bw \
FC_FC_K4me3_average.bw \
Double_KO_K4me3_average.bw \
F_F_K4me2_average.bw \
Mll1-KO_K4me2_average.bw \
FC_FC_K4me2_average.bw \
Double_KO_K4me2_average.bw \
F_F_K4me1_average.bw \
Mll1-KO_K4me1_average.bw \
FC_FC_K4me1_average.bw \
Double_KO_K4me1_average.bw \
"

for i in $samples ; do cd ${chiptracks_path} ; echo "cp '$i' /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/in/test_chipseq_downstream/deeptools_heatmaps/"; cp "$i" /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/in/test_chipseq_downstream/deeptools_heatmaps/ ; cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/   ; done


outname="alltracks"

outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"

cp $outpath/${outname}_deeptools_matrix.gzip in/test_chipseq_downstream/deeptools_heatmaps/

outname="K4me3_DOWN_Double_KO_vs_F_F"

outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO"

cp ${outpath}/${outname}_deeptools_matrix.gzip in/test_chipseq_downstream/deeptools_heatmaps_filtered_by_diffDoubleKO/

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis





##### make degs genes bed

In [ ]:
%%script R

source('git/Lara_MLL2/bin/D0_make_degs_narrow_broad_bed.R')_


##### make plot

In [ ]:
%%script bash

#all track

#ref_regions="outs/downGenesHeatmap/D0dkoVSwt.downgenes.bed"
ref_regions="outs/downGenesHeatmap/D0dkoVSwt.broad.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.narrow.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.negative.downgenes.bed"

outname="alltracks"

outpath="/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis"
plabels="broad narrow negative"


#/outs/downGenesHeatmap/

samples="./in/test_chipseq_downstream/deeptools_heatmaps/Anti-GFP_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Anti-Menin_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Anti-Mll1_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Mll2_KO_Mll1_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Anti-RbBp5_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Double_KO_RbBP5_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/D0_WT_H3K27ac__average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/D0_Mll1KO_H3K27ac__average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/D0_Mll2KO_H3K27ac__average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/D0_DoubleKO_H3K27ac__average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/F_F_K27me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Mll1-KO_K27me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/FC_FC_K27me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Double_KO_K27me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/F_F_K4me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Mll1-KO_K4me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/FC_FC_K4me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Double_KO_K4me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/F_F_K4me2_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Mll1-KO_K4me2_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/FC_FC_K4me2_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Double_KO_K4me2_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/F_F_K4me1_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Mll1-KO_K4me1_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/FC_FC_K4me1_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Double_KO_K4me1_average.bw\
"

slabels="Anti-GFP_av \
Anti-Menin_av \
Anti-Mll1_av \
Mll2_KO_Mll1_av \
Anti-RbBp5_av \
Double_KO_RbBP5_av \
WT_H3K27ac_av \
Mll1KO_H3K27ac_av \
Mll2KO_H3K27ac_av \
DoubleKO_H3K27ac_av \
F_F_K27me3_av \
Mll1-KO_K27me3_av \
FC_FC_K27me3_av \
Double_KO_K27me3_av \
F_F_K4me3_av \
Mll1-KO_K4me3_av \
FC_FC_K4me3_av \
Double_KO_K4me3_av \
F_F_K4me2_av \
Mll1-KO_K4me2_av \
FC_FC_K4me2_av \
Double_KO_K4me2_av \
F_F_K4me1_av \
Mll1-KO_K4me1_av \
FC_FC_K4me1_av \
Double_KO_K4me1_av \
"

thrsholds="10 10 10 10 15 15 10 10 10 10 10 10 10 10 40 40 40 40 30 30 30 30 10 10 10 10"

sudo docker run -e MPLCONFIGDIR=/tmp/matplotlib \
-v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -R $ref_regions -S $samples --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 --averageTypeBins "median" --outFileName outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName outs/downGenesHeatmap/${outname}_plotHeatmap.pdf --sortUsingSamples 7 --regionsLabel ${plabels} --samplesLabel ${slabels} --outFileNameMatrix outs/downGenesHeatmap/${outname}_plotHeatmap.mat.tab


#sorting test

sudo docker run -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName outs/downGenesHeatmap/${outname}_sorted_by_k4me3_plotHeatmap.pdf --sortUsingSamples 15 --regionsLabel ${plabels} --samplesLabel ${slabels} --outFileNameMatrix outs/downGenesHeatmap/${outname}_sorted_by_k4me3_plotHeatmap.mat.tab

sudo docker run -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName outs/downGenesHeatmap/${outname}_sorted_by_k27me3_plotHeatmap.pdf --sortUsingSamples 11 --regionsLabel ${plabels} --samplesLabel ${slabels} --outFileNameMatrix outs/downGenesHeatmap/${outname}_sorted_by_k27me3_plotHeatmap.mat.tab --sortRegions "ascend"




###### same scale of other heatmaps

In [ ]:
%%script bash

#all track

#ref_regions="outs/downGenesHeatmap/D0dkoVSwt.downgenes.bed"
ref_regions="outs/downGenesHeatmap/D0dkoVSwt.broad.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.narrow.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.negative.downgenes.bed"

outname="alltracks"

outpath="/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis"
plabels="broad narrow negative"


#/outs/downGenesHeatmap/

# samples="./in/chip_bw/Anti-GFP_average.bw \
# ./in/chip_bw/Anti-Menin_average.bw \
# ./in/chip_bw/Anti-Mll1_average.bw \
# ./in/chip_bw/Mll2_KO_Mll1_average.bw \
# ./in/chip_bw/Anti-RbBp5_average.bw \
# ./in/chip_bw/Double_KO_RbBP5_average.bw \
# ./in/chip_bw/WT_H3K27ac_C.bigWig \
# ./in/chip_bw/Mll1KO_H3K27ac_C.bigWig \
# ./in/chip_bw/Mll2KO_H3K27ac_C.bigWig \
# ./in/chip_bw/DoubleKO_H3K27ac_C.bigWig \
# ./in/chip_bw/F_F_K27me3_average.bw \
# ./in/chip_bw/Mll1-KO_K27me3_average.bw \
# ./in/chip_bw/FC_FC_K27me3_average.bw \
# ./in/chip_bw/Double_KO_K27me3_average.bw \
# ./in/chip_bw/F_F_K4me3_average.bw \
# ./in/chip_bw/Mll1-KO_K4me3_average.bw \
# ./in/chip_bw/FC_FC_K4me3_average.bw \
# ./in/chip_bw/Double_KO_K4me3_average.bw \
# ./in/chip_bw/F_F_K4me2_average.bw \
# ./in/chip_bw/Mll1-KO_K4me2_average.bw \
# ./in/chip_bw/FC_FC_K4me2_average.bw \
# ./in/chip_bw/Double_KO_K4me2_average.bw \
# ./in/chip_bw/F_F_K4me1_average.bw \
# ./in/chip_bw/Mll1-KO_K4me1_average.bw \
# ./in/chip_bw/FC_FC_K4me1_average.bw \
# ./in/chip_bw/Double_KO_K4me1_average.bw \
# "

# slabels="Anti-GFP_av \
# Anti-Menin_av \
# Anti-Mll1_av \
# Mll2_KO_Mll1_av \
# Anti-RbBp5_av \
# Double_KO_RbBP5_av \
# WT_H3K27ac_av \
# Mll1KO_H3K27ac_av \
# Mll2KO_H3K27ac_av \
# DoubleKO_H3K27ac_av \
# F_F_K27me3_av \
# Mll1-KO_K27me3_av \
# FC_FC_K27me3_av \
# Double_KO_K27me3_av \
# F_F_K4me3_av \
# Mll1-KO_K4me3_av \
# FC_FC_K4me3_av \
# Double_KO_K4me3_av \
# F_F_K4me2_av \
# Mll1-KO_K4me2_av \
# FC_FC_K4me2_av \
# Double_KO_K4me2_av \
# F_F_K4me1_av \
# Mll1-KO_K4me1_av \
# FC_FC_K4me1_av \
# Double_KO_K4me1_av \
# "

thrsholds="10 20 20 20 20 20 20"

samples="in/test_chipseq_downstream/deeptools_heatmaps/Anti-GFP_average.bw \
./in/chip_bw/F_F_K27me3_average.bw \
./in/chip_bw/Double_KO_K27me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/D0_WT_RING1B__average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/D0_KO_RING1B__average.bw \
./in/chip_bw/F_F_K4me3_average.bw \
./in/chip_bw/Double_KO_K4me3_average.bw \
"

slabels="Anti-GFP_av \
WT_K27me3_av \
KO_K27me3_av \
WT_RING1B_av \
KO_RING1B_av \
WT_K4me3_av \
KO_K4me3_av \
"


sudo docker run -e MPLCONFIGDIR=/tmp/matplotlib -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -R $ref_regions -S $samples --sortUsingSamples 2 --numberOfProcessors 6 --scale 1 --binSize 10 --averageTypeBins "median" --outFileName outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

# sudo docker run -e MPLCONFIGDIR=/tmp/matplotlib -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName outs/downGenesHeatmap/same_scale_${outname}_plotHeatmap.pdf --sortUsingSamples 7 --regionsLabel ${plabels} --samplesLabel ${slabels} --outFileNameMatrix outs/downGenesHeatmap/${outname}_plotHeatmap.mat.tab


#sorting test

sudo docker run -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName outs/downGenesHeatmap/same_scale_${outname}_sorted_by_k4me3_plotHeatmap.pdf --sortUsingSamples 6 --regionsLabel ${plabels} --samplesLabel ${slabels} --outFileNameMatrix outs/downGenesHeatmap/${outname}_sorted_by_k4me3_plotHeatmap.mat.tab

# sudo docker run -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName outs/downGenesHeatmap/same_scale_${outname}_sorted_by_k27me3_plotHeatmap.pdf --sortUsingSamples 11 --regionsLabel ${plabels} --samplesLabel ${slabels} --outFileNameMatrix outs/downGenesHeatmap/${outname}_sorted_by_k27me3_plotHeatmap.mat.tab --sortRegions "ascend"



#=============================================
#==============================================


# panel B

#/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/tmp

panel3_B="in/test_chipseq_downstream/deeptools_heatmaps/Anti-GFP_average.bw \
in/test_chipseq_downstream/deeptools_heatmaps/Anti-Menin_average.bw \
in/test_chipseq_downstream/deeptools_heatmaps/Anti-Mll1_average.bw \
in/test_chipseq_downstream/deeptools_heatmaps/Mll2_KO_Mll1_average.bw \
in/test_chipseq_downstream/deeptools_heatmaps/Anti-RbBp5_average.bw \
in/test_chipseq_downstream/deeptools_heatmaps/Double_KO_RbBP5_average.bw \
in/test_chipseq_downstream/deeptools_heatmaps/D0_WT_H3K27ac__average.bw \
./in/chip_bw/F_F_K27me3_average.bw \
./in/chip_bw/F_F_K4me3_average.bw \
in/test_chipseq_downstream/deeptools_heatmaps/D0_WT_RING1B__average.bw \
in/test_chipseq_downstream/deeptools_heatmaps/D0_KO_RING1B__average.bw \
in/test_chipseq_downstream/deeptools_heatmaps/D0_WT_H3K9me3__average.bw \
in/test_chipseq_downstream/deeptools_heatmaps/D0_KO_H3K9me3__average.bw \
"


# thrsholds_B="10 10 10 10 15 15 10 25 40 80 80 3 3"
thrsholds_B="10 10 10 10 15 15 10 20 20 20 20 3 3"

outname_B="RING1B_tracks"
slabels_B="Anti-GFP_av Anti-Menin_av Anti-Mll1_av Mll2_KO_Mll1_av Anti-RbBp5_av Double_KO_RbBP5_av \
WT_H3K27ac_av \
WT_K27me3_av \
WT_K4me3_av \
WT_RING1B_av \
KO_RING1B_av \
WT_H3K9me3_av \
KO_H3K9me3_av \
"

sudo docker run -e MPLCONFIGDIR=/tmp/matplotlib \
-v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -R $ref_regions -S $panel3_B --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 \
--binSize 10 --averageTypeBins "median" --outFileName outs/downGenesHeatmap/${outname_B}_deeptools_matrix.gzip \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -e MPLCONFIGDIR=/tmp/matplotlib -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname_B}_deeptools_matrix.gzip --zMax ${thrsholds_B} --outFileName outs/downGenesHeatmap/same_scale_${outname_B}_plotHeatmap.pdf --sortUsingSamples 7 --regionsLabel ${plabels} --samplesLabel ${slabels_B} --outFileNameMatrix outs/downGenesHeatmap/${outname_B}_plotHeatmap.mat.tab


#sorting test

sudo docker run -e MPLCONFIGDIR=/tmp/matplotlib -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname_B}_deeptools_matrix.gzip --zMax ${thrsholds_B} --outFileName outs/downGenesHeatmap/same_scale_${outname_B}_sorted_by_k4me3_plotHeatmap.pdf --sortUsingSamples 9 --regionsLabel ${plabels} --samplesLabel ${slabels_B} --outFileNameMatrix outs/downGenesHeatmap/${outname_B}_sorted_by_k4me3_plotHeatmap.mat.tab

sudo docker run -e MPLCONFIGDIR=/tmp/matplotlib -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname_B}_deeptools_matrix.gzip --zMax ${thrsholds_B} --outFileName outs/downGenesHeatmap/same_scale_${outname_B}_sorted_by_k27me3_plotHeatmap.pdf --sortUsingSamples 8 --regionsLabel ${plabels} --samplesLabel ${slabels_B} --outFileNameMatrix outs/downGenesHeatmap/${outname_B}_sorted_by_k27me3_plotHeatmap.mat.tab --sortRegions "ascend"



###### same scale of other heatmaps test values

In [ ]:
%%script bash

#all track

#ref_regions="outs/downGenesHeatmap/D0dkoVSwt.downgenes.bed"
ref_regions="outs/downGenesHeatmap/D0dkoVSwt.broad.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.narrow.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.negative.downgenes.bed"


slabels="Anti-GFP_av \
Anti-Menin_av \
Anti-Mll1_av \
Mll2_KO_Mll1_av \
Anti-RbBp5_av \
Double_KO_RbBP5_av \
WT_H3K27ac_av \
Mll1KO_H3K27ac_av \
Mll2KO_H3K27ac_av \
DoubleKO_H3K27ac_av \
F_F_K27me3_av \
Mll1-KO_K27me3_av \
FC_FC_K27me3_av \
Double_KO_K27me3_av \
F_F_K4me3_av \
Mll1-KO_K4me3_av \
FC_FC_K4me3_av \
Double_KO_K4me3_av \
F_F_K4me2_av \
Mll1-KO_K4me2_av \
FC_FC_K4me2_av \
Double_KO_K4me2_av \
F_F_K4me1_av \
Mll1-KO_K4me1_av \
FC_FC_K4me1_av \
Double_KO_K4me1_av \
"

thrsholds="10 10 10 10 15 15 10 10 10 10 10 10 10 10 40 40 40 40 30 30 30 30 10 10 10 10"

#sorting k4me3

#tss
#ref_regions="outs/downGenesHeatmap/D0dkoVSwt.broad.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.narrow.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.negative.downgenes.bed"
outname="alltracks"
outpath="/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis"
plabels="broad narrow negative"

sudo docker run -e MPLCONFIGDIR=/tmp/matplotlib -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName outs/scale_change_chipHeatmaps/scale_test_tss_${outname}_sorted_by_k4me3_plotHeatmap.pdf --sortUsingSamples 15 --regionsLabel ${plabels} --samplesLabel ${slabels} --outFileNameMatrix outs/downGenesHeatmap/scale_test_tss_${outname}_sorted_by_k4me3_plotHeatmap.mat.tab


#all

outname="alltracks"
matrixin_folder="in/test_chipseq_downstream/deeptools_heatmaps/"
plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"

sudo docker run -e MPLCONFIGDIR=/tmp/matplotlib -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile ${matrixin_folder}${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName outs/scale_change_chipHeatmaps/scale_test_${outname}_sorted_by_k4me3_plotHeatmap.pdf --sortUsingSamples 15 --regionsLabel ${plabels} --samplesLabel ${slabels} --outFileNameMatrix outs/downGenesHeatmap/scale_test_${outname}_sorted_by_k4me3_plotHeatmap.mat.tab

#lose k4me3

outname="K4me3_DOWN_Double_KO_vs_F_F"
matrixin_folder="in/test_chipseq_downstream/deeptools_heatmaps_filtered_by_diffDoubleKO/"
plabels="proximal_CpG+ proximal_CpG- distal_CpG+ distal_CpG-"

sudo docker run -e MPLCONFIGDIR=/tmp/matplotlib -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile ${matrixin_folder}${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName outs/scale_change_chipHeatmaps/scale_test_${outname}_sorted_by_k4me3_plotHeatmap.pdf --sortUsingSamples 15 --regionsLabel ${plabels} --samplesLabel ${slabels} --outFileNameMatrix outs/downGenesHeatmap/scale_test_${outname}_sorted_by_k4me3_plotHeatmap.mat.tab


###### tss av profiles

In [ ]:
%%script bash

sh git/Lara_MLL2/bin/profiles_broad_narrow_neg_chipseq.sh


#### venn down gene vs proximal

environment

In [ ]:
%%script bash

cd outs/Lara_multiomic_analysis


execute

In [ ]:
%%script R

library(VennDiagram)
library(RColorBrewer)
myCol <- brewer.pal(2, "Pastel2")

#TODO cambia location di salvataggio di questo file
broad_narrow_distribution <- read.table("/media/lucio/easystore/Lucio/Analysis/Lara/Lara_multiomic_analysis/outs/broad_narrow_distribution.tsv",sep="\t", header = TRUE)


unique(broad_narrow_distribution$V2)

#[1] "down"                  "all"                   "broad"
#[4] "proximal_dko_wt_loose"

myCol <- c("red", "blue")

set1 <- broad_narrow_distribution[which(broad_narrow_distribution$V2 == "proximal_dko_wt_loose"), "intg"]
set2 <- broad_narrow_distribution[which(broad_narrow_distribution$V2 == "down"), "intg"]

setdiff(set1,set2)


venn.diagram(
    x = list(set1, set2),
    category.names = c("Proximal Loss K4me3", "DEGs Down"),
    filename = '/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/venn_downDegs_prox4me3Loss/venn_downDegs_prox4me3Loss.png',

    # Output features
    output = TRUE,
    imagetype = "png",
    height = 480,
    width = 480,
    resolution = 300,
    compression = "lzw",
     margin = 0.2,

    # Cerchi
    lwd = 2,
    fill = myCol,

    # Numeri
    cex = 0.6,
    fontface = "bold",
    fontfamily = "sans",

    # Nomi delle categorie
    cat.cex = 0.4,
    cat.fontface = "bold",
    cat.default.pos = "outer",
    cat.pos = c(-27, 27),
    cat.dist = c(0.055, 0.055),
    cat.fontfamily = "sans"
)






## Figure 2

####  Heatmap ChIP-seq of down regulated genes (DEG as bivalent genes)





##### prepare environment


In [ ]:
%%script bash

outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps/"

outname="K4me3_DOWN_Double_KO_vs_F_F"

outpath="/mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO"

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis


##### make plot

In [ ]:
%%script bash

#all track

#ref_regions="outs/downGenesHeatmap/D0dkoVSwt.downgenes.bed"

outname="downDEGs_heatmap"

ref_regions="outs/downGenesHeatmap/D0dkoVSwt.broad.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.narrow.downgenes.bed outs/downGenesHeatmap/D0dkoVSwt.negative.downgenes.bed"

outpath="/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis"
plabels="broad narrow negative"


#/outs/downGenesHeatmap/

samples="./in/test_chipseq_downstream/deeptools_heatmaps/Anti-GFP_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/F_F_K27me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Double_KO_K27me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/D0_WT_RING1B__average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/D0_KO_RING1B__average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/F_F_K4me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmaps/Double_KO_K4me3_average.bw \
"

thrsholds="10 10 10 50 50 40 40"

slabels="Anti-GFP_av \
WT_K27me3_av \
DKO_K27me3_av \
WT_RING1B_av \
DKO_RING1B_av \
WT_H3K4me3_av \
DKO_H3K4me3_av \
"

sudo docker run -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) \
-e MPLCONFIGDIR=$outpath/.matplotlib \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -R $ref_regions -S $samples --sortUsingSamples 1 --numberOfProcessors 8 --scale 1 --binSize 10 --averageTypeBins "median" --outFileName outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --beforeRegionStartLength 5000 --afterRegionStartLength 5000 --referencePoint "center"

sudo docker run -v $outpath:$outpath -w $outpath -u $(id -u):$(id -g) -e MPLCONFIGDIR=$outpath/.matplotlib quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' --missingDataColor 0.6 --matrixFile outs/downGenesHeatmap/${outname}_deeptools_matrix.gzip --zMax ${thrsholds} --outFileName outs/downGenesHeatmap/${outname}_sorted_by_k4me3_plotHeatmap.pdf --sortUsingSamples 6 --regionsLabel ${plabels} --samplesLabel ${slabels} --outFileNameMatrix outs/downGenesHeatmap/${outname}_plotHeatmap.mat.tab


###### tss av profiles

In [ ]:

sh git/Lara_MLL2/bin/profiles_tss_additionals.sh


#### mESC poised Enhancerlist overlap

make environment

In [ ]:
%%script bash

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis

# cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed

mkdir /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/mESC_PE_overlap

cd /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/mESC_PE_overlap






execute analysis

In [ ]:
%%script bash

../../in/mESC_PE_lists/

head ../../in/mESC_PE_lists/poised_pooled_mm10.bed.txt

pooled="../../in/mESC_PE_lists/poised_pooled_mm10.bed.txt"
pemm10="../../in/mESC_PE_lists/PE_mm10.txt"

head $pooled
head $pemm10

inbed="../../in/test_chipseq_downstream/deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed"

maindir="/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis"

awk 'BEGIN{OFS="\t"} {$2=($2-1000>0)?$2-1000:0; $3=$3+1000; print}' $inbed > ./distal_CpG_plus_extended.bed

head $inbed
head distal_CpG_plus_extended.bed

sudo docker run -v $maindir:$maindir -w $(pwd) -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./distal_CpG_plus_extended.bed -b $pooled \
> overlap_poised_pooled_distal_CpG_plus.bed

wc -l overlap_poised_pooled_distal_CpG_plus.bed
#846 overlap_poised_pooled_distal_CpG_plus.bed
wc -l $inbed
#2578 K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed
wc -l $pooled
#4198

sudo docker run -v $maindir:$maindir -w $(pwd) -u $(id -u):$(id -g) \
quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a ./distal_CpG_plus_extended.bed -b $pemm10 \
> overlap_pemm10_distal_CpG_plus.bed

wc -l overlap_pemm10_distal_CpG_plus.bed
#161
wc -l $pemm10
1014



hypergeometrical test

In [ ]:
%%script R

overlap_pooled=846
inbed=2578
pooled=4198
pemm10=1014
overlap_pemm10=161







##### make environment

In [ ]:
%%script bash

md5sum /mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/in/test_chipseq_downstream/deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed

# 205f249a856ccc157639136533f08525

md5sum /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed

# 205f249a856ccc157639136533f08525

/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/outs/great/Double_KO_vs_F_F/basal/dcp/20250210-public-4.0.4-X2dyH7-mm10-all-gene.txt







##### make analysis

In [ ]:
%%script R

source("./git/Lara_MLL2/bin/D0D4_dist_DEG_dcp.R")


## figure 4

### broad narrow D4 evaluation 

In [ ]:
%%script R

source ("./git/Lara_MLL_2/bin/D4_NMIlen_DEG_vs_ALL_deve_genes.R")

### heatmap induced genes

In [ ]:
%%script bash

mkdir ./in/test_chipseq_downstream/parallel_averageBigwig_D4
mkdir ./in/240926_chip_D4_bw

cp -r /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/otherouts/parallel_averageBigwig_D4/* ./in/test_chipseq_downstream/parallel_averageBigwig_D4/
cp /mnt/datawk1/analysis/Lara/240926_chip_D4/nfout/random/star/mergedLibrary/bigwig/deeptools/D4WTMLL2B.bigWig ./in/240926_chip_D4_bw/

./in/240926_chip_D4_bw/


In [ ]:
%%script bash


macs_peaks="outs/D4_H3K27me3_broad_narrow/induced_broad.bed"
wc -l $macs_peaks
#627 outs/D4_H3K27me3_broad_narrow/induced_broad.bed

macs_peaks_FC2="outs/D4_H3K27me3_broad_narrow/inducedFC2_broad.bed"
wc -l $macs_peaks_FC2
#449 outs/D4_H3K27me3_broad_narrow/inducedFC2_broad.bed

samples="./in/test_chipseq_downstream/deeptools_heatmap_tmp/Anti-GFP_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Anti-Menin_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Anti-Mll1_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Mll2_KO_Mll1_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Anti-RbBp5_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_RbBP5_average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_RbBP5__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig/D0_WT_H3K27ac__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig/D0_DoubleKO_H3K27ac__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K27ac__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K27ac__average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K27me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K27me3_average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K27me3__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K27me3__average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K4me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K4me3_average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K4me3__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K4me3__average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K4me2_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K4me2_average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K4me2__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K4me2__average.bw  \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K4me1_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K4me1_average.bw \
"

thrsholds="10 10 10 10 15 15 15 10 10 10 10 10 10 10 10 40 40 40 40 30 30 30 30"
mins="0 0 0 0 0 0 0 0 0 0 0 0  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0"

outname="inducedFC2_genes_broad_sorted_D4WTh3k4me3"
sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${samples} -R ${macs_peaks_FC2} \
-b 1000 --averageTypeBins "median" --numberOfProcessors 8 --scale 1 --binSize 10 \
--outFileName outs/D4_H3K27me3_broad_narrow/${outname}_deeptools_matrix.gzip --referencePoint "center" \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --missingDataAsZero

sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile outs/D4_H3K27me3_broad_narrow/${outname}_deeptools_matrix.gzip \
--outFileName outs/D4_H3K27me3_broad_narrow/${outname}_plotHeatmap.pdf \
--outFileNameMatrix outs/D4_H3K27me3_broad_narrow/${outname}_plotHeatmap.mat.tab \
--zMax ${thrsholds} --zMin ${mins} --sortUsingSamples 18 --sortRegions "descend"




# outname="inducedFC2_genes_broad_sorted_D4WTh3k27ac"
# sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
# quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
# computeMatrix reference-point -S ${samples} -R ${macs_peaks_FC2} \
# -b 1000 --averageTypeBins "median" --numberOfProcessors 8 --scale 1 --binSize 10 \
# --outFileName outs/D4_H3K27me3_broad_narrow/${outname}_deeptools_matrix.gzip --referencePoint "center" \
# --beforeRegionStartLength 2000 --afterRegionStartLength 2000 --missingDataAsZero

# sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
# quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
# --missingDataColor 0.6 --matrixFile outs/D4_H3K27me3_broad_narrow/${outname}_deeptools_matrix.gzip \
# --outFileName outs/D4_H3K27me3_broad_narrow/${outname}_plotHeatmap.pdf \
# --outFileNameMatrix outs/D4_H3K27me3_broad_narrow/${outname}_plotHeatmap.mat.tab \
# --zMax ${thrsholds} --zMin ${mins} --sortUsingSamples 10 --sortRegions "descend"

echo outs/D4_H3K27me3_broad_narrow/${outname}_plotHeatmap.mat.tab
zcat outs/D4_H3K27me3_broad_narrow/${outname}_plotHeatmap.mat.tab | head -n 1

# outname="inducedFC2_genes_broad_sorted_D4WTh3k27me3"
# sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
# quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
# computeMatrix reference-point -S ${samples} -R ${macs_peaks_FC2} \
# -b 1000 --sortUsingSamples 11 --averageTypeBins "median" --numberOfProcessors 8 --scale 1 --binSize 10 \
# --outFileName outs/D4_H3K27me3_broad_narrow/${outname}_deeptools_matrix.gzip --referencePoint "center" \
# --beforeRegionStartLength 2000 --afterRegionStartLength 2000 --missingDataAsZero

# sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
# quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
# --missingDataColor 0.6 --matrixFile outs/D4_H3K27me3_broad_narrow/${outname}_deeptools_matrix.gzip \
# --outFileName outs/D4_H3K27me3_broad_narrow/${outname}_plotHeatmap.pdf \
# --outFileNameMatrix outs/D4_H3K27me3_broad_narrow/${outname}_plotHeatmap.mat.tab \
# --zMax ${thrsholds} --zMin ${mins} --sortUsingSamples 14 --sortRegions "descend"

# test unstrand

# cut -f1-4 ./outs/D4_H3K27me3_broad_narrow/induced_broad.bed > ./outs/D4_H3K27me3_broad_narrow/induced_broad_unstrand.bed

# outname="unstrandFC2_induced_genes_broad_sorted_D4WTh3k27me3"
# sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
# quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
# computeMatrix reference-point -S ${samples} -R ./outs/D4_H3K27me3_broad_narrow/induced_broad_unstrand.bed \
# -b 1000 --sortUsingSamples 11 --averageTypeBins "median" --numberOfProcessors 8 --scale 1 --binSize 10 \
# --outFileName outs/D4_H3K27me3_broad_narrow/${outname}_deeptools_matrix.gzip --referencePoint "center" \
# --beforeRegionStartLength 2000 --afterRegionStartLength 2000 --missingDataAsZero

# sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
# quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
# --missingDataColor 0.6 --matrixFile outs/D4_H3K27me3_broad_narrow/${outname}_deeptools_matrix.gzip \
# --outFileName outs/D4_H3K27me3_broad_narrow/${outname}_plotHeatmap.pdf \
# --outFileNameMatrix outs/D4_H3K27me3_broad_narrow/${outname}_plotHeatmap.mat.tab \
# --zMax ${thrsholds} --zMin ${mins} --sortUsingSamples 14 --sortRegions "descend"




In [ ]:
%%script bash

sh git/Lara_MLL2/bin/heatmap_promoter_of_PcG_induced_genes.sh


### heatmap dcp loss k4me3 near d4 k27ac

#### set environment

In [ ]:
%%script bash

cp /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/D4WTH3K27acA_macs_coordinate.bed in/test_chipseq_downstream/
mkdir

mkdir outs/dcp_lossk4me3_near_k27ac_heatmap



#### make subset of peaks

In [ ]:
%%script bash

prjpath="/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/"

#sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
#quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools subtract -A -a proximal_preaks.bed -b proximal_CpG_plus_unique.bed \
#> proximal_CpG_minus.bed


awk '{print $1, ($2-1000<0?0:$2-1000), $3+1000, $4, $5, $6}' OFS="\t" /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/D4WTH3K27acA_macs_coordinate.bed >  outs/dcp_lossk4me2_near_k27ac_heatmap/5kb_extendedn_D4WTH3K27acA_macs_coordinate.bed


sudo docker run -v $prjpath:$prjpath -w $prjpath -u $(id -u):$(id -g) quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a in/test_chipseq_downstream/deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed -b outs/dcp_lossk4me2_near_k27ac_heatmap/5kb_extendedn_D4WTH3K27acA_macs_coordinate.bed > outs/dcp_lossk4me2_near_k27ac_heatmap/dcp_lossk4me2_near_D4_k27ac.bed

sudo docker run -v $prjpath:$prjpath -w $prjpath -u $(id -u):$(id -g) quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a in/test_chipseq_downstream/deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_minus_Double_KO_vs_F_F.bed -b outs/dcp_lossk4me2_near_k27ac_heatmap/5kb_extendedn_D4WTH3K27acA_macs_coordinate.bed > outs/dcp_lossk4me2_near_k27ac_heatmap/dcm_lossk4me2_near_D4_k27ac.bed

#TODO see in igv if the output bed is subsetted in proper way

macs_peaks="outs/dcp_lossk4me2_near_k27ac_heatmap/dcp_lossk4me2_near_D4_k27ac.bed"
dcm_macs_peaks="outs/dcp_lossk4me2_near_k27ac_heatmap/dcm_lossk4me2_near_D4_k27ac.bed"


samples="./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_MLL2__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_RbBP5__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig/D0_WT_H3K27ac__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig/D0_DoubleKO_H3K27ac__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K27ac__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K27ac__average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K27me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K27me3_average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K27me3__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K27me3__average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K4me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K4me3_average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K4me3__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K4me3__average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K4me2_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K4me2_average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K4me2__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K4me2__average.bw  \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K4me1_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K4me1_average.bw \
"

thrsholds="4 4 13 13 13 13 7 7 7 7 16 16 16 16 30 30 30 30 8 8"

mins="0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0"

outname="dcp_lossk4me2_near_k27ac"
sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${samples} -R ${macs_peaks} \
-b 1000 --averageTypeBins "median" --numberOfProcessors 8 --scale 1 --binSize 10 \
--outFileName outs/dcp_lossk4me2_near_k27ac_heatmap/${outname}_deeptools_matrix.gzip --referencePoint "center" \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --missingDataAsZero

sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile outs/dcp_lossk4me2_near_k27ac_heatmap/${outname}_deeptools_matrix.gzip \
--outFileName outs/dcp_lossk4me2_near_k27ac_heatmap/${outname}_plotHeatmap.pdf \
--outFileNameMatrix outs/dcp_lossk4me2_near_k27ac_heatmap/${outname}_plotHeatmap.mat.tab \
--zMax ${thrsholds} --zMin ${mins} --sortUsingSamples 5 --sortRegions "descend"

outname="dcm_lossk4me2_near_k27ac"
sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${samples} -R ${dcm_macs_peaks} \
-b 1000 --averageTypeBins "median" --numberOfProcessors 8 --scale 1 --binSize 10 \
--outFileName outs/dcp_lossk4me2_near_k27ac_heatmap/${outname}_deeptools_matrix.gzip --referencePoint "center" \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --missingDataAsZero

sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile outs/dcp_lossk4me2_near_k27ac_heatmap/${outname}_deeptools_matrix.gzip \
--outFileName outs/dcp_lossk4me2_near_k27ac_heatmap/${outname}_plotHeatmap.pdf \
--outFileNameMatrix outs/dcp_lossk4me2_near_k27ac_heatmap/${outname}_plotHeatmap.mat.tab \
--zMax ${thrsholds} --zMin ${mins} --sortUsingSamples 5 --sortRegions "descend"




In [ ]:
%%script bash

prjpath="/mnt/datawk1/analysis/Lara/Lara_multiomic_analysis/"

#sudo docker run -v $outpath:$outpath -v $inpath:$inpath -w $outpath -u $(id -u):$(id -g) \
#quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools subtract -A -a proximal_preaks.bed -b proximal_CpG_plus_unique.bed \
#> proximal_CpG_minus.bed

awk '{print $1, ($2-1000<0?0:$2-1000), $3+1000, $4, $5, $6}' OFS="\t" /mnt/datawk1/analysis/Lara/test_chipseq_dowstream/thr5_D4WTH3K27acA_macs_coordinate.bed >  outs/dcp_lossk4me3_near_k27ac_heatmap/thr5_5kb_extendedn_D4WTH3K27acA_macs_coordinate.bed

sudo docker run -v $prjpath:$prjpath -w $prjpath -u $(id -u):$(id -g) quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a in/test_chipseq_downstream/deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_plus_Double_KO_vs_F_F.bed -b outs/dcp_lossk4me3_near_k27ac_heatmap/thr5_5kb_extendedn_D4WTH3K27acA_macs_coordinate.bed > outs/dcp_lossk4me3_near_k27ac_heatmap/thr5_dcp_lossk4me3_near_D4_k27ac.bed

sudo docker run -v $prjpath:$prjpath -w $prjpath -u $(id -u):$(id -g) quay.io/biocontainers/bedtools:2.31.1--hf5e1c6e_1 bedtools intersect -wa -a in/test_chipseq_downstream/deeptools_heatmaps_filtered_by_diffDoubleKO/K4me3_distal_CpG_minus_Double_KO_vs_F_F.bed -b outs/dcp_lossk4me3_near_k27ac_heatmap/thr5_5kb_extendedn_D4WTH3K27acA_macs_coordinate.bed > outs/dcp_lossk4me3_near_k27ac_heatmap/thr5_dcm_lossk4me3_near_D4_k27ac.bed


#TODO see in igv if the output bed is subsetted in proper way

macs_peaks="outs/dcp_lossk4me3_near_k27ac_heatmap/thr5_dcp_lossk4me3_near_D4_k27ac.bed"

samples="./in/240926_chip_D4_bw/D4WTMLL2B.bigWig \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_RbBP5__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig/D0_WT_H3K27ac__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig/D0_DoubleKO_H3K27ac__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K27ac__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K27ac__average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K27me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K27me3_average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K27me3__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K27me3__average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K4me3_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K4me3_average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K4me3__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K4me3__average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K4me2_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K4me2_average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_WT_H3K4me2__average.bw \
./in/test_chipseq_downstream/parallel_averageBigwig_D4/D4_DKO_H3K4me2__average.bw  \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/F_F_K4me1_average.bw \
./in/test_chipseq_downstream/deeptools_heatmap_tmp/Double_KO_K4me1_average.bw \
"

thrsholds="4 4 16 16 16 16 7 7 7 7 16 16 16 16 30 30 30 30 8 8"

mins="0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0"

outname="thr5_dcp_lossk4me3_near_k27ac"
sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 \
computeMatrix reference-point -S ${samples} -R ${macs_peaks} \
-b 1000 --averageTypeBins "median" --numberOfProcessors 8 --scale 1 --binSize 10 \
--outFileName outs/dcp_lossk4me3_near_k27ac_heatmap/${outname}_deeptools_matrix.gzip --referencePoint "center" \
--beforeRegionStartLength 5000 --afterRegionStartLength 5000 --missingDataAsZero

sudo docker run -v `pwd`:`pwd` -w `pwd` -u $(id -u):$(id -g) -e MPLCONFIGDIR=/tmp \
quay.io/biocontainers/deeptools:3.5.5--pyhdfd78af_0 plotHeatmap --colorMap 'seismic' \
--missingDataColor 0.6 --matrixFile outs/dcp_lossk4me3_near_k27ac_heatmap/${outname}_deeptools_matrix.gzip \
--outFileName outs/dcp_lossk4me3_near_k27ac_heatmap/${outname}_plotHeatmap.pdf \
--outFileNameMatrix outs/dcp_lossk4me3_near_k27ac_heatmap/${outname}_plotHeatmap.mat.tab \
--zMax ${thrsholds} --zMin ${mins} --sortUsingSamples 5 --sortRegions "descend"



### D4_Lara_distance_by_nearest_DEGs

In [ ]:
%script R

#source("git/Lara_MLL2/bin/D4_Lara_distance_by_nearest_DEGs.R")
source("git/Lara_MLL2/bin/D0D4_dist_DEG_dcp.R")

# GEO/ArrayExpress submission

## main environment

In [ ]:
%%scipt bash

mkdir /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/
mkdir /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/git
mkdir /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/in
mkdir /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/out

mkdir /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/out/MLL2_RNAseq_raw
mkdir /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/out/MLL2_microC_raw
mkdir /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/out/MLL2_chipSEQ_raw

mkdir /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/out/MLL2_RNAseq_processed
mkdir /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/out/MLL2_microC_processed
mkdir /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/out/MLL2_chipSEQ_processed


cd /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/git

 git clone git@github.com:lucidif/Lara_MLL2.git

cd /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub

 



## microC

In [ ]:
%%scipt bash

sh git/Lara_MLL2/bin/GEO_microC_fileprep.sh

sed -i 's/\r$//' git/Lara_MLL2/sheets/geo_sub_microC.tsv   # rimuove ^M

#1 bash git/Lara_MLL2/bin/check_geo_files_chipseq.sh git/Lara_MLL2/sheets/geo_sub_RNAseq.tsv out/RNAseq_missing.txt
bash git/Lara_MLL2/bin/check_geo_files.sh git/Lara_MLL2/sheets/geo_sub_microC.tsv out/microC_missing.txt

#2 renames files
#bash git/Lara_MLL2/bin/organize_geo_files.sh out/MLL2_microC_raw  git/Lara_MLL2/sheets/geo_sub_microC.tsv

bash git/Lara_MLL2/bin/organize_geo_files.sh \
  out/MLL2_microC_raw out/MLL2_microC_processed \
  git/Lara_MLL2/sheets/geo_sub_microC.tsv \
  out/microC_md5.tsv

#3 check files

bash git/Lara_MLL2/bin/verify_geo_md5.sh git/Lara_MLL2/sheets/geo_sub_microC.tsv microC_md5.tsv

# RAW-only (es. micro-C)
bash git/Lara_MLL2/bin/compare_and_write_md5.sh \
  git/Lara_MLL2/sheets/geo_sub_microC.tsv \
  out/MLL2_microC_raw \
  out/microC_md5_compare.tsv

# FULL (FASTQ + processed)
# bash compare_and_write_md5.sh \
#   GEOrename_RNAseq.tsv \
#   out/RNAseq_raw \
#   out/RNAseq_proc \
#   out/RNAseq_md5_compare.tsv

bash git/Lara_MLL2/bin/easy_make_md5.sh -t git/Lara_MLL2/sheets/geo_sub_microC.tsv -o out -n microC_easymd5



## RNAseq

In [ ]:
%%scipt bash

sh git/Lara_MLL2/bin/GEO_RNAseq_fileprep.sh

sed -i 's/\r$//' git/Lara_MLL2/sheets/geo_sub_RNAseq.tsv   # rimuove ^M

# 1) Check: esistenza dei file sorgente (FASTQ1/2 + BAM)
bash git/Lara_MLL2/bin/check_geo_files.sh git/Lara_MLL2/sheets/geo_sub_RNAseq.tsv out/rnaseq_missing.txt

#copy and rename (processed data not used)
# bash git/Lara_MLL2/bin/organize_geo_files.sh \
#   out/MLL2_RNAseq_raw out/MLL2_RNAseq_processed \
#   git/Lara_MLL2/sheets/geo_sub_RNAseq.tsv \
#   out/rnaseq_md5.tsv
bash git/Lara_MLL2/bin/geo_copy_md5.sh -i git/Lara_MLL2/sheets/geo_sub_RNAseq.tsv \
  -d out/MLL2_RNAseq_raw \
  -o out/MLL2_RNAseq_md5.tsv


#check md5
bash git/Lara_MLL2/bin/verify_geo_md5.sh \
  git/Lara_MLL2/sheets/geo_sub_RNAseq.tsv \
  out/rnaseq_md5.tsv \
  --tsv

#--quiet > out/rnaseq_md5_mismatches.tsv

#verify md5
#bash git/Lara_MLL2/bin/verify_geo_md5.sh git/Lara_MLL2/sheets/geo_sub_RNAseq.tsv out/rnaseq_md5.tsv

# bash git/Lara_MLL2/bin/check_GEO_md5.sh \
#   -i git/Lara_MLL2/sheets/geo_sub_RNAseq.tsv \
#   -d out/MLL2_RNAseq_raw \
#   -m out/MLL2_RNAseq_md5.tsv

#extract md5
bash git/Lara_MLL2/bin/easy_make_md5.sh -t git/Lara_MLL2/sheets/geo_sub_RNAseq.tsv -o out -n RNAseq_easymd5

#import in final folder
mkdir out/mll2-line1_Shahidian_2025-09-22_v01/rnaseq/processed
cp out/MLL2_RNAseq_processed/all_samples_normalized_counts.tsv out/mll2-line1_Shahidian_2025-09-22_v01/rnaseq/processed
md5sum out/mll2-line1_Shahidian_2025-09-22_v01/rnaseq/processed/all_samples_normalized_counts.tsv

cp /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/git/Lara_MLL2/sheets/GEO_RNA-seq.xlsx /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/out/mll2-line1_Shahidian_2025-09-22_v01/docs/



## chipSEQ

In [ ]:
%%script bash

#check that files exist

bash git/Lara_MLL2/bin/check_geo_files_chipseq.sh git/Lara_MLL2/sheets/geo_sub_chipSEQ.tsv out/geo_sub_chipSEQ_missing_files.txt

#copy and rename

bash git/Lara_MLL2/bin/organize_geo_files_chipseq.sh out/MLL2_chipSEQ_raw out/MLL2_chipSEQ_processed git/Lara_MLL2/sheets/geo_sub_chipSEQ.tsv

#verify md5

#bash git/Lara_MLL2/bin/verify_geo_md5.sh git/Lara_MLL2/sheets/geo_sub_chipSEQ.tsv md5_report.tsv
# bash git/Lara_MLL2/bin/easy_make_md5.sh \
#   -i git/Lara_MLL2/sheets/geo_sub_chipSEQ.tsv \
#   -o out/MLL2_chipSEQ_md5.tsv

bash git/Lara_MLL2/bin/easy_make_md5.sh -t git/Lara_MLL2/sheets/geo_sub_chipSEQ.tsv -o out -n CHIPseq_md5
#bash md5_from_tsv.sh -t git/Lara_MLL2/sheets/geo_sub_chipSEQ.tsv -o out -n CHIPseq_md5 -p #with processed

cp /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/git/Lara_MLL2/sheets/GEO_ChIP-seq.xlsx /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/out/mll2-line1_Shahidian_2025-09-22_v01/docs/

mkdir out/mll2-line1_Shahidian_2025-09-22_v01/chipseq/raw
rsync -av out/MLL2_chipSEQ_raw/*.fq.gz out/mll2-line1_Shahidian_2025-09-22_v01/chipseq/raw

## scRNAseq

### raw data

In [ ]:
cd /media/lucio/easystore/bck_data/Lara/scRNA/20260512_HN00274378/fastq

lftp -u annotare,annotare1 ftp-private.ebi.ac.uk <<'EOF'
cd /prod/mqaqbfca-829oe0vz89xu/
mput *.fastq.gz
bye
EOF

### processed

In [ ]:
mainprj=${PWD}

#cd outs/geo_sub/out/scRNAseq_processed/

chmod +x git/Lara_MLL2/bin/rename_scRNA_AE.sh
git/Lara_MLL2/bin/rename_scRNA_AE.sh

cd outs/geo_sub/out/scRNAseq_AE_upload
md5sum *.gz > hash_processed.txt
cat hash_processed.txt

lftp -u annotare,annotare1 ftp-private.ebi.ac.uk <<'EOF'
cd /prod/mqaqbfca-829oe0vz89xu/
mput *.gz
bye
EOF



## final folder organized

In [ ]:
mkdir out/mll2-line1_Shahidian_2025-09-22_v01
mkdir out/mll2-line1_Shahidian_2025-09-22_v01/docs
mkdir out/mll2-line1_Shahidian_2025-09-22_v01/chipseq
#mkdir out/mll2-line1_Shahidian_2025-09-22_v01/chipseq/raw
#mkdir out/mll2-line1_Shahidian_2025-09-22_v01/chipseq/processed

mkdir out/mll2-line1_Shahidian_2025-09-22_v01/rnaseq
mkdir out/mll2-line1_Shahidian_2025-09-22_v01/rnaseq/raw
ls /media/lucio/easystore/Lucio/Analysis/Lara/geo_sub/out/MLL2_RNAseq_raw/*.fq.gz
mv out/MLL2_RNAseq_raw/*.fq.gz out/mll2-line1_Shahidian_2025-09-22_v01/rnaseq/raw

#mkdir out/mll2-line1_Shahidian_2025-09-22_v01/rnaseq/raw
#mkdir out/mll2-line1_Shahidian_2025-09-22_v01/rnaseq/processed
mkdir out/mll2-line1_Shahidian_2025-09-22_v01/microc
#mkdir out/mll2-line1_Shahidian_2025-09-22_v01/microc/raw
#mkdir out/mll2-line1_Shahidian_2025-09-22_v01/microc/processed


# reviewer asking

### quantile normalization

In [ ]:
%%script R

source("git/Lara_MLL2/bin/quantile_normalization_bigwig.R")




### TE elements near peaks

In [ ]:
%%script R

source("git/Lara_MLL2/bin/TE_distance_extimation.R")

### de novo assembly

In [ ]:
%%script bash

git/Lara_MLL2/bin/de_novo_trascripts.sh


### TADs boundaries exploration

In [ ]:
# in TAD compare scripts I generates TADs boundaries bed file of WT D0 and D4
#git/Lara_MLL2/bin/TADcompare_WT_DKO_d0_d4.R
sh git/Lara_MLL2/bin/TADs_boundaries_pileup.sh
